# Data

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Radio buttons for choosing mode: 'build' or 'load'
mode_selector = widgets.ToggleButtons(
    options=['build', 'load'],
    value='load',  # Default value
    description='Mode:',
    disabled=False
)

# Display the radio buttons
display(mode_selector)
mode = mode_selector.value

## Import Corpus Data

In [ ]:
import os
corpus_dir = os.path.expanduser('~/Kod/fontes_nlp/data/out/')
corpus_files = [ f for f in os.listdir(corpus_dir) if f.endswith("conllu") ]

In [ ]:
from nltk import corpus as corpus
corpus = corpus.reader.conll.ConllCorpusReader(root=corpus_dir, fileids=corpus_files, columntypes = ('words', 'pos', 'tree', 'chunk', 'ne', 'srl', 'ignore'))

In [ ]:
if mode == 'build':
    from conllu import parse
    corpus = dict.fromkeys(corpus_files)
    for file in corpus_files:
        with open(os.path.join(corpus_dir,file), "r", encoding="utf-8") as f:
            txt = f.read()
            sents = parse(txt)
            corpus[file] = sents

## Corpus Parsing Classes

### CONLLU to NLTK

In [ ]:
import os
import re
from nltk.corpus.reader.api import CorpusReader
from nltk.corpus.reader.util import StreamBackedCorpusView

class ConlluCorpusReader(CorpusReader):
    """
    A reader for CoNLL-U format files, extending NLTK's CorpusReader.
    CoNLL-U files have 10 fields per token:
    ID, FORM, LEMMA, UPOS, XPOS, FEATS, HEAD, DEPREL, DEPS, MISC
    """

    def __init__(self, root, fileids):
        """
        Initialize the ConlluCorpusReader.

        :param root: The root directory where corpus files are stored.
        :param fileids: A list of file IDs to read, or a regex pattern to match files.
        """
        # Initialize the base CorpusReader class
        super().__init__(root, fileids)

    def _read_conllu_block(self, stream):
        """
        Read a block of CoNLL-U formatted text, returning a list of lists where each inner
        list represents a sentence.
        """
        block = []
        sent = []

        for line in stream:
            line = line.strip()
            if not line:
                if sent:  # end of a sentence
                    block.append(sent)
                    sent = []
                continue
            if line.startswith('#'):  # comment or metadata
                continue
            # Parse the fields of a CoNLL-U formatted token
            fields = line.split('\t')
            if len(fields) == 10:  # Properly formatted CoNLL-U token
                sent.append({
                    "id": fields[0],      # ID
                    "form": fields[1],    # FORM
                    "lemma": fields[2],   # LEMMA
                    "upos": fields[3],    # UPOS
                    "xpos": fields[4],    # XPOS
                    "feats": fields[5],   # FEATS
                    "head": fields[6],    # HEAD
                    "deprel": fields[7],  # DEPREL
                    "deps": fields[8],    # DEPS
                    "misc": fields[9]     # MISC
                })
        return block if block else None

    def _read_conll_sentences(self, fileid):
        """
        Read sentences from the specified CoNLL-U file.
        """
        file_path = self.abspath(fileid)  # Use NLTK's abspath method for file resolution
        print(f"Attempting to open file: {file_path}")

        with open(file_path, 'r', encoding='utf-8') as f:
            return self._read_conllu_block(f)

    def sents(self, fileids=None):
        """
        Returns a list of sentences, where each sentence is a list of tokens with their forms.
        """
        sentences = []
        for fileid in self.abspaths(fileids):  # Resolve file paths using abspaths
            sentences.extend([[token["form"] for token in sentence] for sentence in self._read_conll_sentences(fileid)])
        return sentences

    def tagged_words(self, fileids=None):
        """
        Returns a list of words and their corresponding tags.
        Each token is represented as a tuple (FORM, LEMMA, UPOS, XPOS, FEATS, MISC).
        """
        words = []
        for fileid in self.abspaths(fileids):  # Resolve file paths using abspaths
            for sentence in self._read_conll_sentences(fileid):
                for token in sentence:
                    words.append((token["form"], token["lemma"], token["upos"], token["xpos"], token["feats"], token["misc"]))
        return words

    def tagged_sents(self, fileids=None):
        """
        Returns a list of sentences, where each sentence is a list of tuples (FORM, LEMMA, UPOS, XPOS, FEATS, MISC).
        """
        tagged_sentences = []
        for fileid in self.abspaths(fileids):  # Resolve file paths using abspaths
            for sentence in self._read_conll_sentences(fileid):
                tagged_sentences.append([(token["form"], token["lemma"], token["upos"], token["xpos"], token["feats"], token["misc"]) for token in sentence])
        return tagged_sentences

    def lemmas(self, fileids=None):
        """
        Returns a list of lemmas for all tokens in all sentences.
        """
        lemmas = []
        for fileid in self.abspaths(fileids):  # Resolve file paths using abspaths
            for sentence in self._read_conll_sentences(fileid):
                for token in sentence:
                    lemmas.append(token["lemma"])
        return lemmas

    def upos_tags(self, fileids=None):
        """
        Returns a list of UPOS tags for all tokens in all sentences.
        """
        upos_tags = []
        for fileid in self.abspaths(fileids):  # Resolve file paths using abspaths
            for sentence in self._read_conll_sentences(fileid):
                for token in sentence:
                    upos_tags.append(token["upos"])
        return upos_tags

    def xpos_tags(self, fileids=None):
        """
        Returns a list of XPOS tags for all tokens in all sentences.
        """
        xpos_tags = []
        for fileid in self.abspaths(fileids):  # Resolve file paths using abspaths
            for sentence in self._read_conll_sentences(fileid):
                for token in sentence:
                    xpos_tags.append(token["xpos"])
        return xpos_tags

    def features(self, fileids=None):
        """
        Returns a list of features for all tokens in all sentences.
        """
        features_list = []
        for fileid in self.abspaths(fileids):  # Resolve file paths using abspaths
            for sentence in self._read_conll_sentences(fileid):
                for token in sentence:
                    features_list.append(token["feats"])
        return features_list

    def misc(self, fileids=None):
        """
        Returns a list of MISC information for all tokens in all sentences.
        """
        misc_info = []
        for fileid in self.abspaths(fileids):  # Resolve file paths using abspaths
            for sentence in self._read_conll_sentences(fileid):
                for token in sentence:
                    misc_info.append(token["misc"])
        return misc_info


In [ ]:
reader = ConlluCorpusReader(root=os.path.expanduser('~/Kod/fontes_nlp/data/out/'), 
                            fileids=r'.+\.conllu')

In [ ]:
sentences = reader.sents()

In [ ]:
lemmas = reader.lemmas('VSt_minor_TEI_final.conllu')

In [ ]:
reader.tagged_words()

### CONLLU to spacy

In [ ]:
import os
from spacy.tokens import Doc
from spacy_conll import init_parser
from spacy_conll.parser import ConllParser
from spacy.language import Language
from spacy.lang.la import Latin

# Define a factory for the ConllParser
#@Language.factory('conll_parser')
#def create_conll_parser(nlp, name):
#    return ConllParser(nlp)

# Initialize spaCy's Latin model with the CoNLL formatter
nlp = Latin()
nlp.add_pipe("conll_formatter", last=True)
conll_parser = ConllParser(nlp)
#nlp = ConllParser(nlp)
#nlp.add_pipe('conll_parser', last=True)
#doc = nlp.parse_conll_file_as_spacy("/home/krzys/Kod/fontes_nlp/data/out/AGZ3_TEI_final.conllu")
#doc = nlp.parse_conll_file_as_spacy("normalized/AGZ4_TEI_final.conllu")
#nlp.add_pipe('sentencizer')
#nlp.add_pipe("conll_formatter", last="true")

In [ ]:
import os
from spacy.tokens import DocBin
from spacy_conll.parser import ConllParser
import pandas as pd

class SpacyCorpus:
    def __init__(self, directory, nlp, log_file=None, file_filter=None, verbose=True):
        """
        Initialize the corpus with the directory containing CoNLL-U files, an nlp pipeline,
        and optionally a log file and file filter.

        :param directory: Directory containing the CoNLL-U files.
        :param nlp: spaCy language model with a conll_formatter pipeline.
        :param log_file: Optional path to a log file for saving detailed logs.
        :param file_filter: Optional filter for selecting files to process (list of filenames, prefix, or extension).
        :param verbose: Boolean indicating whether to print messages during processing.
        """
        self.directory = directory
        self.nlp = nlp
        self.docs = []
        self.log_file = log_file
        self.file_filter = file_filter  # A filter for which files to process
        self.verbose = verbose  # Control print messages

        if self.log_file:
            with open(self.log_file, 'w') as f:
                f.write("SpacyCorpus log\n")

        # Set a custom extension on the Doc class, if not already set
        if not Doc.has_extension("metadata"):
            Doc.set_extension("metadata", default={})
        
        self._load_docs()

    def _log(self, message, batch=False):
        """
        Log a message both to the console and to a log file if specified.
        """
        # Print to the console if verbose is True
        if self.verbose and not batch:
            print(message)

        # Write the full log to a file, if provided
        if self.log_file:
            with open(self.log_file, 'a') as f:
                f.write(f"{message}\n")

    def _filter_files(self, file_name):
        """
        Filter files based on the given file filter criteria.
        """
        # If a list of specific filenames is provided, check if the file matches
        if isinstance(self.file_filter, list):
            return file_name in self.file_filter

        # If a string (prefix or extension) is provided, check if it matches
        if isinstance(self.file_filter, str):
            return file_name.startswith(self.file_filter) or file_name.endswith(self.file_filter)

        # Default: process all files
        return True

    def _load_docs(self):
        """
        Load all documents from the CoNLL-U files in the specified directory and print debugging info.
        """
        for file_name in os.listdir(self.directory):
            # Only process files based on the filter
            if file_name.endswith(".conllu") and self._filter_files(file_name):  # Apply the filter
                file_path = os.path.join(self.directory, file_name)

                # Log which file is being processed
                self._log(f"Processing file: {file_name}")

                # Use spacy_conll's parser to parse the file and create a single document
                try:
                    # This will now create a single document for each CoNLL-U file
                    doc = self.nlp.parse_conll_file_as_spacy(file_path)
                    
                except Exception as e:
                    error_message = f"Error parsing file: {file_name} - {str(e)}"
                    self._log(error_message)
                    continue  # Skip to the next file if there's an error

                doc._.metadata["doc_id"] = file_name
                # Append the document to the list of docs
                self.docs.append(doc)

                # Log the number of sentences processed in the document
                self._log(f"Finished processing file: {file_name}, {len(doc)} tokens loaded.\n")

    def to_docbin(self):
        """
        Convert the processed docs to a DocBin object and return it.
        
        :return: A DocBin object containing the processed docs.
        """
        doc_bin = DocBin(store_user_data=True)
    
        for doc in self.docs:
            # Check for non-serializable attributes in user_data
            non_serializable_attrs = [key for key in doc.user_data.keys() if isinstance(doc.user_data[key], (pd.Series, pd.DataFrame))]
    
            if non_serializable_attrs:
                #print(f"Warning: Skipping serialization of non-serializable attributes: {non_serializable_attrs} for document.")
                # Remove non-serializable attributes from user_data
                for key in non_serializable_attrs:
                    del doc.user_data[key]  # Remove non-serializable attribute
            
            doc_bin.add(doc)  # Add the doc after cleaning up non-serializable attributes
        
        return doc_bin  # Return the DocBin object


    def __iter__(self):
        """
        Iterate over the Doc objects in the corpus.
        """
        return iter(self.docs)

    def __len__(self):
        """
        Return the number of Doc objects in the corpus.
        """
        return len(self.docs)


In [ ]:
import os
from spacy.tokens import DocBin
from spacy_conll.parser import ConllParser
import pandas as pd

if mode == "build":
    spacy_corpus = SpacyCorpus(nlp = conll_parser, log_file="corpus_log.txt",
                     #file_filter=["VAd_TEI_final.conllu"],
                               file_filter=".conllu",
                     directory="normalized/", verbose=True)
    spacy_corpus_docbin = spacy_corpus.to_docbin()
    spacy_corpus_docbin.to_disk("corpora/spacy_corpus_all_docbin.spacy")
    conll_parser.nlp.to_disk("corpora/spacy_corpus_all_docbin_nlp.spacy")
    #spacy_corpus_docbin.to_disk("corpora/spacy_corpus_all.spacy")
elif mode == "load":
    spacy_corpus_docbin = DocBin().from_disk("corpora/spacy_corpus_all_docbin.spacy")

In [ ]:
for doc in spacy_corpus_docbin.get_docs(nlp.vocab):
    print(doc._.metadata["doc_id"])

### Vocabulary Change

In [ ]:
import os
from spacy.tokens import DocBin
from spacy_conll.parser import ConllParser
import pandas as pd

In [ ]:
from typing import Union, List, Dict, Optional
from spacy.tokens import Doc, DocBin
from collections import Counter, defaultdict  # Added defaultdict
import pandas as pd
import fnmatch
import logging
import sys
import plotly.graph_objects as go
import matplotlib.pyplot as plt

class SpacyAnalyzer:
    def __init__(self, docs: Union[Doc, DocBin, 'SpacyCorpus'], 
                 doc_ids: Union[str, List[str]] = None,
                 doc_pattern: str = None,
                 nlp = None,
                 log_level=logging.INFO):
        
        # Setup logging
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(log_level)
        if not self.logger.handlers:
            handler = logging.StreamHandler(sys.stdout)
            formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)

        self.logger.info("Initializing SpacyAnalyzer...")
        
        # Register the custom extension if it doesn't exist
        try:
            if not Doc.has_extension("metadata"):
                self.logger.info("Registering metadata extension")
                Doc.set_extension("metadata", default={})
        except Exception as e:
            self.logger.error(f"Error registering metadata extension: {str(e)}")
            raise
            
        self.nlp = nlp
        self.logger.info("Processing input documents...")
        try:
            self.all_docs = self._process_input(docs)
            self.logger.info(f"Successfully processed {len(self.all_docs)} documents")
        except Exception as e:
            self.logger.error(f"Error processing input documents: {str(e)}")
            raise
        
        try:
            self.logger.info("Filtering documents...")
            self.doc_ids = self._filter_docs(doc_ids, doc_pattern)
            self.docs = [doc for doc in self.all_docs 
                        if doc._.metadata.get('doc_id', '') in self.doc_ids]
            self.logger.info(f"Working with {len(self.docs)} documents out of {len(self.all_docs)} total")
        except Exception as e:
            self.logger.error(f"Error during document filtering: {str(e)}")
            raise
            
    def _process_input(self, docs) -> List[Doc]:
        self.logger.info(f"Processing input of type: {type(docs)}")
        
        try:
            if isinstance(docs, Doc):
                self.logger.info("Processing single Doc")
                return [docs]
            elif isinstance(docs, DocBin):
                self.logger.info("Processing DocBin")
                if self.nlp is None:
                    raise ValueError("When using DocBin, please provide the nlp object")
                docs_list = list(docs.get_docs(self.nlp.vocab))
                self.logger.info(f"Extracted {len(docs_list)} docs from DocBin")
                return docs_list
            elif hasattr(docs, 'docs'):
                self.logger.info("Processing SpacyCorpus")
                return docs.docs
            else:
                raise ValueError(f"Unsupported input type: {type(docs)}")
        except Exception as e:
            self.logger.error(f"Error in _process_input: {str(e)}")
            raise

    def _filter_docs(self, doc_ids: Optional[Union[str, List[str]]], 
                pattern: Optional[str]) -> List[str]:
        """Filter documents with improved pattern matching"""
        try:
            all_ids = []
            self.logger.info("Getting all document IDs...")
            for i, doc in enumerate(self.all_docs):
                try:
                    doc_id = doc._.metadata.get('doc_id', f'doc_{i}')
                    all_ids.append(doc_id)
                except Exception as e:
                    self.logger.warning(f"Error getting metadata for document {i}: {str(e)}")
                    all_ids.append(f'doc_{i}')
            
            self.logger.info(f"Found {len(all_ids)} total document IDs")
            self.logger.debug(f"All document IDs: {all_ids}")
            
            if doc_ids is None and pattern is None:
                return all_ids
                
            if isinstance(doc_ids, str):
                doc_ids = [doc_ids]
                
            if doc_ids:
                valid_ids = [doc_id for doc_id in doc_ids if doc_id in all_ids]
                invalid_ids = set(doc_ids) - set(valid_ids)
                if invalid_ids:
                    self.logger.warning(f"Documents not found: {invalid_ids}")
                self.logger.info(f"Selected {len(valid_ids)} documents by ID")
                return valid_ids
                
            elif pattern:
                import re
                self.logger.info(f"Matching pattern: {pattern}")
                try:
                    regex = re.compile(pattern)
                    matched_ids = [doc_id for doc_id in all_ids if regex.search(doc_id)]
                    self.logger.info(f"Matched {len(matched_ids)} documents")
                    self.logger.debug(f"Matched documents: {matched_ids}")
                    if not matched_ids:
                        self.logger.warning(f"No documents matched pattern: {pattern}")
                    return matched_ids
                except re.error as e:
                    self.logger.error(f"Invalid regex pattern: {str(e)}")
                    raise ValueError(f"Invalid regex pattern: {str(e)}")
                    
        except Exception as e:
            self.logger.error(f"Error in _filter_docs: {str(e)}")
            raise
            
    def update_doc_selection(self, doc_ids: Union[str, List[str]] = None,
                           doc_pattern: str = None):
        self.logger.info("Updating document selection...")
        try:
            self.doc_ids = self._filter_docs(doc_ids, doc_pattern)
            self.docs = [doc for doc in self.all_docs 
                        if doc._.metadata.get('doc_id', '') in self.doc_ids]
            self.logger.info(f"Updated selection: working with {len(self.docs)} documents")
        except Exception as e:
            self.logger.error(f"Error updating document selection: {str(e)}")
            raise
            
    def get_lemma_positions(self, lemmas: Union[str, List[str]], 
                          pos: Union[str, List[str]] = None,
                          doc_id: str = None) -> Dict[str, List[Dict]]:
        self.logger.info(f"Getting positions for lemmas: {lemmas}")
        try:
            if isinstance(lemmas, str):
                lemmas = [lemmas]
            if isinstance(pos, str):
                pos = [pos]
                
            positions = {lemma: [] for lemma in lemmas}
            
            docs_to_analyze = ([doc for doc in self.docs 
                              if doc._.metadata.get('doc_id') == doc_id]
                             if doc_id else self.docs)
            
            self.logger.info(f"Analyzing {len(docs_to_analyze)} documents")
            
            for doc_idx, doc in enumerate(docs_to_analyze):
                try:
                    doc_id = doc._.metadata.get('doc_id', f'doc_{doc_idx}')
                    
                    for token_idx, token in enumerate(doc):
                        if token.lemma_ in lemmas:
                            if pos and token.pos_ not in pos:
                                continue
                                
                            positions[token.lemma_].append({
                                'doc_id': doc_id,
                                'doc_idx': doc_idx,
                                'token_idx': token_idx,
                                'token': token.text,
                                'pos': token.pos_,
                                'sent_idx': token.sent.start,
                                'sent_text': token.sent.text
                            })
                except Exception as e:
                    self.logger.error(f"Error processing document {doc_id}: {str(e)}")
                    continue
            
            total_positions = sum(len(pos) for pos in positions.values())
            self.logger.info(f"Found {total_positions} total positions")
            return positions
            
        except Exception as e:
            self.logger.error(f"Error in get_lemma_positions: {str(e)}")
            raise
    
    def get_top_words(self, n: int = 10, by: str = 'lemma',
                     pos: Union[str, List[str]] = None,
                     doc_id: str = None) -> pd.DataFrame:
        self.logger.info(f"Getting top {n} words (by={by}, pos={pos})")
        try:
            if isinstance(pos, str):
                pos = [pos]
                
            docs_to_analyze = ([doc for doc in self.docs 
                              if doc._.metadata.get('doc_id') == doc_id]
                             if doc_id else self.docs)
                
            self.logger.info(f"Analyzing {len(docs_to_analyze)} documents")
            
            counter = Counter()
            pos_dict = {}
            doc_counts = Counter()
            
            for doc in docs_to_analyze:
                doc_words = set()
                
                for token in doc:
                    if pos and token.pos_ not in pos:
                        continue
                        
                    word = token.lemma_ if by == 'lemma' else token.text
                    counter[word] += 1
                    pos_dict[word] = token.pos_
                    
                    if word not in doc_words:
                        doc_counts[word] += 1
                        doc_words.add(word)
            
            self.logger.info(f"Found {len(counter)} unique words")
            
            top_words = pd.DataFrame([
                {
                    'word': word,
                    'frequency': freq,
                    'pos': pos_dict[word],
                    'doc_count': doc_counts[word],
                    'avg_per_doc': freq / doc_counts[word]
                }
                for word, freq in counter.most_common(n)
            ])
            
            self.logger.info(f"Returning top {len(top_words)} words")
            return top_words
            
        except Exception as e:
            self.logger.error(f"Error in get_top_words: {str(e)}")
            raise
        '''
        positions = analyzer.get_lemma_positions('tempus', pos='NOUN')
        top_words = analyzer.get_top_words(n=10, pos='NOUN')
        '''
    def detect_emerged_words(self, doc_id: str = None, 
                        bin_type: str = 'tokens',
                        bin_size: int = 1000,  # for token-based
                        n_bins: int = 10,      # for percent-based
                        min_freq: int = 3,     # minimum occurrences in subsequent bins
                        min_bins: int = 2,     # minimum bins with occurrences
                        pos: Union[str, List[str]] = None) -> pd.DataFrame:
        """
        Detect words that emerge and maintain usage in a document.
        """
        self.logger.info(f"Detecting emerged words in {doc_id or 'all documents'}")
        
        if isinstance(pos, str):
            pos = [pos]
        
        emerged_patterns = []
        docs_to_analyze = ([doc for doc in self.docs 
                           if doc._.metadata.get('doc_id') == doc_id]
                          if doc_id else self.docs)
        
        for doc in docs_to_analyze:
            doc_id = doc._.metadata.get('doc_id', '')
            self.logger.info(f"Analyzing document: {doc_id}")
            
            # Filter tokens by POS if specified
            tokens = [token for token in doc if not pos or token.pos_ in pos]
            total_tokens = len(tokens)
            
            if total_tokens == 0:
                self.logger.warning(f"No tokens found in document {doc_id}")
                continue
            
            # Create bins
            if bin_type == 'tokens':
                n_bins = max(1, total_tokens // bin_size)
                bin_size = total_tokens // n_bins  # Adjust bin size to ensure equal bins
            else:  # percent-based
                bin_size = max(1, total_tokens // n_bins)  # Ensure bin size is at least 1
                n_bins = total_tokens // bin_size  # Adjust number of bins
                
            self.logger.info(f"Created {n_bins} bins with approximately {bin_size} tokens each")
            
            # Initialize word bins with explicit size
            word_bins = defaultdict(lambda: [0] * n_bins)
            
            # Count tokens in bins
            for token_idx, token in enumerate(tokens):
                bin_idx = min(token_idx // bin_size, n_bins - 1)  # Ensure bin_idx is in range
                word_bins[token.lemma_][bin_idx] += 1
            
            # Analyze emergence patterns
            for lemma, bin_counts in word_bins.items():
                try:
                    # Find first appearance
                    first_bin = next(i for i, count in enumerate(bin_counts) if count > 0)
                    
                    # Skip if too close to end
                    if first_bin > n_bins - min_bins:
                        continue
                    
                    # Check subsequent usage
                    subsequent_bins = bin_counts[first_bin + 1:first_bin + 1 + min_bins]
                    subsequent_freq = sum(subsequent_bins)
                    bins_with_word = sum(1 for count in subsequent_bins if count > 0)
                    
                    if subsequent_freq >= min_freq and bins_with_word >= min_bins:
                        # Get example contexts
                        example_tokens = [t for t in tokens if t.lemma_ == lemma][:3]
                        example_contexts = [t.sent.text for t in example_tokens]
                        
                        emerged_patterns.append({
                            'doc_id': doc_id,
                            'lemma': lemma,
                            'pos': example_tokens[0].pos_,
                            'first_bin': first_bin,
                            'first_bin_count': bin_counts[first_bin],
                            'subsequent_bins': bins_with_word,
                            'subsequent_freq': subsequent_freq,
                            'bin_pattern': bin_counts,
                            'examples': example_contexts,
                            'total_occurrences': sum(bin_counts),
                            'bins_appeared': sum(1 for count in bin_counts if count > 0),
                            'emergence_position': first_bin / n_bins if bin_type == 'percent' else first_bin * bin_size
                        })
                except (StopIteration, IndexError) as e:
                    self.logger.warning(f"Error processing lemma {lemma}: {str(e)}")
                    continue
        
        results = pd.DataFrame(emerged_patterns)
        if not results.empty:
            results = results.sort_values(['doc_id', 'emergence_position'])
        
        self.logger.info(f"Found {len(results)} emerged words")
        return results
    
    def plot_emergence_patterns(self, emerged_df: pd.DataFrame, 
                          words: List[str] = None,
                          doc_id: str = None,
                          plot_lib: str = 'plotly',
                          show_legend: bool = True):
        """
        Visualize emergence patterns with both hover info and optional legend panel.
        
        Parameters:
        emerged_df: DataFrame from detect_emerged_words()
        words: List of specific words to plot (None for all)
        doc_id: Specific document to plot (None for all)
        plot_lib: 'plotly' or 'seaborn'
        show_legend: Whether to show the legend panel
        """
        df = emerged_df.copy()
        
        if doc_id:
            df = df[df['doc_id'] == doc_id]
        if words:
            df = df[df['lemma'].isin(words)]
            
        if df.empty:
            self.logger.warning("No data to plot")
            return
        
        if plot_lib == 'plotly':
            fig = go.Figure()
            
            for _, row in df.iterrows():
                # Create hover text with word and stats
                hover_text = [
                    f"Word: {row['lemma']} ({row['pos']})<br>" +
                    f"Bin: {i}<br>" +
                    f"Count: {count}<br>" +
                    f"Total occurrences: {row['total_occurrences']}<br>" +
                    f"Total bins appeared: {row['bins_appeared']}<br>" +
                    f"First appeared in bin: {row['first_bin']}"
                    for i, count in enumerate(row['bin_pattern'])
                ]
                
                fig.add_trace(go.Scatter(
                    x=list(range(len(row['bin_pattern']))),
                    y=row['bin_pattern'],
                    name=f"{row['lemma']} ({row['pos']})",  # include POS in legend
                    mode='lines+markers',
                    hoverinfo='text',
                    hovertext=hover_text,
                    line=dict(width=2),
                    marker=dict(
                        size=8,
                        symbol='circle' if row['pos'] == 'NOUN' else 'square'
                    ),
                    showlegend=show_legend
                ))
                
            fig.update_layout(
                title="Word Emergence Patterns",
                xaxis_title="Bin Number",
                yaxis_title="Word Count",
                hovermode='closest',
                showlegend=show_legend,
                legend=dict(
                    x=1.05,
                    y=1,
                    xanchor='left',
                    bgcolor='rgba(255, 255, 255, 0.8)',
                    bordercolor='rgba(0, 0, 0, 0.2)',
                    borderwidth=1
                ),
                height=600,
                width=1000 if not show_legend else 1200  # wider if showing legend
            )
            
            # Update margins to accommodate legend
            if show_legend:
                fig.update_layout(margin=dict(r=150))
            
            return fig
            
        else:  # seaborn
            plt.figure(figsize=(12 if not show_legend else 15, 6))
            
            for _, row in df.iterrows():
                plt.plot(row['bin_pattern'], 
                        label=f"{row['lemma']} ({row['pos']})",
                        marker='o')
                
            plt.title("Word Emergence Patterns")
            plt.xlabel("Bin Number")
            plt.ylabel("Word Count")
            if show_legend:
                plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            
            return plt.gcf()
    def detect_terms_by_emergence_pattern(self, 
                                          start_bin: int,
                                          end_bin: int = None,
                                          min_freq: int = 3, 
                                          min_bins: int = 2,
                                          pos: Union[str, List[str]] = None,
                                          doc_id: str = None) -> pd.DataFrame:
        """
        Detect terms that:
        1. Start appearing in a given bin range (start_bin to end_bin)
        2. Maintain a minimum frequency (min_freq) in one or more subsequent bins (min_bins)
        
        Parameters:
        start_bin (int): The bin index where the term must first appear
        end_bin (int, optional): The bin index up to which the term must maintain frequency
        min_freq (int, optional): Minimum frequency the term must have in subsequent bins
        min_bins (int, optional): Minimum number of subsequent bins the term must appear in
        pos (Union[str, List[str]], optional): Part-of-speech filter
        doc_id (str, optional): Specific document to analyze
        
        Returns:
        pd.DataFrame: Detected terms with their emergence patterns
        """
        self.logger.info(f"Detecting terms with emergence pattern: start_bin={start_bin}, end_bin={end_bin}")

        if isinstance(pos, str):
            pos = [pos]
        
        detected_terms = []
        docs_to_analyze = ([doc for doc in self.docs 
                           if doc._.metadata.get('doc_id') == doc_id]
                          if doc_id else self.docs)
        
        for doc in docs_to_analyze:
            doc_id = doc._.metadata.get('doc_id', '')
            self.logger.info(f"Analyzing document: {doc_id}")
            
            # Filter tokens by POS if specified
            tokens = [token for token in doc if not pos or token.pos_ in pos]
            total_tokens = len(tokens)
            
            if total_tokens == 0:
                self.logger.warning(f"No tokens found in document {doc_id}")
                continue
            
            # Create bins
            bin_size = max(1, total_tokens // 10)  # 10 bins by default
            n_bins = total_tokens // bin_size
            
            self.logger.info(f"Created {n_bins} bins with approximately {bin_size} tokens each")
            
            # Count tokens in bins
            word_bins = defaultdict(lambda: [0] * n_bins)
            for token_idx, token in enumerate(tokens):
                bin_idx = min(token_idx // bin_size, n_bins - 1)  # Ensure bin_idx is in range
                word_bins[token.lemma_][bin_idx] += 1
            
            # Analyze emergence patterns
            for lemma, bin_counts in word_bins.items():
                try:
                    # Find first appearance
                    first_bin = next(i for i, count in enumerate(bin_counts) if count > 0)
                    
                    # Check if first appearance is in the target bin range
                    if first_bin < start_bin:
                        continue
                    
                    # Check subsequent usage
                    if end_bin is None:
                        end_bin = n_bins - 1
                    subsequent_bins = bin_counts[first_bin + 1:end_bin + 1]
                    subsequent_freq = sum(subsequent_bins)
                    bins_with_word = sum(1 for count in subsequent_bins if count > 0)
                    
                    if subsequent_freq >= min_freq and bins_with_word >= min_bins:
                        # Get example contexts
                        example_tokens = [t for t in tokens if t.lemma_ == lemma][:3]
                        example_contexts = [t.sent.text for t in example_tokens]
                        
                        detected_terms.append({
                            'doc_id': doc_id,
                            'lemma': lemma,
                            'pos': example_tokens[0].pos_,
                            'first_bin': first_bin,
                            'subsequent_bins': bins_with_word,
                            'subsequent_freq': subsequent_freq,
                            'bin_pattern': bin_counts,
                            'examples': example_contexts,
                            'total_occurrences': sum(bin_counts),
                            'bins_appeared': sum(1 for count in bin_counts if count > 0),
                            'emergence_position': first_bin
                        })
                except (StopIteration, IndexError) as e:
                    self.logger.warning(f"Error processing lemma {lemma}: {str(e)}")
                    continue
        
        results = pd.DataFrame(detected_terms)
        if not results.empty:
            results = results.sort_values(['doc_id', 'emergence_position'])
        
        self.logger.info(f"Found {len(results)} terms with the specified emergence pattern")
        return results


In [ ]:
import spacy
from spacy.tokens import DocBin
spacy_corpus_docbin = DocBin().from_disk("corpora/spacy_corpus_all_docbin.spacy")
analyzer = SpacyAnalyzer(spacy_corpus_docbin, doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
                         nlp=spacy.load("corpora/spacy_corpus_all_docbin_nlp.spacy"))

In [ ]:
#top_words = analyzer.get_top_words(n=10, pos='NOUN')

In [ ]:
#positions = analyzer.get_lemma_positions()

#### Emerging words

In [ ]:
emerging_words = {}
for doc in analyzer.doc_ids:
    emerging_words[doc] = {}
    # Token-based binning
    for POS in ['NOUN', 'VERB', 'ADJ']:
        
        emerged = analyzer.detect_emerged_words(
            doc_id=doc,
            bin_type='percent',
            bin_size=1000,
            min_freq=5,
            min_bins=5,
            pos=POS
        )
        emerging_words[doc][POS] = emerged

In [ ]:
# Token-based binning
emerged = analyzer.detect_emerged_words(
    doc_id='AKapSąd1Pozn_TEI_final.conllu',
    bin_type='tokens',
    bin_size=1000,
    min_freq=3,
    min_bins=5,
    pos='NOUN'
)

In [ ]:
# Percentage-based binning
emerged = analyzer.detect_emerged_words(
    doc_id='AKapSąd1Pozn_TEI_final.conllu',
    bin_type='percent',
    n_bins=10,
    min_freq=3, #at least min_freq occurrences in min_bins
    min_bins=2, #at least in min_bins after the first occurrence
    pos='NOUN'
)

In [ ]:
for doc in emerging_words.keys():
    for pos in emerging_words[doc].keys():
        fig = analyzer.plot_emergence_patterns(emerging_words[doc][pos])
        print(doc)
        fig.show()

In [ ]:
# Detect terms that start appearing in bin 2 and maintain frequency in at least 3 subsequent bins
emerging_terms = {}
for doc in analyzer.doc_ids:
    emerging_terms[doc] = {}
    # Token-based binning
    for POS in ['NOUN', 'VERB', 'ADJ']:
        emerging_terms[doc][POS] = {}
        for position in range(0,8):
            emerging_terms[doc][POS][position] = None
            
            #emerging_terms[doc][POS] = {}.setdefault(str(position))
            emerged = analyzer.detect_terms_by_emergence_pattern(
                #doc_id=doc,
                start_bin=position,
                #bin_type='percent',
                #bin_size=1000,
                min_freq=5,
                min_bins=3,
                pos=POS
            )
            emerging_terms[doc][POS][position] = emerged

In [ ]:
import pickle
with open('out/emerging_terms.pickle', 'wb') as f:
    pickle.dump(emerging_terms,file=f)

In [ ]:
import pickle
with open('out/emerging_terms.pickle', 'rb') as f:
    emerging_terms = pickle.load(f)

In [ ]:
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def handle_multiple_plots(emerging_terms, output_dir=None, save_individual=False, create_combined=True):
    """
    Handle multiple emergence pattern plots with options to save individually or create combined views.
    
    Args:
        emerging_terms (dict): Dictionary containing the emergence terms data
        output_dir (str): Directory to save plots (if None, plots won't be saved)
        save_individual (bool): Whether to save individual plots
        create_combined (bool): Whether to create combined plots per document-POS pair
    
    Returns:
        dict: Dictionary of generated figures
    """
    figures = {}
    
    # Create output directory if it doesn't exist
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for doc in emerging_terms.keys():
        figures[doc] = {}
        for pos in emerging_terms[doc].keys():
            figures[doc][pos] = {}
            
            if create_combined:
                # Create a subplot figure for all positions in this doc-POS pair
                valid_positions = [p for p in range(8) if emerging_terms[doc][pos][p] is not None]
                if valid_positions:
                    rows = (len(valid_positions) + 1) // 2  # 2 columns
                    combined_fig = make_subplots(
                        rows=rows,
                        cols=2,
                        subplot_titles=[f'Position {p}' for p in valid_positions]
                    )
            
            for position in range(8):
                try:
                    print(f'Processing - Document: {doc}, POS: {pos}, First bin: {position}')
                    
                    # Generate the figure
                    fig = analyzer.plot_emergence_patterns(emerging_terms[doc][pos][position])
                    figures[doc][pos][position] = fig
                    
                    # Save individual figure if requested
                    if save_individual and output_dir:
                        filename = f"{doc}_{pos}_bin{position}.html"
                        fig.write_html(os.path.join(output_dir, filename))
                    
                    # Add to combined figure if requested
                    if create_combined and emerging_terms[doc][pos][position] is not None:
                        subplot_idx = valid_positions.index(position)
                        row = subplot_idx // 2 + 1
                        col = subplot_idx % 2 + 1
                        
                        # Add traces from individual figure to combined figure
                        for trace in fig.data:
                            combined_fig.add_trace(trace, row=row, col=col)
                
                except Exception as e:
                    print(f'Error processing - Document: {doc}, POS: {pos}, First bin: {position}')
                    print(f'Error message: {str(e)}')
                    figures[doc][pos][position] = None
            
            # Save combined figure if created
            if create_combined and output_dir and any(figures[doc][pos].values()):
                combined_filename = f"{doc}_{pos}_combined.html"
                combined_fig.update_layout(height=400*rows, width=1000, 
                                        title=f'Emergence Patterns - {doc} - {pos}')
                combined_fig.write_html(os.path.join(output_dir, combined_filename))
                figures[doc][pos]['combined'] = combined_fig
    
    return figures

# Usage example:
output_directory = "out/emergence_plots"
figures = handle_multiple_plots(
    emerging_terms,
    output_dir=output_directory,
    save_individual=True,
    create_combined=True
)

# To display a specific plot later:
# figures['doc_name']['POS'][position].show()

# To display a combined plot:
# figures['doc_name']['POS']['combined'].show()

#### Spacy corpus as a df

In [ ]:
if mode == "build":
    corpus_df_l = []
    for doc in spacy_corpus:
        doc_df = pd.DataFrame([{        
            "token": token.text,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "tag": token.tag_,
            "morph": token.morph.to_dict(),;
            "shape": token.shape_,
            "is_alpha": token.is_alpha,
            "is_stop": token.is_stop
        } for token in doc])
        doc_df["doc_id"] = doc._.metadata["doc_id"]
        corpus_df_l.append(doc_df)
    corpus_df = pd.concat(corpus_df_l, ignore_index=True)
    corpus_df = corpus_df.join(pd.json_normalize(corpus_df["morph"])) #explode on morph dictionary
    corpus_df["doc_id"] = corpus_df["doc_id"].str.replace('_final.conllu','')
    corpus_df.drop("morph", axis=1, inplace=True)
    corpus_df = corpus_df.astype('category')
    #corpus_df.to_pickle("corpora/spacy_corpus_all_df.pkl")
    corpus_df.to_parquet("corpora/spacy_corpus_all_df.parquet")
elif mode == "load":
    import pandas as pd
    corpus_df = pd.read_parquet("corpora/spacy_corpus_all_df.parquet")

In [ ]:
if mode == "build":
    corpus_df_counts = corpus_df.groupby(['doc_id'], observed=True).size().reset_index(name='total')
    corpus_df_features = ['pos', 'doc_id',
               'Case', 'Gender', 'Number', 'Degree', 'Mood', 'Person', 'Tense',
               'VerbForm', 'Voice', 'Aspect', 'NumForm', 'Abbr']
    corpus_df_freq = corpus_df[corpus_df_features].groupby(corpus_df_features, observed=True, dropna=False, as_index=False).size().reset_index()
    corpus_df_freq.rename(columns={'size':'count'}, inplace=True)
    #oldcorpus_df_freq = corpus_df[corpus_df_features].groupby("doc_id", observed=True).value_counts().reset_index(name="count")
    corpus_df_freq = corpus_df_freq.merge(corpus_df_counts, on="doc_id")
    corpus_df_freq["count_rel"] = corpus_df_freq["count"] / corpus_df_freq["total"]
    corpus_df_freq.to_pickle("corpora/spacy_corpus_all_df_freq.pkl")
    
elif mode == "load":
    corpus_df_freq = pd.read_parquet("corpora/spacy_corpus_all_df_freq.parquet")

### Ortography

In [ ]:
import pandas as pd
import re
import logging
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Define variation patterns
variation_patterns = {
    'ae_e': {
        'standard': 'ae',
        'variants': ['e', 'ę'],
        'context': 'any',
        'description': 'ae monophthongization'
    },
    'oe_e': {
        'standard': 'oe',
        'variants': ['e', 'ę'],
        'context': 'any',
        'description': 'oe monophthongization'
    },
    'ti_ci': {
        'standard': 'ti',
        'variants': ['ci', 'cii'],
        'context': 'before_vowel',
        'description': 'ti palatalization'
    },
    'i_j': {
        'standard': 'i',
        'variants': ['j'],
        'context': 'before_vowel',
        'description': 'i/j variation'
    },
    'ph_f': {
        'standard': 'ph',
        'variants': ['f'],
        'context': 'any',
        'description': 'ph/f variation'
    },
    'th_t': {
        'standard': 'th',
        'variants': ['t'],
        'context': 'any',
        'description': 'th/t variation'
    },
     'd_t': {
        'standard': 'd',
        'variants': ['t'],
        'context': 'any',
        'description': 'd/t variation'
    },
    'ch_c': {
        'standard': 'ch',
        'variants': ['k', 'c', 'h'],
        'context': 'any',
        'description': 'ch/ variation'
    },
     'cc_c': {
        'standard': 'cc',
        'variants': ['c'],
        'context': 'any',
        'description': 'cc/c variation'
    },
    'z_s': {
        'standard': 'z',
        'variants': ['s'],
        'context': 'any',
        'description': 'z/s variation'
    },
    'i_*': {
        'standard': 'i',
        'variants': ['y', 'ii'],
        'context': 'any',
        'description': 'i/y variation'
    },
     'h_': {
        'standard': 'h',
        'variants': [''],
        'context': 'before_vowel',
        'description': 'h/ variation'
    },
     'mn_mpn': {
        'standard': 'mn',
        'variants': ['mpn'],
        'context': 'any',
        'description': 'mn/mpn variation'
    },
     'gn_ngn': {
        'standard': 'gn',
        'variants': ['ngn'],
        'context': 'any',
        'description': 'gn/ngn variation'
    },
      'f_': {
        'standard': 'f',
        'variants': ['ff', 'ph'],
        'context': 'any',
        'description': 'f/ variation'
    },
    'l_': {
        'standard': 'l',
        'variants': ['ll'],
        'context': 'any',
        'description': 'l/ variation'
    }
}

def check_context(context, position, word):
    """Check if the variation context is valid"""
    if context == 'any':
        return True
    elif context == 'before_vowel':
        next_pos = position + 2
        return next_pos < len(word) and word[next_pos] in 'aeiou'
    return False

def analyze_variations(df):
    """Analyze orthographic variations in the corpus"""
    results = []
    total_rows = len(df)
    
    for index, row in df.iterrows():
        if (index + 1) % 1000 == 0:
            logging.info(f"Processing {index + 1}/{total_rows} rows ({((index + 1)/total_rows * 100):.1f}%) - Current token: '{row['token']}'")
        
        token = row['token'].lower()
        lemma = row['lemma'].lower()
        
        for var_type, pattern in variation_patterns.items():
            # Find standard form in lemma
            matches = re.finditer(pattern['standard'], lemma)
            
            for match in matches:
                pos = match.start()
                
                if not check_context(pattern['context'], pos, lemma):
                    continue
                
                # Check if the position exists in token
                if pos + len(pattern['standard']) <= len(token):
                    token_form = token[pos:pos + len(pattern['standard'])]
                    
                    # Check if there's an orthographic variation
                    if token_form in pattern['variants']:
                        # Found actual orthographic variation
                        results.append({
                            'doc_id': row['doc_id'],
                            'token': row['token'],
                            'lemma': row['lemma'],
                            'feature': pattern['standard'],
                            'variation': token_form,
                            'position': pos,
                            'type': var_type,
                            'description': pattern['description']
                        })
    
    logging.info(f"Processing completed. Found {len(results)} variations in {total_rows} rows.")
    return pd.DataFrame(results)

def find_variations_by_lemma(df, lemma_pattern):
    """Find variations for specific lemmas"""
    mask = df['lemma'].str.contains(lemma_pattern, case=False, na=False)
    matching_rows = df[mask]
    logging.info(f"Found {len(matching_rows)} matching lemmas. Starting analysis...")
    return analyze_variations(matching_rows)

In [ ]:
corpus_df_orthography_all = analyze_variations(corpus_df)

In [ ]:
corpus_df_orthography_all.to_parquet("out/corpus_df_orthography_all.parquet")

In [ ]:
corpus_df_orthography_all = pd.read_parquet("out/corpus_df_orthography_all.parquet")

In [ ]:
corpus_df_orthography_all["variation"].value_counts()

In [ ]:
corpus_df_orthography_all[corpus_df_orthography_all["variation"].notnull()]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.manifold import TSNE
import umap
import prince
import matplotlib.pyplot as plt
import seaborn as sns
import re

def get_cluster_centers(plot_data):
    """Calculate cluster centers from plot data"""
    return plot_data.groupby('Cluster')[['Dim1', 'Dim2']].mean()
def filter_documents(corpus_df, doc_ids=None, doc_pattern=None):
    if doc_ids is not None:
        return corpus_df[corpus_df['doc_id'].isin(doc_ids)]
    elif doc_pattern is not None:
        pattern = re.compile(doc_pattern)
        matching_docs = [doc_id for doc_id in corpus_df['doc_id'].unique() 
                        if pattern.match(str(doc_id))]
        return corpus_df[corpus_df['doc_id'].isin(matching_docs)]
    return corpus_df  
      
    
def plot_variation_clusters(variations_df, columns=None, doc_ids=None, doc_pattern=None, 
                          n_clusters=3, clustering_method='kmeans', 
                          dim_reduction='pca',
                          plot_centers=True,
                          plot_features=True, n_top_features=5,
                          perplexity=30,
                          n_neighbors=15,
                          min_dist=0.1,
                          dbscan_eps=0.5, dbscan_min_samples=5,
                          random_state=42):
    """
    Create and analyze clusters with multiple dimensionality reduction methods
    """
    if columns is None:
        columns = ['feature', 'variation']

    #filtered_df = variations_df
    filtered_df = filter_documents(variations_df, doc_ids, doc_pattern)
    
    if filtered_df.empty:
        raise ValueError("No documents found after filtering")
        
    print("Analyzing documents:", sorted(filtered_df['doc_id'].unique()))
    
    # Prepare feature matrix
    feature_matrix = pd.crosstab(
        filtered_df['doc_id'],
        [filtered_df[col] for col in columns],
        normalize='index'
    )
    
    new_columns = []
    for col in feature_matrix.columns:
        if isinstance(col, tuple):
            label = '_'.join(str(x) for x in col if pd.notna(x))
        else:
            label = str(col)
        new_columns.append(label)
    feature_matrix.columns = new_columns
    
    # Scale features for all methods except MCA
    if dim_reduction != 'mca':
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(feature_matrix)
    else:
        scaled_features = feature_matrix

    # Dimensionality reduction
    if dim_reduction == 'pca':
        reducer = PCA(n_components=2, random_state=random_state)
        reduced_features = reducer.fit_transform(scaled_features)
        
    elif dim_reduction == 'mca':
        # Określamy maksymalną możliwą liczbę komponentów
        n_components = min(2, len(feature_matrix.columns) - 1)
        reducer = prince.MCA(n_components=n_components, random_state=random_state)
        reduced_features = reducer.fit_transform(feature_matrix).values
        scaled_features = feature_matrix  #
        
    elif dim_reduction == 'tsne':
        reducer = TSNE(n_components=2, perplexity=perplexity, random_state=random_state)
        reduced_features = reducer.fit_transform(scaled_features)
        
    elif dim_reduction == 'umap':
        reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, 
                          min_dist=min_dist, random_state=random_state)
        reduced_features = reducer.fit_transform(scaled_features)
    
    # Clustering
    n_docs = len(feature_matrix)
    if n_docs < n_clusters and clustering_method != 'dbscan':
        print(f"Warning: Reducing number of clusters to {n_docs}")
        n_clusters = n_docs
    
    if clustering_method == 'kmeans':
        clusterer = KMeans(n_clusters=n_clusters, random_state=random_state)
        clusters = clusterer.fit_predict(scaled_features)
    elif clustering_method == 'dbscan':
        clusterer = DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples)
        clusters = clusterer.fit_predict(scaled_features)
    elif clustering_method == 'hierarchical':
        clusterer = AgglomerativeClustering(n_clusters=n_clusters)
        clusters = clusterer.fit_predict(scaled_features)
    
    # Reszta funkcji pozostaje bez zmian...
    
    # Create plot data
    plot_data = pd.DataFrame({
        'Dim1': reduced_features[:, 0],
        'Dim2': reduced_features[:, 1],
        'Cluster': clusters,
        'Document': feature_matrix.index
    })
    
    # Visualization
    plt.figure(figsize=(15, 10))
    
    # Plot points with different style for DBSCAN noise points
    if clustering_method == 'dbscan':
        noise_mask = clusters == -1
        if noise_mask.any():
            noise_data = plot_data[noise_mask]
            plt.scatter(noise_data['Dim1'], noise_data['Dim2'], 
                       c='gray', marker='x', s=100, label='Noise',
                       alpha=0.5)
        clustered_data = plot_data[~noise_mask]
        sns.scatterplot(data=clustered_data,
                       x='Dim1', y='Dim2',
                       hue='Cluster', style='Cluster',
                       s=100, palette='deep')
    else:
        sns.scatterplot(data=plot_data,
                       x='Dim1', y='Dim2',
                       hue='Cluster', style='Cluster',
                       s=100, palette='deep')
    
    # Plot cluster centers if requested
    if plot_centers and clustering_method != 'dbscan':
        centers = get_cluster_centers(plot_data)
        plt.scatter(centers['Dim1'], centers['Dim2'], 
                   c='red', marker='*', s=200, 
                   label='Cluster Centers', zorder=5)
    
    # Add document labels - poprawiona sekcja
    for idx, row in plot_data.iterrows():
        label = str(row['Document'])  # Upewniamy się, że etykieta jest stringiem
        x, y = row['Dim1'], row['Dim2']
        plt.annotate(
            label,
            (x, y),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=8,
            alpha=0.7,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.7),  # Dodajemy tło dla lepszej czytelności
            zorder=6  # Upewniamy się, że etykiety są na wierzchu
        )
    
    # Add feature vectors (only for PCA and MCA)
    if plot_features and dim_reduction in ['pca', 'mca']:
        if dim_reduction == 'pca':
            loadings = reducer.components_.T * np.sqrt(reducer.explained_variance_)
            feature_importance = np.sum(loadings**2, axis=1)
            n_features = min(n_top_features, len(feature_importance))
            top_indices = np.argsort(feature_importance)[-n_features:]
        else:  # MCA
            # Pobieramy współrzędne kolumn dla MCA
            coord = reducer.column_coordinates(feature_matrix)
            loadings = coord.values
            # Obliczamy ważność cech na podstawie współrzędnych
            feature_importance = np.sum(loadings**2, axis=1)
            n_features = min(n_top_features, len(feature_importance))
            top_indices = np.argsort(feature_importance)[-n_features:]
            
        scale_factor = 1.5
        for idx in top_indices:
            if idx >= len(feature_matrix.columns):
                continue
            feature_name = feature_matrix.columns[idx]
            x = loadings[idx, 0] * scale_factor
            y = loadings[idx, 1] * scale_factor
            
            plt.arrow(0, 0, x, y, color='darkred', alpha=0.5,
                     head_width=0.05, head_length=0.1, fc='darkred', ec='darkred',
                     zorder=4)
            
            text_x = x * 1.1
            text_y = y * 1.1
            
            plt.text(text_x, text_y, feature_name,
                    color='darkred', ha='center', va='center',
                    fontsize=8, fontweight='bold',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7),
                    zorder=7)
    
    # Add title and labels
    method_name = dim_reduction.upper()
    dimension_labels = {
        'pca': ('First Principal Component', 'Second Principal Component'),
        'mca': ('First MCA Dimension', 'Second MCA Dimension'),
        'tsne': ('t-SNE Dimension 1', 't-SNE Dimension 2'),
        'umap': ('UMAP Dimension 1', 'UMAP Dimension 2')
    }

    
    column_str = ', '.join(columns) if columns else 'all features'
    plt.title(f'Document Clusters Based on {column_str}\n'
              f'Method: {clustering_method}, Reduction: {method_name}', 
              pad=20)
    plt.xlabel(dimension_labels[dim_reduction][0])
    plt.ylabel(dimension_labels[dim_reduction][1])
    plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    
    return plot_data, feature_matrix, clusterer, reducer

In [ ]:
def analyze_and_report_clusters(variations_df, columns=None, doc_ids=None, doc_pattern=None, 
                              n_clusters=3, clustering_method='kmeans', 
                              dim_reduction='pca',
                              plot_centers=True,
                              plot_features=True, n_top_features=5,
                              perplexity=30,
                              n_neighbors=15,
                              min_dist=0.1,
                              dbscan_eps=0.5, dbscan_min_samples=5,
                              random_state=42):
    try:
        plot_data, feature_matrix, clusterer, reducer = plot_variation_clusters(
            variations_df, 
            columns=columns,
            doc_ids=doc_ids, 
            doc_pattern=doc_pattern,
            n_clusters=n_clusters,
            clustering_method=clustering_method,
            dim_reduction=dim_reduction,
            plot_centers=plot_centers,
            plot_features=plot_features,
            n_top_features=n_top_features,
            perplexity=perplexity,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            dbscan_eps=dbscan_eps,
            dbscan_min_samples=dbscan_min_samples,
            random_state=random_state
        )
        
        # Get cluster labels
        labels = clusterer.labels_
        if clustering_method == 'dbscan':
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        
        print("\nAnalysis Details:")
        print("----------------")
        print(f"Dimensionality Reduction: {dim_reduction.upper()}")
        print(f"Clustering Method: {clustering_method}")
        print(f"Number of clusters: {n_clusters}")
        
        # Method-specific information
        if dim_reduction == 'pca':
            var_explained = reducer.explained_variance_ratio_
            print("\nPCA Variance Explained:")
            print(f"Total (2 components): {sum(var_explained):.1%}")
            print(f"Component 1: {var_explained[0]:.1%}")
            print(f"Component 2: {var_explained[1]:.1%}")
        
        elif dim_reduction == 'mca':
            print("\nMCA Analysis:")
            try:
                eigenvalues = reducer.eigenvalues_
                if hasattr(reducer, 'total_inertia_'):
                    # Obliczamy proporcje wyjaśnionej inercji
                    ratios = eigenvalues / reducer.total_inertia_
                    print(f"Total inertia explained: {sum(ratios):.1%}")
                    for i, ratio in enumerate(ratios):
                        print(f"Dimension {i+1}: {ratio:.1%}")
                else:
                    # Jeśli nie ma total_inertia_, pokazujemy surowe wartości własne
                    print("Eigenvalues:")
                    for i, val in enumerate(eigenvalues):
                        print(f"Dimension {i+1}: {val:.3f}")
            except Exception as e:
                print(f"Could not calculate MCA statistics: {str(e)}")
        
        # Feature importance analysis (for PCA and MCA)
        if dim_reduction in ['pca', 'mca'] and plot_features:
            print("\nTop Contributing Features:")
            if dim_reduction == 'pca':
                loadings = reducer.components_.T * np.sqrt(reducer.explained_variance_)
                feature_importance = np.sum(loadings**2, axis=1)
            else:  # MCA
                coord = reducer.column_coordinates(feature_matrix)
                loadings = coord.values
                feature_importance = np.sum(loadings**2, axis=1)
            
            top_indices = np.argsort(feature_importance)[-n_top_features:]
            for idx in reversed(top_indices):
                if idx < len(feature_matrix.columns):
                    feature_name = feature_matrix.columns[idx]
                    importance = feature_importance[idx]
                    print(f"\n{feature_name}:")
                    print(f"  Overall importance: {importance:.3f}")
                    if dim_reduction == 'pca':
                        print(f"  Contribution to PC1: {loadings[idx, 0]:.3f}")
                        print(f"  Contribution to PC2: {loadings[idx, 1]:.3f}")
                    else:
                        print(f"  Contribution to MCA1: {loadings[idx, 0]:.3f}")
                        print(f"  Contribution to MCA2: {loadings[idx, 1]:.3f}")

        # Cluster analysis
        print("\nCluster Details:")
        print("---------------")
        analysis_df = feature_matrix.copy()
        analysis_df['Cluster'] = labels
        
        if clustering_method == 'dbscan':
            noise_points = (labels == -1).sum()
            print(f"\nNumber of noise points: {noise_points}")
        
        for cluster in sorted(set(labels)):
            if clustering_method == 'dbscan' and cluster == -1:
                print("\nNoise points:")
                cluster_docs = plot_data[plot_data['Cluster'] == -1]['Document'].tolist()
                print("Documents:", ", ".join(cluster_docs))
                continue
            
            print(f"\nCluster {cluster}:")
            cluster_docs = plot_data[plot_data['Cluster'] == cluster]['Document'].tolist()
            print("Documents:", ", ".join(cluster_docs))
            print(f"Size: {len(cluster_docs)} documents")
            
            cluster_data = analysis_df[analysis_df['Cluster'] == cluster]
            cluster_means = cluster_data.drop('Cluster', axis=1).mean()
            
            top_features = cluster_means.sort_values(ascending=False)[:3]
            print("Characteristic features:")
            for feature, value in top_features.items():
                overall_mean = feature_matrix[feature].mean()
                diff = value - overall_mean
                print(f"  - {feature}: {value:.3f} (diff from mean: {diff:+.3f})")
        
        return plot_data, feature_matrix, clusterer, reducer
        
    except ValueError as e:
        print(f"Error: {e}")
        return None, None, None, None


In [ ]:
        
def list_available_documents(variations_df):
    """List all available document IDs and any patterns found"""
    docs = sorted(variations_df['doc_id'].unique())
    
    print("Available documents:", len(docs))
    print("\nFirst few document IDs:")
    for doc in docs[:5]:
        print(f"  - {doc}")
    if len(docs) > 5:
        print(f"  ... and {len(docs)-5} more")
    
    # Try to detect patterns in doc_ids
    patterns = set()
    for doc in docs:
        prefix = re.match(r'^([a-zA-Z_]+)', doc)
        if prefix:
            patterns.add(prefix.group(1))
    
    if patterns:
        print("\nDetected patterns:")
        for pattern in sorted(patterns):
            print(f"  - {pattern}*")
    
    return docs

In [ ]:
corpus_df_orthography_all['doc_id'] = corpus_df_orthography_all['doc_id'].str.replace('_TEI', '')

In [ ]:
plot_data, feature_matrix, clusterer, reducer = analyze_and_report_clusters(
    corpus_df_orthography_all, 
    #doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    plot_features=True, n_top_features=5, n_clusters=8
)

In [ ]:
plot_data, feature_matrix, clusterer, reducer = analyze_and_report_clusters(
    corpus_df_orthography_all, 
    doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    clustering_method='hierarchical', n_clusters=5
)

In [ ]:
import pandas as pd
import numpy as np

def analyze_document_features(feature_matrix, doc_id):
    """
    Analyze how a specific document's features differ from the mean
    
    Parameters:
    feature_matrix: DataFrame with documents as index and features as columns
    doc_id: ID of the document to analyze
    
    Returns:
    DataFrame with feature differences sorted by absolute difference
    """
    # Get document's features and overall means
    doc_features = feature_matrix.loc[doc_id]
    mean_features = feature_matrix.mean()
    
    # Calculate differences
    differences = pd.DataFrame({
        'Document_Value': doc_features,
        'Mean_Value': mean_features,
        'Difference': doc_features - mean_features
    })
    
    # Sort by absolute difference
    differences['Abs_Difference'] = differences['Difference'].abs()
    differences = differences.sort_values('Abs_Difference', ascending=False)
    
    return differences

def analyze_outliers(plot_data, feature_matrix, n_outliers=3, method='distance'):
    """
    Identify and analyze outlier documents
    
    Parameters:
    plot_data: DataFrame with PCA coordinates and cluster assignments
    feature_matrix: Original feature matrix
    n_outliers: Number of outliers to analyze
    method: 'distance' (from center) or 'cluster' (from cluster center)
    
    Returns:
    DataFrame with outlier analysis
    """
    if method == 'distance':
        # Calculate distance from origin (0,0) in PCA space
        plot_data['Distance'] = np.sqrt(plot_data['Dim1']**2 + plot_data['Dim2']**2)
        outliers = plot_data.nlargest(n_outliers, 'Distance')
    else:
        # Calculate distance from cluster center
        plot_data['Distance_From_Cluster'] = plot_data.apply(
            lambda row: np.sqrt(
                (row['PC1'] - plot_data[plot_data['Cluster'] == row['Cluster']]['Dim1'].mean())**2 +
                (row['PC2'] - plot_data[plot_data['Cluster'] == row['Cluster']]['Dim2'].mean())**2
            ),
            axis=1
        )
        outliers = plot_data.nlargest(n_outliers, 'Distance_From_Cluster')
    
    # Analyze each outlier
    analyses = []
    for _, outlier in outliers.iterrows():
        doc_id = outlier['Document']
        diff_analysis = analyze_document_features(feature_matrix, doc_id)
        
        # Get top differentiating features
        top_features = diff_analysis.head(5)
        
        analyses.append({
            'Document': doc_id,
            'Cluster': outlier['Cluster'],
            'Distance': outlier['Distance'] if method == 'distance' else outlier['Distance_From_Cluster'],
            'Top_Differentiating_Features': top_features
        })
    
    return analyses

# Przykład użycia:
def print_outlier_analysis(analyses):
    """
    Print readable outlier analysis
    """
    print("\nOutlier Analysis:")
    print("-----------------")
    for analysis in analyses:
        print(f"\nDocument: {analysis['Document']}")
        print(f"Cluster: {analysis['Cluster']}")
        print(f"Distance: {analysis['Distance']:.3f}")
        print("\nTop differentiating features:")
        
        features = analysis['Top_Differentiating_Features']
        for feature in features.index:
            print(f"  {feature}:")
            print(f"    Document value: {features.loc[feature, 'Document_Value']:.3f}")
            print(f"    Mean value: {features.loc[feature, 'Mean_Value']:.3f}")
            print(f"    Difference: {features.loc[feature, 'Difference']:.3f}")

In [ ]:
#Analiza outlierów - od wykresu
outlier_analysis = analyze_outliers(plot_data, feature_matrix, n_outliers=3, method='distance')
print_outlier_analysis(outlier_analysis)

In [ ]:
#Analiza outlierów - od centrów
cluster_outlier_analysis = analyze_outliers(plot_data, feature_matrix, n_outliers=3)
print_outlier_analysis(cluster_outlier_analysis)

### Syntax

In [ ]:
corpus_df = pd.read_parquet("corpora/spacy_corpus_all_df.parquet")
corpus_df_syntax_conj = corpus_df[corpus_df["pos"].isin(['CCONJ', 'SCONJ'])]

In [ ]:
n = 5
corpus_df_syntax_conj["token_norm"] = corpus_df_syntax_conj["token"].str.lower().str.replace('v', 'u').astype(str)
token_counts = corpus_df_syntax_conj["token_norm"].str.lower().value_counts()
frequent_tokens = token_counts[token_counts > n].index
corpus_df_syntax_conj_filtered = corpus_df_syntax_conj[corpus_df_syntax_conj["token_norm"].isin(frequent_tokens)]

In [ ]:
corpus_df_syntax_conj_filtered["doc_id"] = corpus_df_syntax_conj_filtered["doc_id"].str.replace('_TEI', '')

In [ ]:
corpus_df_syntax_conj_filtered["token_norm"].value_counts()

In [ ]:
filtered_conj = ['o', 'amen', '[', 'etwan', 'und', 'umb', 'das', 'noch', 'i', 'bey', 'wenn', 'eyn',
       'sy', 'ane', 'ais', 'uns', 'uff', 'ader', 'czu', 'ir', 'alz', 's',
       'ouch', 'sin', 'y', 'unnd', 'gy', 'a', 'e', 'c', 'm', 't', 'el', 'ct', 'że', 'oraz', 'jak', 'bądź', 'lecz', 'ale', 'czy',
       'jako', 'acz', 'sew', 'ia', 'gi', 'iako', 'ani', 'ge', 'qnod', 'dacz', 'aby', 'jakom', 'ot', 'czsom', 'iakom', 'sz', 'u', 'cle', 'ya', 'it.']
corpus_df_syntax_conj_filtered_out = corpus_df_syntax_conj_filtered[~corpus_df_syntax_conj_filtered["token_norm"].isin(filtered_conj)]
corpus_df_syntax_conj_filtered_out['token_norm'].value_counts()

In [ ]:
#all texts
plot_data, feature_matrix, clusterer, reducer = analyze_and_report_clusters(
    corpus_df_syntax_conj_filtered_out, 
    #doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    plot_features=True, n_top_features=5, n_clusters=10, clustering_method='hierarchical',
    columns=['token_norm']
)

In [ ]:
#books only
plot_data, feature_matrix, clusterer, reducer = analyze_and_report_clusters(
    corpus_df_syntax_conj_filtered_out, 
    doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    plot_features=True, n_top_features=5, n_clusters=5, clustering_method='hierarchical',
    columns=['token_norm']
)

### Leksyka

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re

def filter_corpus(corpus_df, doc_ids=None, doc_pattern=None):
    """
    Filter corpus based on document IDs or pattern
    
    Parameters:
    -----------
    corpus_df : pandas.DataFrame
        Original corpus DataFrame
    doc_ids : list, optional
        List of document IDs to include
    doc_pattern : str, optional
        Regex pattern to match document IDs
        
    Returns:
    --------
    pandas.DataFrame
        Filtered corpus DataFrame
        
    Notes:
    ------
    If both doc_ids and doc_pattern are provided, doc_ids takes precedence
    """
    if doc_ids is not None:
        return corpus_df[corpus_df['doc_id'].isin(doc_ids)]
    elif doc_pattern is not None:
        pattern = re.compile(doc_pattern)
        matching_docs = [doc_id for doc_id in corpus_df['doc_id'].unique() 
                        if pattern.match(str(doc_id))]
        return corpus_df[corpus_df['doc_id'].isin(matching_docs)]
    return corpus_df

def calculate_lexical_stats(corpus_df, doc_ids=None, doc_pattern=None):
    """
    Calculate various lexical statistics per text
    
    Parameters:
    -----------
    corpus_df : pandas.DataFrame
        DataFrame containing columns: doc_id, lemma
    doc_ids : list, optional
        List of document IDs to include
    doc_pattern : str, optional
        Regex pattern to match document IDs
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with lexical statistics per text
    """
    # Filter corpus if necessary
    filtered_df = filter_corpus(corpus_df, doc_ids, doc_pattern)
    
    stats = []
    
    for doc_id in filtered_df['doc_id'].unique():
        doc_df = filtered_df[filtered_df['doc_id'] == doc_id]
        
        total_tokens = len(doc_df)
        unique_lemmas = len(doc_df['lemma'].unique())
        
        # Type-Token Ratio (TTR)
        ttr = unique_lemmas / total_tokens if total_tokens > 0 else 0
        
        # Guiraud's R (normalized TTR)
        guiraud = unique_lemmas / np.sqrt(total_tokens) if total_tokens > 0 else 0
        
        # Herdan's C (logarithmic TTR)
        herdan = np.log(unique_lemmas) / np.log(total_tokens) if total_tokens > 0 else 0
        
        # MTLD (Measure of Textual Lexical Diversity)
        mtld = ttr * np.log(total_tokens) if total_tokens > 0 else 0
        
        stats.append({
            'doc_id': doc_id,
            'total_tokens': total_tokens,
            'unique_lemmas': unique_lemmas,
            'ttr': ttr,
            'guiraud': guiraud,
            'herdan': herdan,
            'mtld': mtld
        })
    
    return pd.DataFrame(stats)

def calculate_vocabulary_growth_by_text(corpus_df, doc_ids=None, doc_pattern=None):
    """
    Calculate cumulative vocabulary growth for each text
    
    Parameters:
    -----------
    corpus_df : pandas.DataFrame
        DataFrame containing columns: doc_id, lemma
    doc_ids : list, optional
        List of document IDs to include
    doc_pattern : str, optional
        Regex pattern to match document IDs
        
    Returns:
    --------
    pandas.DataFrame
        DataFrame with vocabulary growth data
    """
    # Filter corpus if necessary
    filtered_df = filter_corpus(corpus_df, doc_ids, doc_pattern)
    
    growth_data = []
    
    for doc_id in filtered_df['doc_id'].unique():
        doc_df = filtered_df[filtered_df['doc_id'] == doc_id]
        unique_lemmas = set()
        vocab_sizes = []
        
        for lemma in doc_df['lemma']:
            unique_lemmas.add(lemma)
            vocab_sizes.append(len(unique_lemmas))
        
        positions = np.linspace(0, 100, len(vocab_sizes))
        
        for pos, size in zip(positions, vocab_sizes):
            growth_data.append({
                'doc_id': doc_id,
                'position_percent': pos,
                'vocabulary_size': size
            })
    
    return pd.DataFrame(growth_data)

def plot_lexical_analysis(growth_df=None, stats_df=None, corpus_df=None, 
                         doc_ids=None, doc_pattern=None, title_suffix=None):
    """
    Create comprehensive lexical analysis plots using plotly
    
    Parameters:
    -----------
    growth_df : pandas.DataFrame, optional
        Precomputed vocabulary growth DataFrame
    stats_df : pandas.DataFrame, optional
        Precomputed lexical statistics DataFrame
    corpus_df : pandas.DataFrame, optional
        Raw corpus DataFrame if growth_df or stats_df not provided
    doc_ids : list, optional
        List of document IDs to include
    doc_pattern : str, optional
        Regex pattern to match document IDs
    title_suffix : str, optional
        Additional text to append to the plot title
        
    Returns:
    --------
    plotly.graph_objects.Figure
        Figure containing the lexical analysis plots
    """
    # Input validation
    if corpus_df is None and (growth_df is None or stats_df is None):
        raise ValueError("Either corpus_df or both growth_df and stats_df must be provided")
    
    # Calculate statistics if not provided
    if growth_df is None:
        growth_df = calculate_vocabulary_growth_by_text(corpus_df, doc_ids, doc_pattern)
    if stats_df is None:
        stats_df = calculate_lexical_stats(corpus_df, doc_ids, doc_pattern)
    
    # Create subplot figure
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Vocabulary Growth by Text',
            'Type-Token Ratio vs Text Length',
            'Lexical Diversity Measures',
            'Distribution of Lexical Measures'
        )
    )
    
    # 1. Vocabulary Growth Plot
    for doc_id in growth_df['doc_id'].unique():
        doc_data = growth_df[growth_df['doc_id'] == doc_id]
        fig.add_trace(
            go.Scatter(
                x=doc_data['position_percent'],
                y=doc_data['vocabulary_size'],
                name=f'Text {doc_id}',
                mode='lines'
            ),
            row=1, col=1
        )
    
    # 2. TTR vs Text Length Scatter
    fig.add_trace(
        go.Scatter(
            x=stats_df['total_tokens'],
            y=stats_df['ttr'],
            mode='markers+text',
            text=stats_df['doc_id'],
            name='TTR vs Length',
            textposition='top center'
        ),
        row=1, col=2
    )
    
    # 3. Lexical Diversity Measures Bar Plot
    measures = ['ttr', 'guiraud', 'herdan', 'mtld']
    for measure in measures:
        fig.add_trace(
            go.Bar(
                x=stats_df['doc_id'],
                y=stats_df[measure],
                name=measure.upper()
            ),
            row=2, col=1
        )
    
    # 4. Box Plots of Measures
    fig.add_trace(
        go.Box(
            y=stats_df[measures].values.flatten(),
            x=[measure.upper() * len(stats_df) for measure in measures],
            name='Distribution'
        ),
        row=2, col=2
    )
    
    # Create title with optional suffix
    title = "Comprehensive Lexical Analysis"
    if title_suffix:
        title = f"{title} - {title_suffix}"
    
    # Update layout
    fig.update_layout(
        height=800,
        width=1200,
        showlegend=True,
        title_text=title,
        template='plotly_white'
    )
    
    # Update axes labels
    fig.update_xaxes(title_text="Position in Text (%)", row=1, col=1)
    fig.update_yaxes(title_text="Vocabulary Size", row=1, col=1)
    
    fig.update_xaxes(title_text="Total Tokens", row=1, col=2)
    fig.update_yaxes(title_text="Type-Token Ratio", row=1, col=2)
    
    fig.update_xaxes(title_text="Text ID", row=2, col=1)
    fig.update_yaxes(title_text="Measure Value", row=2, col=1)
    
    fig.update_xaxes(title_text="Measure", row=2, col=2)
    fig.update_yaxes(title_text="Value Distribution", row=2, col=2)
    
    return fig

# Example usage:
"""
# Using document IDs
doc_ids = ['Ksg1', 'AKap', 'PomnLw']
fig = plot_lexical_analysis(
    corpus_df=corpus_df,
    doc_ids=doc_ids,
    title_suffix="Selected Documents"
)

# Using pattern
doc_pattern = 'Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib'
fig = plot_lexical_analysis(
    corpus_df=corpus_df,
    doc_pattern=doc_pattern,
    title_suffix="Matched Documents"
)

# Using precomputed data with selection
growth_df = calculate_vocabulary_growth_by_text(corpus_df, doc_pattern=doc_pattern)
stats_df = calculate_lexical_stats(corpus_df, doc_pattern=doc_pattern)
fig = plot_lexical_analysis(
    growth_df=growth_df,
    stats_df=stats_df,
    title_suffix="Matched Documents"
)

fig.show()
"""

In [ ]:
growth_df = calculate_vocabulary_growth_by_text(corpus_df, 
                                                doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib')
growth_df.to_parquet('out/corpus_df_growth.parquet')
stats_df = calculate_lexical_stats(corpus_df,
                                   doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib')
stats_df.to_parquet('out/corpus_df_stats.parquet')

In [ ]:
#too large
fig = plot_lexical_analysis(growth_df=growth_df, stats_df=stats_df)
fig.show()

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# Assuming growth_df has columns: doc_id, position_percent, vocabulary_size

# Create faceted plot
fig = px.line(
    growth_df,
    x='position_percent',
    y='vocabulary_size',
    facet_col='doc_id',  # Create separate subplot for each doc_id
    facet_col_wrap=3,    # Number of plots per row
    title='Vocabulary Growth by Text',
    labels={
        'position_percent': 'Position in Text (%)',
        'vocabulary_size': 'Vocabulary Size',
        'doc_id': 'Document ID'
    },
    height=200 * ((len(growth_df['doc_id'].unique()) + 2) // 3),  # Adjust height based on number of texts
    width=1000
)

# Update layout
fig.update_layout(
    showlegend=False,
    title_x=0.5,  # Center the title
    title_y=0.98  # Adjust title position
)

# Update axes
fig.update_xaxes(
    tickformat='.0f',  # Remove decimal places from percentage
    range=[0, 100]     # Fix x-axis range
)

# Update facet layout
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))  # Simplify facet titles

# Optional: Make all y-axes the same scale
fig.update_yaxes(matches=None)  # Comment this line if you want independent y-axes

fig.show()

In [ ]:
import plotly.graph_objects as go

# Create figure
fig = go.Figure()

# Add traces for each document
for doc_id in growth_df['doc_id'].unique():
    doc_data = growth_df[growth_df['doc_id'] == doc_id]
    fig.add_trace(
        go.Scatter(
            x=doc_data['position_percent'],
            y=doc_data['vocabulary_size'],
            name=str(doc_id),
            mode='lines',
            hovertemplate="Position: %{x:.1f}%<br>Vocabulary: %{y}<br>%{fullData.name}<extra></extra>"
        )
    )

# Update layout
fig.update_layout(
    title={
        'text': 'Vocabulary Growth Across Texts',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title="Position in Text (%)",
    yaxis_title="Vocabulary Size",
    height=600,
    width=1000,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02,
        title="Document ID"
    ),
    margin=dict(r=150),
    hovermode='x unified',
    xaxis=dict(
        tickformat='.0f',
        range=[0, 100]
    ),
    template='plotly_white'
)

fig.show()

In [ ]:
import pandas as pd
import numpy as np
from typing import Literal, Union, Optional
from tqdm.notebook import tqdm  # for Jupyter notebooks
import time
import plotly.graph_objects as go

def calculate_vocabulary_growth(
    corpus_df: pd.DataFrame,
    method: Literal['percentage', 'fixed_tokens', 'growth_curve'] = 'percentage',
    doc_ids: Optional[list] = None,
    doc_pattern: Optional[str] = None,
    n_bins: int = 100,
    token_interval: int = 100,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Calculate vocabulary growth using different methods with progress tracking
    """
    start_time = time.time()
    if verbose:
        print(f"Starting vocabulary growth calculation using {method} method...")
        print(f"Total documents to process: {len(corpus_df['doc_id'].unique())}")
    
    # Filter corpus if necessary
    filtered_df = filter_corpus(corpus_df, doc_ids, doc_pattern)
    
    if verbose:
        print(f"Processing {len(filtered_df['doc_id'].unique())} documents after filtering")
    
    if method == 'percentage':
        result = _calculate_percentage_bins(filtered_df, n_bins, verbose)
    elif method == 'fixed_tokens':
        result = _calculate_fixed_intervals(filtered_df, token_interval, verbose)
    elif method == 'growth_curve':
        result = _calculate_growth_curve(filtered_df, verbose)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    end_time = time.time()
    if verbose:
        print(f"\nCalculation completed in {end_time - start_time:.2f} seconds")
        print(f"Generated {len(result)} data points")
    
    return result

def _calculate_percentage_bins(
    corpus_df: pd.DataFrame, 
    n_bins: int,
    verbose: bool
) -> pd.DataFrame:
    """Calculate vocabulary growth using percentage-based bins"""
    growth_data = []
    unique_docs = corpus_df['doc_id'].unique()
    
    if verbose:
        print(f"\nProcessing documents with {n_bins} bins per document...")
        pbar = tqdm(unique_docs, desc="Processing documents")
    else:
        pbar = unique_docs
    
    for doc_id in pbar:
        if verbose:
            pbar.set_description(f"Processing document {doc_id}")
        
        doc_df = corpus_df[corpus_df['doc_id'] == doc_id]
        total_tokens = len(doc_df)
        
        bin_size = max(1, total_tokens // n_bins)
        unique_lemmas = set()
        current_bin = 0
        bin_vocab_sizes = []
        
        for idx, lemma in enumerate(doc_df['lemma']):
            unique_lemmas.add(lemma)
            
            if idx > 0 and idx % bin_size == 0:
                bin_vocab_sizes.append(len(unique_lemmas))
                current_bin += 1
        
        if len(bin_vocab_sizes) < n_bins:
            bin_vocab_sizes.append(len(unique_lemmas))
        
        positions = np.linspace(0, 100, len(bin_vocab_sizes))
        
        for pos, size in zip(positions, bin_vocab_sizes):
            growth_data.append({
                'doc_id': doc_id,
                'position': pos,
                'vocabulary_size': size,
                'position_type': 'percentage',
                'total_tokens': total_tokens
            })
    
    return pd.DataFrame(growth_data)

def _calculate_fixed_intervals(
    corpus_df: pd.DataFrame, 
    token_interval: int,
    verbose: bool
) -> pd.DataFrame:
    """Calculate vocabulary growth using fixed token intervals"""
    growth_data = []
    unique_docs = corpus_df['doc_id'].unique()
    
    if verbose:
        print(f"\nProcessing documents with {token_interval} tokens interval...")
        pbar = tqdm(unique_docs, desc="Processing documents")
    else:
        pbar = unique_docs
    
    for doc_id in pbar:
        if verbose:
            pbar.set_description(f"Processing document {doc_id}")
        
        doc_df = corpus_df[corpus_df['doc_id'] == doc_id]
        total_tokens = len(doc_df)
        
        unique_lemmas = set()
        vocab_sizes = []
        positions = []
        
        for idx, lemma in enumerate(doc_df['lemma'], 1):
            unique_lemmas.add(lemma)
            
            if idx % token_interval == 0:
                vocab_sizes.append(len(unique_lemmas))
                positions.append(idx)
        
        if total_tokens % token_interval != 0:
            vocab_sizes.append(len(unique_lemmas))
            positions.append(total_tokens)
        
        for pos, size in zip(positions, vocab_sizes):
            growth_data.append({
                'doc_id': doc_id,
                'position': pos,
                'vocabulary_size': size,
                'position_type': 'tokens',
                'total_tokens': total_tokens
            })
    
    return pd.DataFrame(growth_data)

def _calculate_growth_curve(
    corpus_df: pd.DataFrame,
    verbose: bool,
    n_samples: int = 50
) -> pd.DataFrame:
    """Calculate theoretical vocabulary growth curve using subsampling"""
    growth_data = []
    unique_docs = corpus_df['doc_id'].unique()
    
    if verbose:
        print(f"\nCalculating growth curves with {n_samples} bootstrap samples...")
        doc_pbar = tqdm(unique_docs, desc="Processing documents")
    else:
        doc_pbar = unique_docs
    
    for doc_id in doc_pbar:
        if verbose:
            doc_pbar.set_description(f"Processing document {doc_id}")
        
        doc_df = corpus_df[corpus_df['doc_id'] == doc_id]
        total_tokens = len(doc_df)
        lemmas = doc_df['lemma'].tolist()
        
        # Define sampling points
        sample_sizes = np.linspace(100, total_tokens, 20, dtype=int)
        
        if verbose:
            print(f"\nDocument {doc_id}: Calculating {len(sample_sizes)} sample points...")
            sample_pbar = tqdm(sample_sizes, desc="Processing sample sizes", leave=False)
        else:
            sample_pbar = sample_sizes
        
        for sample_size in sample_pbar:
            if verbose:
                sample_pbar.set_description(f"Sample size: {sample_size}")
            
            vocab_sizes = []
            for _ in range(n_samples):
                sample = np.random.choice(lemmas, size=sample_size, replace=True)
                vocab_sizes.append(len(set(sample)))
            
            mean_vocab_size = np.mean(vocab_sizes)
            std_vocab_size = np.std(vocab_sizes)
            
            growth_data.append({
                'doc_id': doc_id,
                'position': sample_size,
                'vocabulary_size': mean_vocab_size,
                'vocab_std': std_vocab_size,
                'position_type': 'sampled_tokens',
                'total_tokens': total_tokens
            })
    
    return pd.DataFrame(growth_data)

# Example usage with progress tracking:
"""
# Calculate growth with verbose output
growth_df = calculate_vocabulary_growth(
    corpus_df,
    method='growth_curve',
    verbose=True
)
"""

In [ ]:
def plot_vocabulary_growth(
    growth_df: pd.DataFrame,
    plot_type: Literal['individual', 'combined'] = 'combined',
    normalize_position: bool = False
) -> go.Figure:
    """
    Plot vocabulary growth curves
    
    Parameters:
    -----------
    growth_df : pandas.DataFrame
        DataFrame with vocabulary growth data
    plot_type : str
        'individual' for faceted plot, 'combined' for single plot
    normalize_position : bool
        If True, normalize positions to percentages for fixed_tokens method
    
    Returns:
    --------
    plotly.graph_objects.Figure
        Plot of vocabulary growth
    """
    # Create copy to avoid modifying original
    plot_df = growth_df.copy()
    
    # Normalize positions if requested and not already percentages
    if normalize_position and 'tokens' in plot_df['position_type'].iloc[0]:
        plot_df['position'] = plot_df.apply(
            lambda x: (x['position'] / x['total_tokens']) * 100, 
            axis=1
        )
    
    # Create figure based on plot type
    if plot_type == 'individual':
        fig = px.line(
            plot_df,
            x='position',
            y='vocabulary_size',
            facet_col='doc_id',
            facet_col_wrap=3,
            title='Vocabulary Growth by Text',
            labels={
                'position': 'Position' + (' (%)' if normalize_position else ' (tokens)'),
                'vocabulary_size': 'Vocabulary Size',
                'doc_id': 'Document ID'
            },
            height=200 * ((len(plot_df['doc_id'].unique()) + 2) // 3)
        )
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
        
    else:  # combined plot
        fig = px.line(
            plot_df,
            x='position',
            y='vocabulary_size',
            color='doc_id',
            title='Vocabulary Growth Across Texts',
            labels={
                'position': 'Position' + (' (%)' if normalize_position else ' (tokens)'),
                'vocabulary_size': 'Vocabulary Size',
                'doc_id': 'Document ID'
            }
        )
        
        fig.update_layout(
            legend=dict(
                yanchor="top",
                y=0.99,
                xanchor="left",
                x=1.02
            ),
            margin=dict(r=150)
        )
    
    # Update hover template
    fig.update_traces(
        hovertemplate=(
            "Position: %{x:.1f}" + ("%" if normalize_position else "") +
            "<br>Vocabulary: %{y:.0f}<br>%{fullData.name}<extra></extra>"
        )
    )
    
    return fig

# Example usage:
"""
# Calculate growth using different methods
percent_growth = calculate_vocabulary_growth(
    corpus_df, 
    method='percentage', 
    n_bins=100
)

fixed_growth = calculate_vocabulary_growth(
    corpus_df, 
    method='fixed_tokens', 
    token_interval=100
)

curve_growth = calculate_vocabulary_growth(
    corpus_df, 
    method='growth_curve'
)

# Plot results
fig1 = plot_vocabulary_growth(percent_growth, plot_type='combined')
fig2 = plot_vocabulary_growth(fixed_growth, plot_type='combined', normalize_position=True)
fig3 = plot_vocabulary_growth(curve_growth, plot_type='combined')

# Show plots
fig1.show()
fig2.show()
fig3.show()
"""

In [ ]:
percent_growth = calculate_vocabulary_growth(
    corpus_df,
    doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    method='percentage', 
    n_bins=100
)
fixed_growth = calculate_vocabulary_growth(
    corpus_df, 
    doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    method='fixed_tokens', 
    token_interval=100
)

In [ ]:
curve_growth = calculate_vocabulary_growth(
    corpus_df, 
    doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    method='growth_curve'
)

In [ ]:
percent_growth.to_parquet('out/corpus_df_percent_growth.parquet')
fixed_growth.to_parquet('out/corpus_df_fixed_growth.parquet')
curve_growth.to_parquet('out/corpus_df_curve_growth.parquet')

In [ ]:
percent_growth = pd.read_parquet('out/corpus_df_percent_growth.parquet')
fixed_growth = pd.read_parquet('out/corpus_df_fixed_growth.parquet')
curve_growth = pd.read_parquet('out/corpus_df_curve_growth.parquet')

In [ ]:
import plotly.express as px
fig1 = plot_vocabulary_growth(percent_growth, plot_type='combined')
fig2 = plot_vocabulary_growth(fixed_growth, plot_type='combined', normalize_position=True)
fig3 = plot_vocabulary_growth(curve_growth, plot_type='combined')
fig1.show()
fig2.show()
fig3.show()

##### Filter out certain POS

In [ ]:
import pandas as pd
corpus_df = pd.read_parquet("corpora/spacy_corpus_all_df.parquet")
corpus_df_filtered = corpus_df[corpus_df["pos"].isin(['NOUN', 'ADV', 'ADJ', 'VERB'])]

In [ ]:
percent_growth_pos = calculate_vocabulary_growth(
    corpus_df_filtered,
    doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    method='percentage', 
    n_bins=100
)
fixed_growth_pos = calculate_vocabulary_growth(
    corpus_df_filtered, 
    doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    method='fixed_tokens', 
    token_interval=100
)
curve_growth_pos = calculate_vocabulary_growth(
    corpus_df_filtered, 
    doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    method='growth_curve'
)

In [ ]:
percent_growth_pos.to_parquet('out/corpus_df_percent_growth_pos.parquet')
fixed_growth_pos.to_parquet('out/corpus_df_fixed_growth_pos.parquet')
curve_growth_pos.to_parquet('out/corpus_df_curve_growth_pos.parquet')

In [ ]:
import pandas as pd
percent_growth_pos = pd.read_parquet('out/corpus_df_percent_growth_pos.parquet')
fixed_growth_pos = pd.read_parquet('out/corpus_df_fixed_growth_pos.parquet')
curve_growth_pos = pd.read_parquet('out/corpus_df_curve_growth_pos.parquet')

In [ ]:
fig1 = plot_vocabulary_growth(percent_growth_pos, plot_type='combined')
fig2 = plot_vocabulary_growth(fixed_growth_pos, plot_type='combined', normalize_position=True)
fig3 = plot_vocabulary_growth(curve_growth_pos, plot_type='combined')
fig1.show()
fig2.show()
fig3.show()

In [ ]:
fig1 = plot_vocabulary_growth(percent_growth_pos, plot_type='individual')
fig2 = plot_vocabulary_growth(fixed_growth_pos, plot_type='individual', normalize_position=True)
fig3 = plot_vocabulary_growth(curve_growth_pos, plot_type='individual')
fig1.show()
fig2.show()
fig3.show()

#### New vocabulary

In [ ]:
corpus_df_filtered[corpus_df_filtered['doc_id'].str.startswith('PomnLw1')]["lemma"][0:10]

In [ ]:
# emergent terms

In [ ]:
# Required imports
from collections import defaultdict
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import fnmatch
import re
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt

class TextAnalyzer:
    """
    A class for analyzing word emergence patterns in sequential texts.
    
    Attributes:
        df (pd.DataFrame): Input DataFrame containing text data
        doc_ids (list): List of document IDs in sequence
        emerged_patterns (dict): Dictionary storing emergence patterns
    """
    
    def __init__(self, df, doc_pattern=None, doc_list=None, pos_filter=None):
        """
        Initialize TextAnalyzer with document filtering and POS filtering options.
        
        Parameters:
            df (pd.DataFrame): DataFrame with columns: 'doc_id', 'lemma', and optionally 'pos'
            doc_pattern (str, optional): Pattern to match document IDs (e.g., 'VAd_*')
            doc_list (list, optional): Explicit list of document IDs to include
            pos_filter (str or list, optional): POS tag(s) to include (e.g., 'NOUN' or ['NOUN', 'ADJ'])
        """
        print("Initializing TextAnalyzer...")
        
        # Create a safe copy of the dataframe
        self.df = df.copy()
        
        # Apply POS filter if specified
        if pos_filter:
            if 'pos' not in self.df.columns:
                raise ValueError("POS filtering requested but 'pos' column not found in DataFrame")
            self.pos_filter = [pos_filter] if isinstance(pos_filter, str) else pos_filter
            print(f"Filtering for POS tags: {self.pos_filter}")
            self.df = self.df[self.df['pos'].isin(self.pos_filter)]
            print(f"Retained {len(self.df)} tokens after POS filtering")
        
        # Get document IDs
        self.all_doc_ids = sorted(self.df['doc_id'].unique())
        self.doc_ids = self._filter_documents(doc_pattern, doc_list)
        self.df = self.df[self.df['doc_id'].isin(self.doc_ids)].copy()
        
        print(f"Working with {len(self.doc_ids)} documents")
        
        # Calculate frequencies
        print("Computing word frequencies...")
        self._doc_frequencies = {
            doc_id: self.df[self.df['doc_id'] == doc_id]['lemma'].value_counts()
            for doc_id in tqdm(self.doc_ids, desc="Processing documents")
        }
        
        self.emerged_patterns = {}
    
    def _filter_documents(self, pattern=None, doc_list=None):
        """
        Filter documents based on pattern or explicit list.
        
        Parameters:
            pattern (str, optional): Pattern to match document IDs
            doc_list (list, optional): Explicit list of document IDs
            
        Returns:
            list: Filtered list of document IDs
        """
        if doc_list is not None:
            valid_docs = [doc for doc in doc_list if doc in self.all_doc_ids]
            invalid_docs = set(doc_list) - set(valid_docs)
            if invalid_docs:
                print(f"Warning: Documents not found: {invalid_docs}")
            return sorted(valid_docs)
        
        elif pattern is not None:
            matched_docs = fnmatch.filter(self.all_doc_ids, pattern)
            if not matched_docs:
                print(f"Warning: No documents matched pattern: {pattern}")
            return sorted(matched_docs)
        
        else:
            return self.all_doc_ids

    def compute_emerged_words(self, threshold=10, window_size=5):
        """
        Compute emerged words with position tracking.
        
        Parameters:
            threshold (int): Minimum number of occurrences in subsequent documents
            window_size (int): Number of subsequent documents to check
            
        Returns:
            dict: Dictionary with emerged words and their position information
        """
        print(f"Computing emerged words (threshold={threshold}, window={window_size})...")
        
        # Create word positions dictionary
        word_positions = defaultdict(list)
        
        # Track positions
        print("Tracking word positions...")
        for _, row in tqdm(self.df.iterrows(), total=len(self.df), desc="Processing tokens"):
            word_positions[row['lemma']].append({
                'doc_id': row['doc_id'],
                'token': row.get('token', row['lemma']),
                'pos': row.get('pos', None),
                'doc_idx': self.doc_ids.index(row['doc_id'])
            })
        
        emerged_words = defaultdict(dict)
        
        print("Analyzing emergence patterns...")
        for lemma, positions in tqdm(word_positions.items(), desc="Analyzing words"):
            # Sort by document order
            sorted_positions = sorted(positions, key=lambda x: x['doc_idx'])
            
            # Get first occurrence
            first_pos = sorted_positions[0]
            first_doc = first_pos['doc_id']
            first_doc_idx = first_pos['doc_idx']
            
            # Skip if we can't look ahead enough documents
            if first_doc_idx + window_size >= len(self.doc_ids):
                continue
            
            # Get subsequent documents to check
            subsequent_docs = self.doc_ids[first_doc_idx + 1:first_doc_idx + 1 + window_size]
            
            # Count occurrences
            first_doc_occurrences = [pos for pos in sorted_positions if pos['doc_id'] == first_doc]
            subsequent_occurrences = [pos for pos in sorted_positions if pos['doc_id'] in subsequent_docs]
            
            subsequent_by_doc = defaultdict(list)
            for pos in subsequent_occurrences:
                subsequent_by_doc[pos['doc_id']].append(pos)
            
            total_subsequent = len(subsequent_occurrences)
            docs_with_word = len(subsequent_by_doc)
            
            # Check emergence criteria
            if total_subsequent >= threshold and docs_with_word >= window_size // 2:
                emerged_words[first_doc][lemma] = {
                    'first_doc': first_doc,
                    'first_occurrences': first_doc_occurrences,
                    'first_count': len(first_doc_occurrences),
                    'subsequent_docs': docs_with_word,
                    'total_subsequent': total_subsequent,
                    'avg_usage': total_subsequent / len(subsequent_docs),
                    'usage_pattern': {doc: len(occs) for doc, occs in subsequent_by_doc.items()},
                    'all_positions': sorted_positions,
                    'subsequent_positions': subsequent_occurrences
                }
        
        self.emerged_patterns = emerged_words
        
        # Print summary
        total_emerged = sum(len(words) for words in emerged_words.values())
        print(f"\nFound {total_emerged} emerged words across {len(emerged_words)} documents")
        
        return emerged_words

    def plot_emerged_words(self, words=None, n_top=None, plot_lib='plotly', single_doc=None, faceted=False):
        """
        Plot emergence patterns with options for single document or faceted view.
        
        Parameters:
            words (list, optional): List of specific words to plot
            n_top (int, optional): Number of top words to plot if words=None
            plot_lib (str): 'plotly' or 'seaborn'
            single_doc (str, optional): Document ID to show patterns for a single document
            faceted (bool): Whether to create separate facets/subplots for each document
            
        Returns:
            Figure object (plotly.graph_objects.Figure or matplotlib.figure.Figure)
        """
        if not self.emerged_patterns:
            print("No emergence patterns computed. Run compute_emerged_words() first.")
            return None
        
        # Filter for single document if specified
        doc_patterns = (
            {single_doc: self.emerged_patterns[single_doc]} 
            if single_doc and single_doc in self.emerged_patterns
            else self.emerged_patterns
        )
    
        if single_doc and single_doc not in self.emerged_patterns:
            print(f"No emergence patterns found for document: {single_doc}")
            return None
    
        # Collect all emergence data
        plot_data = []
        for doc_id, patterns in doc_patterns.items():
            for word, stats in patterns.items():
                if words is None or word in words:
                    doc_idx = self.doc_ids.index(doc_id)
                    
                    plot_data.append({
                        'word': word,
                        'doc_id': doc_id,
                        'emerged_in': doc_id,  # track emergence document
                        'count': stats['first_count'],
                        'doc_idx': doc_idx,
                        'type': 'emergence'
                    })
                    
                    for subsequent_doc, count in stats['usage_pattern'].items():
                        plot_data.append({
                            'word': word,
                            'doc_id': subsequent_doc,
                            'emerged_in': doc_id,  # track emergence document
                            'count': count,
                            'doc_idx': self.doc_ids.index(subsequent_doc),
                            'type': 'subsequent'
                        })
    
        if not plot_data:
            print("No data to plot.")
            return None
    
        df_plot = pd.DataFrame(plot_data)
    
        # Select top words if needed
        if n_top and words is None:
            top_words = (df_plot.groupby('word')['count']
                        .sum()
                        .sort_values(ascending=False)
                        .head(n_top)
                        .index)
            df_plot = df_plot[df_plot['word'].isin(top_words)]
    
        if plot_lib == 'plotly':
            if faceted and not single_doc:
                # Create subplot grid
                n_docs = len(doc_patterns)
                n_cols = min(2, n_docs)
                n_rows = (n_docs + n_cols - 1) // n_cols
                
                fig = make_subplots(
                    rows=n_rows, 
                    cols=n_cols,
                    subplot_titles=[f"Emerged in {doc}" for doc in doc_patterns.keys()]
                )
                
                # Plot each document in its own subplot
                for idx, (doc_id, patterns) in enumerate(doc_patterns.items(), 1):
                    doc_data = df_plot[df_plot['emerged_in'] == doc_id]
                    row = (idx - 1) // n_cols + 1
                    col = (idx - 1) % n_cols + 1
                    
                    for word in doc_data['word'].unique():
                        word_data = doc_data[doc_data['word'] == word]
                        
                        fig.add_trace(
                            go.Scatter(
                                x=word_data['doc_idx'],
                                y=word_data['count'],
                                name=word,
                                mode='lines+markers',
                                line=dict(width=2),
                                marker=dict(
                                    size=10,
                                    symbol=['star' if t == 'emergence' else 'circle'
                                           for t in word_data['type']]
                                ),
                                hovertemplate=(
                                    "Word: %{text}<br>" +
                                    "Document: %{customdata}<br>" +
                                    "Count: %{y}<br>" +
                                    "<extra></extra>"
                                ),
                                text=word_data['word'],
                                customdata=word_data['doc_id'],
                                showlegend=idx == 1  # show legend only for first subplot
                            ),
                            row=row, col=col
                        )
                
                # Update layout
                fig.update_layout(
                    title="Word Emergence Patterns by Document",
                    height=300 * n_rows,
                    width=1000,
                    showlegend=True,
                    legend_title="Words"
                )
                
                # Update all xaxes
                for i in range(1, n_docs + 1):
                    fig.update_xaxes(
                        title_text="Document Sequence",
                        tickmode='array',
                        ticktext=self.doc_ids,
                        tickvals=list(range(len(self.doc_ids))),
                        tickangle=45,
                        row=(i-1)//n_cols + 1,
                        col=(i-1)%n_cols + 1
                    )
                    fig.update_yaxes(
                        title_text="Word Count",
                        row=(i-1)//n_cols + 1,
                        col=(i-1)%n_cols + 1
                    )
                
            else:
                # Single plot (original behavior)
                fig = go.Figure()
                
                for word in df_plot['word'].unique():
                    word_data = df_plot[df_plot['word'] == word]
                    
                    fig.add_trace(go.Scatter(
                        x=word_data['doc_idx'],
                        y=word_data['count'],
                        name=word,
                        mode='lines+markers',
                        line=dict(width=2),
                        marker=dict(
                            size=10,
                            symbol=['star' if t == 'emergence' else 'circle'
                                   for t in word_data['type']]
                        ),
                        hovertemplate=(
                            "Word: %{text}<br>" +
                            "Document: %{customdata}<br>" +
                            "Count: %{y}<br>" +
                            "<extra></extra>"
                        ),
                        text=word_data['word'],
                        customdata=word_data['doc_id']
                    ))
                
                title = (f"Word Emergence Patterns for {single_doc}" 
                        if single_doc 
                        else "Word Emergence Patterns")
                
                fig.update_layout(
                    title=title,
                    xaxis_title="Document Sequence",
                    yaxis_title="Word Count",
                    xaxis=dict(
                        tickmode='array',
                        ticktext=self.doc_ids,
                        tickvals=list(range(len(self.doc_ids))),
                        tickangle=45
                    ),
                    hovermode='x unified',
                    showlegend=True,
                    legend_title="Words",
                    height=600
                )
            
            return fig
                
        else:  # seaborn
            if faceted and not single_doc:
                # Create figure with subplots
                n_docs = len(doc_patterns)
                fig, axes = plt.subplots(
                    n_docs, 1, 
                    figsize=(12, 5*n_docs), 
                    squeeze=False
                )
                
                # Plot each document in its own subplot
                for idx, (doc_id, patterns) in enumerate(doc_patterns.items()):
                    doc_data = df_plot[df_plot['emerged_in'] == doc_id]
                    ax = axes[idx, 0]
                    
                    sns.scatterplot(
                        data=doc_data,
                        x='doc_idx',
                        y='count',
                        hue='word',
                        style='type',
                        markers={'emergence': 'X', 'subsequent': 'o'},
                        s=100,
                        ax=ax
                    )
                    
                    ax.set_title(f"Emerged in {doc_id}")
                    ax.set_xlabel("Document Sequence")
                    ax.set_ylabel("Word Count")
                    
                    ax.set_xticks(range(len(self.doc_ids)))
                    ax.set_xticklabels(self.doc_ids, rotation=45, ha='right')
                
                plt.tight_layout()
                return fig
                
            else:
                # Single plot (original behavior)
                plt.figure(figsize=(12, 6))
                
                g = sns.scatterplot(
                    data=df_plot,
                    x='doc_idx',
                    y='count',
                    hue='word',
                    style='type',
                    markers={'emergence': 'X', 'subsequent': 'o'},
                    s=100
                )
                
                title = (f"Word Emergence Patterns for {single_doc}" 
                        if single_doc 
                        else "Word Emergence Patterns")
                
                plt.title(title)
                plt.xlabel("Document Sequence")
                plt.ylabel("Word Count")
                
                plt.xticks(
                    range(len(self.doc_ids)),
                    self.doc_ids,
                    rotation=45,
                    ha='right'
                )
                
                plt.tight_layout()
                return plt.gcf()
    
    # Example usage:
    """
    # Single document view
    analyzer.plot_emerged_words(single_doc='VAd_TEI_001', plot_lib='plotly')
    
    # Faceted view
    analyzer.plot_emerged_words(faceted=True, plot_lib='plotly')
    
    # Seaborn faceted view
    analyzer.plot_emerged_words(faceted=True, plot_lib='seaborn')
    """
    def plot_emergence_heatmap(self, words=None, n_top=None):
        """
        Create a heatmap of word emergence patterns.
        
        Parameters:
            words (list, optional): List of specific words to plot
            n_top (int, optional): Number of top words to plot if words=None
            
        Returns:
            plotly.graph_objects.Figure
        """
        if not self.emerged_patterns:
            print("No emergence patterns computed. Run compute_emerged_words() first.")
            return None

        heatmap_data = []
        for doc_id, doc_patterns in self.emerged_patterns.items():
            for word, stats in doc_patterns.items():
                if words is None or word in words:
                    heatmap_data.append({
                        'word': word,
                        'doc_id': doc_id,
                        'count': stats['first_count']
                    })
                    
                    for subsequent_doc, count in stats['usage_pattern'].items():
                        heatmap_data.append({
                            'word': word,
                            'doc_id': subsequent_doc,
                            'count': count
                        })

        if not heatmap_data:
            print("No data to plot.")
            return None

        df_heatmap = pd.DataFrame(heatmap_data)

        # Select top words if needed
        if n_top and words is None:
            top_words = (df_heatmap.groupby('word')['count']
                        .sum()
                        .sort_values(ascending=False)
                        .head(n_top)
                        .index)
            df_heatmap = df_heatmap[df_heatmap['word'].isin(top_words)]

        # Pivot data for heatmap
        heatmap_pivot = df_heatmap.pivot(
            index='word',
            columns='doc_id',
            values='count'
        ).fillna(0)

        fig = px.imshow(
            heatmap_pivot,
            aspect='auto',
            color_continuous_scale='YlOrRd',
            title='Word Usage Patterns Across Documents'
        )

        fig.update_layout(
            xaxis_title="Document",
            yaxis_title="Word",
            xaxis={'side': 'bottom'},
            height=max(400, len(heatmap_pivot) * 30),
            width=1000
        )

        return fig

# Example usage:
"""
# Initialize analyzer
analyzer = TextAnalyzer(df)

# Compute emergence patterns
emerged = analyzer.compute_emerged_words(threshold=10, window_size=5)

# Create visualizations
plotly_fig = analyzer.plot_emerged_words(n_top=10, plot_lib='plotly')
plotly_fig.show()

seaborn_fig = analyzer.plot_emerged_words(n_top=10, plot_lib='seaborn')
plt.show()

heatmap_fig = analyzer.plot_emergence_heatmap(n_top=15)
heatmap_fig.show()
"""

In [ ]:
analyzer = TextAnalyzer(corpus_df_filtered, doc_pattern='Ksg*', pos_filter=['NOUN', 'VERB', 'ADJ'])

In [ ]:
emerged = analyzer.compute_emerged_words(
    threshold=5,  # minimum subsequent occurrences
    window_size=5,  # number of documents to check
    
)

In [ ]:
emerged

In [ ]:
# Plot single document
fig1 = analyzer.plot_emerged_words(
    single_doc='AKapSąd1Wladisl_TEI',  # specific document
    n_top=5,                   # top 5 words
    plot_lib='plotly'          # using plotly
)
fig1.show()

In [ ]:
# Plot faceted view with all documents
fig2 = analyzer.plot_emerged_words(
    faceted=True,              # separate subplot for each document
    n_top=10,                  # top 10 words
    plot_lib='plotly'          # using plotly
)
fig2.show()

### Lingwistyka tekstu

#### Formulaiczność

In [ ]:
corpus_gensim_sents_words = GensimCorpus.load(input_prefix="gensim_sents_words",directory="corpora")
corpus_gensim_sents_lemmas = GensimCorpus.load(input_prefix="gensim_sents_lemmas",directory="corpora")

In [ ]:
stopwords=["ab", "ac", "ad", "adhic", "aliqui", "aliquis", "an", "ante", "apud", "at", "atque", "aut", "autem", "cum", "cur", "de", "deinde", "dum", "ego", "enim", "ergo", "es", "est", "et", "etiam", "etsi", "ex", "fio", "haud", "hic", "iam", "idem", "igitur", "ille", "in", "infra", "inter", "interim", "ipse", "is", "ita", "magis", "modo", "mox", "nam", "ne", "nec", "necque", "neque", "nisi", "non", "nos", "o", "ob", "per", "possum", "post", "pro", "quae", "quam", "quare", "qui", "quia", "quicumque", "quidem", "quilibet", "quis", "quisnam", "quisquam", "quisque", "quisquis", "quo", "quoniam", "sed", "si", "sic", "sive", "sub", "sui", "sum", "super", "suus", "tam", "tamen", "trans", "tu", "tum", "ubi", "uel", "uero", "unus", "ut"]
corpus_gensim_words_sents = GensimCorpusSents(corpus_dir, corpus_files, stopwords_list=stopwords, use_lemmas=False)
corpus_geGensimCorpusSentsnsim_lemmas_sents = GensimCorpusSents(corpus_dir, corpus_files, stopwords_list=stopwords, use_lemmas=True)

In [ ]:
from gensim.models import phrases
if mode == 'build':
    words_sents = corpus_gensim_words_sents.get_sentences()
    words_phrases_model = phrases.Phrases(corpus_gensim_words_sents)  # corpus_sents is your GensimCorpusSents instance

    lemmas_sents = corpus_gensim_lemmas_sents.get_sentences()
    lemmas_phrases_model = phrases.Phrases(corpus_gensim_lemmas_sents)  # corpus_sents is your GensimCorpusSents instance

    words_phrases_model.save("models/words_phrases_model.bin", separately=None, sep_limit=10485760, ignore=frozenset({}), pickle_protocol=4)
    lemmas_phrases_model.save("models/lemmas_phrases_model.bin", separately=None, sep_limit=10485760, ignore=frozenset({}), pickle_protocol=4)
elif mode == 'load':
    words_phrases_model.load("models/words_phrases_model.bin")
    lemmas_phrases_model.load("models/lemmas_phrases_model.bin")

In [ ]:
from gensim.models import phrases

words_sents = corpus_gensim_words_sents.get_sentences()
words_phrases_model = phrases.Phrases(corpus_gensim_words_sents)  # corpus_sents is your GensimCorpusSents instance

In [ ]:
words_sents = corpus_gensim_words_sents.get_sentences(text_list=['VAd_TEI_final.conllu', 'VITELO_Persp1_TEI_final.conllu'])
words_phrases_model.find_phrases(words_sents).items()

In [ ]:
from gensim.models.phrases import Phrases, Phraser
import os
import pickle
import re
from collections import Counter
from typing import List, Dict, Set, Tuple, Union
import logging

class PhraseModelProcessor:
    def __init__(self, 
                 corpus: GensimCorpusSents,
                 min_count: int = 5,
                 threshold: float = 10.0,
                 scoring: str = 'default',
                 save_dir: str = 'phrase_models'):
        """
        Initialize the phrase model processor.
        
        Args:
            corpus: GensimCorpusSents instance
            min_count: Minimum frequency of phrases
            threshold: Higher means fewer phrases
            scoring: Scoring method for phrases ('default', 'npmi')
            save_dir: Directory to save/load models
        """
        self.corpus = corpus
        self.min_count = min_count
        self.threshold = threshold
        self.scoring = scoring
        self.save_dir = save_dir
        self.models = {}
        
        # Create save directory if it doesn't exist
        os.makedirs(save_dir, exist_ok=True)
        
        # Setup logging
        logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
    
    def filter_documents(self, doc_filter: Union[str, List[str], str]) -> List[str]:
        """
        Filter document IDs based on the provided filter.
        
        Args:
            doc_filter: Can be one of:
                - Single document ID (str)
                - List of document IDs (List[str])
                - Regex pattern for document IDs (str, e.g., 'Ksg[^H]|AKap')
        
        Returns:
            List of matching document IDs
        """
        all_docs = list(self.corpus.corpus.keys())
        
        if isinstance(doc_filter, str):
            # Check if it's a regex pattern by attempting to compile it
            try:
                pattern = re.compile(doc_filter)
                # If it compiles, use it as a regex
                filtered_docs = [doc for doc in all_docs if pattern.search(doc)]
                if not filtered_docs:
                    # If no matches, treat it as a single document ID
                    if doc_filter in all_docs:
                        filtered_docs = [doc_filter]
            except re.error:
                # If regex compilation fails, treat it as a single document ID
                if doc_filter in all_docs:
                    filtered_docs = [doc_filter]
                else:
                    filtered_docs = []
        elif isinstance(doc_filter, list):
            # Filter list of document IDs
            filtered_docs = [doc for doc in doc_filter if doc in all_docs]
        else:
            raise ValueError("doc_filter must be either a string or a list of strings")
        
        if not filtered_docs:
            logging.warning("No matching documents found for the given filter")
        else:
            logging.info(f"Found {len(filtered_docs)} matching documents")
            
        return filtered_docs
        
    def build_models(self, n_grams: List[int] = [2, 3, 4, 5], doc_filter: Union[str, List[str], str] = None):
        """
        Build phrase models for specified n-grams.
        
        Args:
            n_grams: List of n-gram sizes to build models for
            doc_filter: Optional filter to build models on subset of documents
        """
        if doc_filter:
            filtered_docs = self.filter_documents(doc_filter)
            sentences = list(self.corpus.get_sentences(filtered_docs))
        else:
            sentences = list(self.corpus)
            
        logging.info(f"Building models using {len(sentences)} sentences")
        
        for n in n_grams:
            logging.info(f"Building {n}-gram model...")
            
            # For bigrams, start with raw sentences
            if n == 2:
                phrases = Phrases(sentences, 
                                min_count=self.min_count,
                                threshold=self.threshold,
                                scoring=self.scoring)
                self.models[n] = Phraser(phrases)
                transformed_sentences = [self.models[n][sent] for sent in sentences]
            
            # For higher n-grams, use previously transformed sentences
            else:
                phrases = Phrases(transformed_sentences,
                                min_count=self.min_count,
                                threshold=self.threshold,
                                scoring=self.scoring)
                self.models[n] = Phraser(phrases)
                transformed_sentences = [self.models[n][sent] for sent in transformed_sentences]
    
    def save_models(self):
        """
        Save all phrase models to disk.
        """
        for n, model in self.models.items():
            model_path = os.path.join(self.save_dir, f'phrase_model_{n}gram.pkl')
            with open(model_path, 'wb') as f:
                pickle.dump(model, f)
            logging.info(f"Saved {n}-gram model to {model_path}")
    
    def load_models(self, n_grams: List[int] = [2, 3, 4, 5]):
        """
        Load phrase models from disk.
        """
        self.models = {}
        for n in n_grams:
            model_path = os.path.join(self.save_dir, f'phrase_model_{n}gram.pkl')
            if os.path.exists(model_path):
                with open(model_path, 'rb') as f:
                    self.models[n] = pickle.load(f)
                logging.info(f"Loaded {n}-gram model from {model_path}")
            else:
                logging.warning(f"Model file not found: {model_path}")
    
    def analyze_documents(self,
                         doc_filter: Union[str, List[str], str],
                         min_frequency: int = 2,
                         top_n: int = 10) -> Dict[str, Dict[int, List[Tuple[str, int]]]]:
        """
        Analyze phrases in multiple documents matching the filter.
        
        Args:
            doc_filter: Document filter (single ID, list of IDs, or regex pattern)
            min_frequency: Minimum frequency for phrases to be included
            top_n: Number of top phrases to return per n-gram
            
        Returns:
            Dictionary mapping document IDs to their phrase analysis results
        """
        filtered_docs = self.filter_documents(doc_filter)
        results = {}
        
        for doc_id in filtered_docs:
            results[doc_id] = self.extract_phrases_from_text(doc_id, min_frequency, top_n)
            
        return results
    
    def extract_phrases_from_text(self, 
                                doc_id: str, 
                                min_frequency: int = 2,
                                top_n: int = 10) -> Dict[int, List[Tuple[str, int]]]:
        """
        Extract most important phrases from a specific document.
        
        Args:
            doc_id: Document identifier
            min_frequency: Minimum frequency for phrases to be included
            top_n: Number of top phrases to return per n-gram
            
        Returns:
            Dictionary mapping n-gram size to list of (phrase, frequency) tuples
        """
        # Get sentences for the specific document
        doc_sentences = self.corpus.get_sentences([doc_id])
        
        results = {}
        transformed_sentences = doc_sentences
        
        for n in sorted(self.models.keys()):
            # Transform sentences using the current model
            transformed_sentences = [self.models[n][sent] for sent in transformed_sentences]
            
            # Count phrases
            phrase_counter = Counter()
            for sent in transformed_sentences:
                phrases = [token for token in sent if '_' in token]
                phrase_counter.update(phrases)
            
            # Filter and sort phrases
            important_phrases = [(phrase, count) 
                               for phrase, count in phrase_counter.items() 
                               if count >= min_frequency]
            important_phrases.sort(key=lambda x: x[1], reverse=True)
            
            results[n] = important_phrases[:top_n]
        
        return results

def print_phrase_analysis(results: Union[Dict[int, List[Tuple[str, int]]], 
                                       Dict[str, Dict[int, List[Tuple[str, int]]]]]):
    """
    Pretty print the phrase analysis results.
    
    Args:
        results: Either results for a single document or multiple documents
    """
    if not any(isinstance(v, dict) for v in results.values()):
        # Single document results
        print("\nPhrase Analysis Results")
        print("=" * 50)
        
        for n_gram, phrases in results.items():
            print(f"\nTop {n_gram}-gram phrases:")
            print("-" * 30)
            if phrases:
                for phrase, count in phrases:
                    print(f"{phrase.replace('_', ' '):<40} {count}")
            else:
                print("No significant phrases found")
    else:
        # Multiple document results
        for doc_id, doc_results in results.items():
            print(f"\nPhrase Analysis for document: {doc_id}")
            print("=" * 50)
            
            for n_gram, phrases in doc_results.items():
                print(f"\nTop {n_gram}-gram phrases:")
                print("-" * 30)
                if phrases:
                    for phrase, count in phrases:
                        print(f"{phrase.replace('_', ' '):<40} {count}")
                else:
                    print("No significant phrases found")
            print("\n")

In [ ]:
processor = PhraseModelProcessor(corpus=corpus_gensim_words_sents)

In [ ]:
processor.build_models(doc_filter='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib')  # Using regex pattern

In [ ]:
processor.save_models()

In [ ]:
# Analyze multiple documents
phrase_results = processor.analyze_documents(
    doc_filter='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib',
    min_frequency=2,
    top_n=10
)

In [ ]:
print_phrase_analysis(phrase_results)[0]

In [ ]:
from scipy import stats
import numpy as np
from typing import List, Dict, Tuple, Set, Union
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

class PhraseComparison:
    def __init__(self, phrase_results: Dict[str, Dict[int, List[Tuple[str, int]]]]):
        """
        Initialize with phrase analysis results from multiple documents.
        
        Args:
            phrase_results: Dictionary mapping document IDs to their phrase analysis results
                          (output from PhraseModelProcessor.analyze_documents)
        """
        self.results = phrase_results
        self.doc_ids = list(phrase_results.keys())
        
    def _get_phrase_ranks(self, doc_id: str, n_gram: int) -> Dict[str, int]:
        """Get phrases with their ranks for a specific document and n-gram size."""
        phrases = self.results[doc_id][n_gram]
        return {phrase: rank + 1 for rank, (phrase, _) in enumerate(phrases)}
    
    def _get_phrase_frequencies(self, doc_id: str, n_gram: int) -> Dict[str, int]:
        """Get phrases with their frequencies for a specific document and n-gram size."""
        return dict(self.results[doc_id][n_gram])

    def compute_rank_correlation(self, 
                               doc1_id: str, 
                               doc2_id: str, 
                               n_gram: int,
                               method: str = 'spearman') -> Tuple[float, float]:
        """
        Compute rank correlation between phrases in two documents.
        
        Args:
            doc1_id: First document ID
            doc2_id: Second document ID
            n_gram: Size of n-grams to compare
            method: 'spearman' or 'kendall'
            
        Returns:
            Tuple of (correlation coefficient, p-value)
        """
        phrases1 = self._get_phrase_ranks(doc1_id, n_gram)
        phrases2 = self._get_phrase_ranks(doc2_id, n_gram)
        
        # Get common phrases
        common_phrases = set(phrases1.keys()) & set(phrases2.keys())
        
        if len(common_phrases) < 2:
            return (0.0, 1.0)
        
        # Extract ranks for common phrases
        ranks1 = [phrases1[phrase] for phrase in common_phrases]
        ranks2 = [phrases2[phrase] for phrase in common_phrases]
        
        if method == 'spearman':
            return stats.spearmanr(ranks1, ranks2)
        else:
            return stats.kendalltau(ranks1, ranks2)

    def compute_overlap_coefficient(self, 
                                  doc1_id: str, 
                                  doc2_id: str, 
                                  n_gram: int,
                                  top_k: int = None) -> float:
        """
        Compute overlap coefficient between phrase sets.
        
        Args:
            doc1_id: First document ID
            doc2_id: Second document ID
            n_gram: Size of n-grams to compare
            top_k: If provided, only consider top k phrases
            
        Returns:
            Overlap coefficient (intersection size / size of smaller set)
        """
        phrases1 = set(dict(self.results[doc1_id][n_gram][:top_k]).keys())
        phrases2 = set(dict(self.results[doc2_id][n_gram][:top_k]).keys())
        
        intersection = len(phrases1 & phrases2)
        smaller_set = min(len(phrases1), len(phrases2))
        
        return intersection / smaller_set if smaller_set > 0 else 0.0

    def compute_rbo(self, 
                    doc1_id: str, 
                    doc2_id: str, 
                    n_gram: int, 
                    p: float = 0.9) -> float:
        """
        Compute Rank-Biased Overlap (RBO) between two ranked lists.
        RBO is particularly suitable for comparing indefinite ranked lists
        and gives more weight to higher ranks.
        
        Args:
            doc1_id: First document ID
            doc2_id: Second document ID
            n_gram: Size of n-grams to compare
            p: Persistence parameter (higher values give more weight to lower ranks)
            
        Returns:
            RBO score between 0 and 1
        """
        phrases1 = [phrase for phrase, _ in self.results[doc1_id][n_gram]]
        phrases2 = [phrase for phrase, _ in self.results[doc2_id][n_gram]]
        
        score = 0.0
        overlap_sizes = []
        
        for depth in range(1, max(len(phrases1), len(phrases2)) + 1):
            set1 = set(phrases1[:depth])
            set2 = set(phrases2[:depth])
            overlap = len(set1 & set2)
            overlap_sizes.append(overlap / depth)
        
        # Calculate RBO
        weight = 1
        for d, overlap in enumerate(overlap_sizes):
            score += weight * overlap
            weight *= p
            
        return score * (1 - p)

    def create_similarity_matrix(self, 
                               n_gram: int, 
                               metric: str = 'rbo',
                               **metric_params) -> pd.DataFrame:
        """
        Create a similarity matrix between all documents for a specific n-gram size.
        
        Args:
            n_gram: Size of n-grams to compare
            metric: 'rbo', 'spearman', 'kendall', or 'overlap'
            metric_params: Additional parameters for the chosen metric
            
        Returns:
            Pandas DataFrame containing similarity scores
        """
        n_docs = len(self.doc_ids)
        matrix = np.zeros((n_docs, n_docs))
        
        for i, doc1 in enumerate(self.doc_ids):
            for j, doc2 in enumerate(self.doc_ids):
                if i == j:
                    matrix[i, j] = 1.0
                else:
                    if metric == 'rbo':
                        score = self.compute_rbo(doc1, doc2, n_gram, **metric_params)
                    elif metric in ['spearman', 'kendall']:
                        score, _ = self.compute_rank_correlation(doc1, doc2, n_gram, method=metric)
                    elif metric == 'overlap':
                        score = self.compute_overlap_coefficient(doc1, doc2, n_gram, **metric_params)
                    
                    matrix[i, j] = score
        
        return pd.DataFrame(matrix, index=self.doc_ids, columns=self.doc_ids)

    def plot_similarity_heatmap(self, 
                              similarity_matrix: pd.DataFrame,
                              title: str = None):
        """
        Plot a heatmap of the similarity matrix.
        
        Args:
            similarity_matrix: Output from create_similarity_matrix
            title: Optional title for the plot
        """
        plt.figure(figsize=(10, 8))
        sns.heatmap(similarity_matrix, 
                   annot=True, 
                   cmap='YlOrRd', 
                   vmin=0, 
                   vmax=1,
                   fmt='.2f')
        
        if title:
            plt.title(title)
        plt.tight_layout()
        plt.show()

    def find_distinctive_phrases(self, 
                               doc_id: str, 
                               n_gram: int,
                               comparison_docs: List[str] = None,
                               method: str = 'frequency_ratio') -> List[Tuple[str, float]]:
        """
        Find phrases that are distinctive/unique to a document compared to others.
        
        Args:
            doc_id: Target document ID
            n_gram: Size of n-grams to analyze
            comparison_docs: List of documents to compare against (default: all others)
            method: 'frequency_ratio' or 'rank_difference'
            
        Returns:
            List of (phrase, distinctiveness_score) tuples
        """
        if comparison_docs is None:
            comparison_docs = [d for d in self.doc_ids if d != doc_id]
            
        target_phrases = self._get_phrase_frequencies(doc_id, n_gram)
        
        if method == 'frequency_ratio':
            # Compare frequency ratios
            distinctiveness = {}
            for phrase in target_phrases:
                target_freq = target_phrases[phrase]
                other_freqs = []
                
                for other_doc in comparison_docs:
                    other_dict = self._get_phrase_frequencies(other_doc, n_gram)
                    if phrase in other_dict:
                        other_freqs.append(other_dict[phrase])
                
                if other_freqs:
                    avg_other_freq = sum(other_freqs) / len(other_freqs)
                    distinctiveness[phrase] = target_freq / (avg_other_freq + 1)
                else:
                    distinctiveness[phrase] = float('inf')
                    
        elif method == 'rank_difference':
            # Compare rank differences
            target_ranks = self._get_phrase_ranks(doc_id, n_gram)
            distinctiveness = {}
            
            for phrase in target_phrases:
                target_rank = target_ranks[phrase]
                other_ranks = []
                
                for other_doc in comparison_docs:
                    other_dict = self._get_phrase_ranks(other_doc, n_gram)
                    if phrase in other_dict:
                        other_ranks.append(other_dict[phrase])
                
                if other_ranks:
                    avg_other_rank = sum(other_ranks) / len(other_ranks)
                    distinctiveness[phrase] = avg_other_rank - target_rank
                else:
                    distinctiveness[phrase] = float('inf')
        
        # Sort by distinctiveness score
        distinctive_phrases = sorted(distinctiveness.items(), 
                                  key=lambda x: x[1], 
                                  reverse=True)
        
        return distinctive_phrases

In [ ]:
comparison = PhraseComparison(phrase_results)

In [ ]:
similarity_measures = calculate_similarity_measures(phrases_list)

In [ ]:
# Create similarity matrices using different metrics
rbo_matrix = comparison.create_similarity_matrix(n_gram=2, metric='rbo', p=0.9)
correlation_matrix = comparison.create_similarity_matrix(n_gram=2, metric='spearman')
overlap_matrix = comparison.create_similarity_matrix(n_gram=2, metric='overlap', top_k=10)

In [ ]:
# Visualize similarities
comparison.plot_similarity_heatmap(rbo_matrix, "RBO Similarity (2-grams)")
comparison.plot_similarity_heatmap(correlation_matrix, "Rank Correlation (2-grams)")
comparison.plot_similarity_heatmap(overlap_matrix, "Overlap Similarity (2-grams)")

In [ ]:
# Find distinctive phrases for each document
for doc_id in comparison.doc_ids:
    distinctive_phrases = comparison.find_distinctive_phrases(
        doc_id, n_gram=2, method='frequency_ratio'
    )
    print(f"\nDistinctive phrases for {doc_id}:")
    for phrase, score in distinctive_phrases[:10]:
        print(f"{phrase}: {score:.2f}")

In [ ]:
from scipy import stats
import numpy as np
from typing import List, Dict, Tuple, Set, Union
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
import networkx as nx
from collections import defaultdict
import colorsys

class PhraseVisualization(PhraseComparison):
    def __init__(self, phrase_results: Dict[str, Dict[int, List[Tuple[str, int]]]]):
        """
        Extends PhraseComparison with additional visualization capabilities.
        """
        super().__init__(phrase_results)
        
    def _generate_distinct_colors(self, n: int) -> List[str]:
        """Generate n visually distinct colors."""
        colors = []
        for i in range(n):
            hue = i / n
            saturation = 0.7 + np.random.random() * 0.3
            value = 0.7 + np.random.random() * 0.3
            rgb = colorsys.hsv_to_rgb(hue, saturation, value)
            colors.append(f'#{int(rgb[0]*255):02x}{int(rgb[1]*255):02x}{int(rgb[2]*255):02x}')
        return colors

    def create_phrase_network(self,
                            n_gram: int,
                            min_overlap: int = 1,
                            top_k: int = None,
                            layout: str = 'spring') -> Tuple[nx.Graph, plt.Figure]:
        """
        Create a network visualization where nodes are phrases and edges represent
        co-occurrence in documents.
        
        Args:
            n_gram: Size of n-grams to visualize
            min_overlap: Minimum number of documents that must share a phrase for an edge
            top_k: If provided, only consider top k phrases per document
            layout: NetworkX layout algorithm ('spring', 'kamada_kawai', 'circular')
        
        Returns:
            Tuple of (network graph, figure)
        """
        # Create graph
        G = nx.Graph()
        
        # Track phrase occurrences and document associations
        phrase_docs = defaultdict(set)
        doc_colors = self._generate_distinct_colors(len(self.doc_ids))
        doc_color_map = dict(zip(self.doc_ids, doc_colors))
        
        # Collect phrases and their documents
        for doc_id in self.doc_ids:
            phrases = self.results[doc_id][n_gram][:top_k]
            for phrase, freq in phrases:
                phrase_docs[phrase].add(doc_id)
                
        # Add nodes (phrases)
        for phrase, docs in phrase_docs.items():
            # Node color is based on the document where the phrase has highest frequency
            max_freq_doc = max(docs, key=lambda doc: dict(self.results[doc][n_gram])[phrase])
            G.add_node(phrase, 
                      color=doc_color_map[max_freq_doc],
                      docs=list(docs),
                      size=len(docs) * 100)  # Node size based on document count
            
        # Add edges between phrases that co-occur
        phrases = list(phrase_docs.keys())
        for i in range(len(phrases)):
            for j in range(i + 1, len(phrases)):
                common_docs = phrase_docs[phrases[i]] & phrase_docs[phrases[j]]
                if len(common_docs) >= min_overlap:
                    G.add_edge(phrases[i], phrases[j], 
                             weight=len(common_docs),
                             width=len(common_docs) * 0.5)
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(15, 12))
        
        # Set layout
        if layout == 'spring':
            pos = nx.spring_layout(G, k=1/np.sqrt(len(G.nodes())), iterations=50)
        elif layout == 'kamada_kawai':
            pos = nx.kamada_kawai_layout(G)
        else:  # circular
            pos = nx.circular_layout(G)
            
        # Draw network
        nodes = nx.draw_networkx_nodes(G, pos,
                                     node_color=[G.nodes[node]['color'] for node in G.nodes()],
                                     node_size=[G.nodes[node]['size'] for node in G.nodes()],
                                     alpha=0.7)
        
        edges = nx.draw_networkx_edges(G, pos,
                                     width=[G.edges[edge]['width'] for edge in G.edges()],
                                     alpha=0.4)
        
        # Add labels with smaller font for readability
        nx.draw_networkx_labels(G, pos, font_size=8)
        
        # Add legend for documents
        legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                                    markerfacecolor=color, label=doc_id, markersize=10)
                         for doc_id, color in doc_color_map.items()]
        plt.legend(handles=legend_elements, title='Documents', 
                  loc='center left', bbox_to_anchor=(1, 0.5))
        
        title = f'Phrase Network ({n_gram}-grams)\nMin overlap: {min_overlap}'
        plt.title(title)
        plt.axis('off')
        plt.tight_layout()
        
        return G, fig

    def cluster_documents(self, 
                         n_gram: int,
                         method: str = 'mds',
                         metric: str = 'rbo',
                         **metric_params) -> Tuple[np.ndarray, plt.Figure]:
        """
        Cluster documents using dimensionality reduction techniques.
        
        Args:
            n_gram: Size of n-grams to compare
            method: 'pca', 'mds', or 'tsne'
            metric: Similarity metric to use
            metric_params: Additional parameters for the similarity metric
            
        Returns:
            Tuple of (coordinates, figure)
        """
        # Get similarity matrix
        sim_matrix = self.create_similarity_matrix(n_gram, metric, **metric_params)
        
        # Convert similarity to distance (if using similarity-based metric)
        if metric in ['rbo', 'overlap']:
            dist_matrix = 1 - sim_matrix
        else:
            dist_matrix = (1 - sim_matrix.abs())
        
        n_samples = len(self.doc_ids)
        
        # Apply dimensionality reduction
        if method == 'pca':
            reducer = PCA(n_components=min(2, n_samples))
            coords = reducer.fit_transform(dist_matrix)
        elif method == 'mds':
            reducer = MDS(n_components=min(2, n_samples), 
                        dissimilarity='precomputed', 
                        random_state=42)
            coords = reducer.fit_transform(dist_matrix)
        else:  # tsne
            # Adjust perplexity based on number of samples
            perplexity = min(30, n_samples - 1)  # Default is 30
            if n_samples < 4:
                # For very small datasets, fall back to MDS
                print("Warning: Too few samples for t-SNE, falling back to MDS")
                reducer = MDS(n_components=min(2, n_samples), 
                            dissimilarity='precomputed', 
                            random_state=42)
            else:
                reducer = TSNE(n_components=2, 
                             metric='precomputed',
                             perplexity=perplexity,
                             random_state=42)
            coords = reducer.fit_transform(dist_matrix)
            
        # Create visualization
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # If we have only one dimension (rare case with 2 samples in PCA)
        if coords.shape[1] == 1:
            coords = np.column_stack((coords, np.zeros_like(coords)))
            
        scatter = ax.scatter(coords[:, 0], coords[:, 1], s=100)
        
        # Add labels
        for i, doc_id in enumerate(self.doc_ids):
            ax.annotate(doc_id, (coords[i, 0], coords[i, 1]), 
                       xytext=(5, 5), textcoords='offset points')
            
        title = f'Document Clustering ({n_gram}-grams)\nMethod: {method.upper()}, Metric: {metric}'
        plt.title(title)
        plt.tight_layout()
        
        return coords, fig

    def analyze_network_metrics(self, G: nx.Graph) -> pd.DataFrame:
        """
        Compute various network metrics for the phrase network.
        
        Args:
            G: NetworkX graph from create_phrase_network
            
        Returns:
            DataFrame with network metrics
        """
        metrics = {
            'degree_centrality': nx.degree_centrality(G),
            'betweenness_centrality': nx.betweenness_centrality(G),
            'eigenvector_centrality': nx.eigenvector_centrality(G, max_iter=1000),
            'clustering_coefficient': nx.clustering(G)
        }
        
        # Convert to DataFrame
        df = pd.DataFrame(metrics)
        
        # Add document information
        df['documents'] = [', '.join(G.nodes[node]['docs']) for node in df.index]
        df['num_documents'] = [len(G.nodes[node]['docs']) for node in df.index]
        
        return df.sort_values('degree_centrality', ascending=False)

In [ ]:
visualizer = PhraseVisualization(phrase_results)

# Try different visualization methods
for method in ['pca', 'mds']:  # Removed 'tsne' if you have very few documents
    coords, fig = visualizer.cluster_documents(
        n_gram=2,
        method=method,
        metric='rbo',
        p=0.9
    )
    plt.show()
    

In [ ]:
# 2. Network Visualization
# Create phrase network
G, fig = visualizer.create_phrase_network(
    n_gram=2,
    min_overlap=2,  # Minimum number of documents that must share a phrase
    top_k=20,       # Only consider top 20 phrases per document
    layout='spring'  # Try different layouts: 'spring', 'kamada_kawai', 'circular'
)
plt.show()

# Analyze network metrics
metrics_df = visualizer.analyze_network_metrics(G)
print("\nTop phrases by centrality:")
print(metrics_df.head())

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.manifold import MDS, TSNE
import umap
import prince
from typing import List, Dict, Tuple, Union, Optional
import logging

class EnhancedPhraseVisualization(PhraseComparison):
    def __init__(self, phrase_results: Dict[str, Dict[int, List[Tuple[str, int]]]], random_state: int = 42):
        super().__init__(phrase_results)
        self.random_state = random_state
        
    def create_feature_matrix(self, n_gram: int) -> pd.DataFrame:
        """Create a feature matrix from phrase frequencies."""
        data = []
        for doc_id in self.doc_ids:
            phrases = dict(self.results[doc_id][n_gram])
            data.append(phrases)
        
        feature_matrix = pd.DataFrame(data, index=self.doc_ids)
        feature_matrix = feature_matrix.fillna(0)
        return feature_matrix
    
    def cluster_and_visualize(self,
                            n_gram: int,
                            clustering_method: str = 'kmeans',
                            dim_reduction: str = 'pca',
                            n_clusters: int = 3,
                            plot_type: str = 'plotly',
                            plot_centers: bool = True,
                            plot_features: bool = True,
                            n_top_features: int = 5,
                            height: int = 800,
                            return_fig: bool = False,
                            **kwargs) -> Union[Tuple[pd.DataFrame, any, any, any], 
                                            Tuple[pd.DataFrame, any, any, any, go.Figure]]:
        """
        Cluster documents and create interactive visualization.
        
        Args:
            n_gram: Size of n-grams to analyze
            clustering_method: 'kmeans', 'dbscan', or 'hierarchical'
            dim_reduction: 'pca', 'mca', 'tsne', 'umap', or 'mds'
            n_clusters: Number of clusters for kmeans/hierarchical
            plot_type: 'plotly' or 'matplotlib'
            plot_centers: Whether to plot cluster centers
            plot_features: Whether to plot feature vectors (PCA/MCA only)
            n_top_features: Number of top features to plot
            perplexity: t-SNE perplexity parameter
            n_neighbors: UMAP n_neighbors parameter
            min_dist: UMAP min_dist parameter
            dbscan_eps: DBSCAN eps parameter
            dbscan_min_samples: DBSCAN min_samples parameter
        """
        # Create and scale feature matrix
        feature_matrix = self.create_feature_matrix(n_gram)
        
        if dim_reduction != 'mca':
            scaler = StandardScaler()
            scaled_features = scaler.fit_transform(feature_matrix)
        else:
            scaled_features = feature_matrix
            
        # Dimensionality reduction
        if dim_reduction == 'pca':
            reducer = PCA(n_components=2, random_state=self.random_state)
            reduced_features = reducer.fit_transform(scaled_features)
            
        elif dim_reduction == 'mca':
            n_components = min(2, len(feature_matrix.columns) - 1)
            reducer = prince.MCA(n_components=n_components, random_state=self.random_state)
            reduced_features = reducer.fit_transform(feature_matrix).values
            
        elif dim_reduction == 'tsne':
            perplexity = min(perplexity, len(self.doc_ids) - 1)
            reducer = TSNE(n_components=2, perplexity=perplexity, 
                         random_state=self.random_state)
            reduced_features = reducer.fit_transform(scaled_features)
            
        elif dim_reduction == 'umap':
            reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors,
                              min_dist=min_dist, random_state=self.random_state)
            reduced_features = reducer.fit_transform(scaled_features)
            
        elif dim_reduction == 'mds':
            reducer = MDS(n_components=2, random_state=self.random_state)
            reduced_features = reducer.fit_transform(scaled_features)
            
        # Clustering
        n_docs = len(feature_matrix)
        if n_docs < n_clusters and clustering_method != 'dbscan':
            logging.warning(f"Reducing number of clusters to {n_docs}")
            n_clusters = n_docs
            
        if clustering_method == 'kmeans':
            clusterer = KMeans(n_clusters=n_clusters, random_state=self.random_state)
            clusters = clusterer.fit_predict(scaled_features)
        elif clustering_method == 'dbscan':
            clusterer = DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples)
            clusters = clusterer.fit_predict(scaled_features)
        elif clustering_method == 'hierarchical':
            clusterer = AgglomerativeClustering(n_clusters=n_clusters)
            clusters = clusterer.fit_predict(scaled_features)
            
        # Create plot data
        plot_data = pd.DataFrame({
            'Dim1': reduced_features[:, 0],
            'Dim2': reduced_features[:, 1],
            'Cluster': clusters,
            'Document': feature_matrix.index
        })
        
        if plot_type == 'plotly':
            self._create_plotly_visualization(
                plot_data, feature_matrix, reducer, dim_reduction,
                clustering_method, plot_centers, plot_features, n_top_features
            )
        if plot_type == 'plotly':
            fig = self._create_plotly_visualization(
                plot_data, feature_matrix, reducer, dim_reduction,
                clustering_method, plot_centers, plot_features, 
                n_top_features, height=height
            )
            
            if not return_fig:
                fig.show()
                return plot_data, feature_matrix, clusterer, reducer
            else:
                return plot_data, feature_matrix, clusterer, reducer, fig

        else:
            self._create_matplotlib_visualization(
                plot_data, feature_matrix, reducer, dim_reduction,
                clustering_method, plot_centers, plot_features, n_top_features
            )
            
        # Analyze and report clusters
        self._analyze_clusters(plot_data, feature_matrix, clusterer, reducer, 
                             dim_reduction, clustering_method, n_top_features)
        
        return plot_data, feature_matrix, clusterer, reducer
    
    def _create_plotly_visualization(self, plot_data, feature_matrix, reducer, 
                                   dim_reduction, clustering_method, plot_centers,
                                   plot_features, n_top_features, height=800):
        """
        Create interactive Plotly visualization with controls and customizable height.
        
        Args:
            height: Height of the plot in pixels
        """
        # Calculate plot boundaries for scaling feature vectors
        x_range = plot_data['Dim1'].max() - plot_data['Dim1'].min()
        y_range = plot_data['Dim2'].max() - plot_data['Dim2'].min()
        scale_factor = min(x_range, y_range) * 0.2

        # Base scatter plot
        fig = px.scatter(plot_data, x='Dim1', y='Dim2', 
                        color='Cluster', symbol='Cluster',
                        hover_data=['Document'])

        # Create buttons for controlling visibility
        updatemenus = [
            # Cluster centers visibility button
            dict(
                type="buttons",
                direction="left",
                buttons=[
                    dict(label="Show Centers",
                         method="update",
                         args=[{"visible": [True] * len(fig.data)}]),
                    dict(label="Hide Centers",
                         method="update",
                         args=[{"visible": [True if i < len(plot_data['Cluster'].unique()) 
                                          else False for i in range(len(fig.data))]}])
                ],
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.1,
                xanchor="left",
                y=1.1,
                yanchor="top"
            ),
            # Document labels visibility button
            dict(
                type="buttons",
                direction="left",
                buttons=[
                    dict(label="Show Labels",
                         method="update",
                         args=[{}, {"annotations": fig.layout.annotations}]),
                    dict(label="Hide Labels",
                         method="update",
                         args=[{}, {"annotations": []}])
                ],
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.3,
                xanchor="left",
                y=1.1,
                yanchor="top"
            )
        ]

        if plot_features and dim_reduction in ['pca', 'mca']:
            # Add feature vectors button
            updatemenus.append(
                dict(
                    type="buttons",
                    direction="left",
                    buttons=[
                        dict(label="Show Features",
                             method="update",
                             args=[{"visible": [True] * len(fig.data)}]),
                        dict(label="Hide Features",
                             method="update",
                             args=[{"visible": [i < len(plot_data['Cluster'].unique()) + 1 
                                              for i in range(len(fig.data))]}])
                    ],
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.5,
                    xanchor="left",
                    y=1.1,
                    yanchor="top"
                )
            )

        # Add document labels
        annotations = []
        for idx, row in plot_data.iterrows():
            annotations.append(
                dict(
                    x=row['Dim1'],
                    y=row['Dim2'],
                    text=str(row['Document']),
                    showarrow=True,
                    arrowhead=0,
                    ax=10,
                    ay=10,
                    font=dict(size=10)
                )
            )

        # Add cluster centers if requested
        if plot_centers and clustering_method != 'dbscan':
            centers = plot_data.groupby('Cluster')[['Dim1', 'Dim2']].mean()
            fig.add_trace(go.Scatter(
                x=centers['Dim1'],
                y=centers['Dim2'],
                mode='markers',
                marker=dict(symbol='star', size=15, color='red'),
                name='Cluster Centers',
                showlegend=True
            ))

        # Add feature vectors for PCA/MCA
        if plot_features and dim_reduction in ['pca', 'mca']:
            if dim_reduction == 'pca':
                loadings = reducer.components_.T * np.sqrt(reducer.explained_variance_)
                feature_importance = np.sum(loadings**2, axis=1)
            else:
                coord = reducer.column_coordinates(feature_matrix)
                loadings = coord.values
                feature_importance = np.sum(loadings**2, axis=1)

            top_indices = np.argsort(feature_importance)[-n_top_features:]
            
            for idx in top_indices:
                if idx >= len(feature_matrix.columns):
                    continue

                feature_name = feature_matrix.columns[idx]
                x = loadings[idx, 0] * scale_factor
                y = loadings[idx, 1] * scale_factor

                # Add arrow
                fig.add_trace(go.Scatter(
                    x=[0, x],
                    y=[0, y],
                    mode='lines+text',
                    name=feature_name,
                    line=dict(color='darkred', width=1),
                    text=['', feature_name],
                    textposition='top center',
                    showlegend=True
                ))

        # Update layout with buttons and custom height
        fig.update_layout(
            updatemenus=updatemenus,
            annotations=annotations,
            height=height,
            title=dict(
                text=f'Document Clusters ({dim_reduction.upper()})',
                y=0.95,
                x=0.5,
                xanchor='center',
                yanchor='top'
            ),
            # Add sliders for controlling visualization parameters
            sliders=[
                dict(
                    active=1,
                    currentvalue={"prefix": "Point Size: "},
                    pad={"t": 50},
                    steps=[dict(
                        method='update',
                        args=[{'marker.size': size}],
                        label=str(size)
                    ) for size in [5, 10, 15, 20, 25]]
                )
            ],
            legend=dict(
                yanchor="top",
                y=0.99,
                xanchor="left",
                x=1.05
            ),
            margin=dict(t=150)  # Increase top margin to accommodate buttons
        )

        # Add hover template
        fig.update_traces(
            hovertemplate="<br>".join([
                "Document: %{customdata[0]}",
                "Dimension 1: %{x:.2f}",
                "Dimension 2: %{y:.2f}",
                "Cluster: %{marker.color}"
            ])
        )

        return fig

    
    def _analyze_clusters(self, plot_data, feature_matrix, clusterer, reducer,
                         dim_reduction, clustering_method, n_top_features):
        """Analyze and print cluster information."""
        labels = clusterer.labels_
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        
        print("\nAnalysis Details:")
        print("----------------")
        print(f"Dimensionality Reduction: {dim_reduction.upper()}")
        print(f"Clustering Method: {clustering_method}")
        print(f"Number of clusters: {n_clusters}")
        
        # Method-specific information
        if dim_reduction == 'pca':
            var_explained = reducer.explained_variance_ratio_
            print("\nPCA Variance Explained:")
            print(f"Total (2 components): {sum(var_explained):.1%}")
            print(f"Component 1: {var_explained[0]:.1%}")
            print(f"Component 2: {var_explained[1]:.1%}")
            
        elif dim_reduction == 'mca':
            print("\nMCA Analysis:")
            try:
                eigenvalues = reducer.eigenvalues_
                if hasattr(reducer, 'total_inertia_'):
                    ratios = eigenvalues / reducer.total_inertia_
                    print(f"Total inertia explained: {sum(ratios):.1%}")
                    for i, ratio in enumerate(ratios):
                        print(f"Dimension {i+1}: {ratio:.1%}")
            except Exception as e:
                print(f"Could not calculate MCA statistics: {str(e)}")
        
        # Cluster analysis
        print("\nCluster Details:")
        print("---------------")
        analysis_df = feature_matrix.copy()
        analysis_df['Cluster'] = labels
        
        if clustering_method == 'dbscan':
            noise_points = (labels == -1).sum()
            print(f"\nNumber of noise points: {noise_points}")
        
        for cluster in sorted(set(labels)):
            if clustering_method == 'dbscan' and cluster == -1:
                print("\nNoise points:")
                cluster_docs = plot_data[plot_data['Cluster'] == -1]['Document'].tolist()
                print("Documents:", ", ".join(map(str, cluster_docs)))
                continue
                
            print(f"\nCluster {cluster}:")
            cluster_docs = plot_data[plot_data['Cluster'] == cluster]['Document'].tolist()
            print("Documents:", ", ".join(map(str, cluster_docs)))
            print(f"Size: {len(cluster_docs)} documents")
            
            cluster_data = analysis_df[analysis_df['Cluster'] == cluster]
            cluster_means = cluster_data.drop('Cluster', axis=1).mean()
            
            top_features = cluster_means.sort_values(ascending=False)[:3]
            print("Characteristic features:")
            for feature, value in top_features.items():
                overall_mean = feature_matrix[feature].mean()
                diff = value - overall_mean
                print(f"  - {feature}: {value:.3f} (diff from mean: {diff:+.3f})")

In [ ]:
#rename

new_dict = {}
for key, value in phrase_results.items():
    new_dict[key.replace('_TEI_final.conllu', '')] = value

In [ ]:
del phrase_results
phrase_results = new_dict

In [ ]:
# Initialize the visualizer
visualizer = EnhancedPhraseVisualization(phrase_results)
#perplexity = 5
#dbscan_eps = 0.3
#dbscan_min_samples = 5
#clusters = 5
# Create interactive visualization with cluster analysis
plot_data, feature_matrix, clusterer, reducer = visualizer.cluster_and_visualize(
    n_gram=3,                    # Analyze bigrams
    clustering_method='kmeans',  # Use k-means clustering
    dim_reduction='pca',        # Use PCA for dimensionality reduction
    n_clusters=3,               # Number of clusters
    plot_type='plotly',        # Create interactive Plotly visualization
    plot_centers=True,         # Show cluster centers
    plot_features=False,        # Show feature vectors
    n_top_features=5 ,         # Number of top features to display,
)

In [ ]:
# Try different dimensionality reduction methods
methods = ['pca', 'mca', 'tsne', 'umap', 'mds']
clustering = ['kmeans', 'hierarchical']
ns = [2,3,4]
ns_clusters = [3,4,5]

for n in ns:
    for clust in clustering:
        for method in methods:
            for n_clust in ns_clusters:
                print("ngrams: " +str(n) + " clust: " + clust + " n clust: " + str(n_clust) + " method: " + method)
                try:
                    #print("ngrams: " + n + " clust: " + str(clust) + " method: " + method)
                    visualizer.cluster_and_visualize(
                        n_gram=n,
                        n_clusters = n_clust,
                        dim_reduction=method,
                        clustering_method=clust,
                        plot_type='plotly'
                    )
                except:
                    print('error')

In [ ]:
from scipy import stats
import numpy as np
from typing import List, Dict, Tuple, Set, Union
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
import networkx as nx
from collections import defaultdict
import colorsys

class PhraseVisualization(PhraseComparison):
    def __init__(self, phrase_results: Dict[str, Dict[int, List[Tuple[str, int]]]]):
        """
        Extends PhraseComparison with additional visualization capabilities.
        """
        super().__init__(phrase_results)
        
    def _generate_distinct_colors(self, n: int) -> List[str]:
        """Generate n visually distinct colors."""
        colors = []
        for i in range(n):
            hue = i / n
            saturation = 0.7 + np.random.random() * 0.3
            value = 0.7 + np.random.random() * 0.3
            rgb = colorsys.hsv_to_rgb(hue, saturation, value)
            colors.append(f'#{int(rgb[0]*255):02x}{int(rgb[1]*255):02x}{int(rgb[2]*255):02x}')
        return colors

    def cluster_documents(self, 
                         n_gram: int,
                         method: str = 'mds',
                         metric: str = 'rbo',
                         **metric_params) -> Tuple[np.ndarray, plt.Figure]:
        """
        Cluster documents using dimensionality reduction techniques.
        
        Args:
            n_gram: Size of n-grams to compare
            method: 'pca', 'mds', or 'tsne'
            metric: Similarity metric to use
            metric_params: Additional parameters for the similarity metric
            
        Returns:
            Tuple of (coordinates, figure)
        """
        # Get similarity matrix
        sim_matrix = self.create_similarity_matrix(n_gram, metric, **metric_params)
        
        # Convert similarity to distance (if using similarity-based metric)
        if metric in ['rbo', 'overlap']:
            dist_matrix = 1 - sim_matrix
        else:
            dist_matrix = (1 - sim_matrix.abs())
        
        # Apply dimensionality reduction
        if method == 'pca':
            reducer = PCA(n_components=2)
            coords = reducer.fit_transform(dist_matrix)
        elif method == 'mds':
            reducer = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
            coords = reducer.fit_transform(dist_matrix)
        else:  # tsne
            reducer = TSNE(n_components=2, metric='precomputed', random_state=42)
            coords = reducer.fit_transform(dist_matrix)
            
        # Create visualization
        fig, ax = plt.subplots(figsize=(10, 8))
        scatter = ax.scatter(coords[:, 0], coords[:, 1], s=100)
        
        # Add labels
        for i, doc_id in enumerate(self.doc_ids):
            ax.annotate(doc_id, (coords[i, 0], coords[i, 1]), 
                       xytext=(5, 5), textcoords='offset points')
            
        title = f'Document Clustering ({n_gram}-grams)\nMethod: {method.upper()}, Metric: {metric}'
        plt.title(title)
        plt.tight_layout()
        
        return coords, fig

    def create_phrase_network(self,
                            n_gram: int,
                            min_overlap: int = 1,
                            top_k: int = None,
                            layout: str = 'spring') -> Tuple[nx.Graph, plt.Figure]:
        """
        Create a network visualization where nodes are phrases and edges represent
        co-occurrence in documents.
        
        Args:
            n_gram: Size of n-grams to visualize
            min_overlap: Minimum number of documents that must share a phrase for an edge
            top_k: If provided, only consider top k phrases per document
            layout: NetworkX layout algorithm ('spring', 'kamada_kawai', 'circular')
        
        Returns:
            Tuple of (network graph, figure)
        """
        # Create graph
        G = nx.Graph()
        
        # Track phrase occurrences and document associations
        phrase_docs = defaultdict(set)
        doc_colors = self._generate_distinct_colors(len(self.doc_ids))
        doc_color_map = dict(zip(self.doc_ids, doc_colors))
        
        # Collect phrases and their documents
        for doc_id in self.doc_ids:
            phrases = self.results[doc_id][n_gram][:top_k]
            for phrase, freq in phrases:
                phrase_docs[phrase].add(doc_id)
                
        # Add nodes (phrases)
        for phrase, docs in phrase_docs.items():
            # Node color is based on the document where the phrase has highest frequency
            max_freq_doc = max(docs, key=lambda doc: dict(self.results[doc][n_gram])[phrase])
            G.add_node(phrase, 
                      color=doc_color_map[max_freq_doc],
                      docs=list(docs),
                      size=len(docs) * 100)  # Node size based on document count
            
        # Add edges between phrases that co-occur
        phrases = list(phrase_docs.keys())
        for i in range(len(phrases)):
            for j in range(i + 1, len(phrases)):
                common_docs = phrase_docs[phrases[i]] & phrase_docs[phrases[j]]
                if len(common_docs) >= min_overlap:
                    G.add_edge(phrases[i], phrases[j], 
                             weight=len(common_docs),
                             width=len(common_docs) * 0.5)
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(15, 12))
        
        # Set layout
        if layout == 'spring':
            pos = nx.spring_layout(G, k=1/np.sqrt(len(G.nodes())), iterations=50)
        elif layout == 'kamada_kawai':
            pos = nx.kamada_kawai_layout(G)
        else:  # circular
            pos = nx.circular_layout(G)
            
        # Draw network
        nodes = nx.draw_networkx_nodes(G, pos,
                                     node_color=[G.nodes[node]['color'] for node in G.nodes()],
                                     node_size=[G.nodes[node]['size'] for node in G.nodes()],
                                     alpha=0.7)
        
        edges = nx.draw_networkx_edges(G, pos,
                                     width=[G.edges[edge]['width'] for edge in G.edges()],
                                     alpha=0.4)
        
        # Add labels with smaller font for readability
        nx.draw_networkx_labels(G, pos, font_size=8)
        
        # Add legend for documents
        legend_elements = [plt.Line2D([0], [0], marker='o', color='w', 
                                    markerfacecolor=color, label=doc_id, markersize=10)
                         for doc_id, color in doc_color_map.items()]
        plt.legend(handles=legend_elements, title='Documents', 
                  loc='center left', bbox_to_anchor=(1, 0.5))
        
        title = f'Phrase Network ({n_gram}-grams)\nMin overlap: {min_overlap}'
        plt.title(title)
        plt.axis('off')
        plt.tight_layout()
        
        return G, fig

    def create_interactive_phrase_network(self,
                                    n_gram: int,
                                    min_overlap: int = 1,
                                    top_k: None = None) -> nx.Graph:
        """
        Create an interactive network visualization using Plotly where nodes are phrases
        and edges represent co-occurrence in documents.
        
        Args:
            n_gram: Size of n-grams to visualize
            min_overlap: Minimum number of documents that must share a phrase for an edge
            top_k: If provided, only consider top k phrases per document
        
        Returns:
            Plotly figure object
        """
        import plotly.graph_objects as go
        
        # Create graph structure (reusing existing logic)
        G = nx.Graph()
        
        # Track phrase occurrences and document associations
        phrase_docs = defaultdict(set)
        doc_colors = self._generate_distinct_colors(len(self.doc_ids))
        doc_color_map = dict(zip(self.doc_ids, doc_colors))
        
        # Collect phrases and their documents
        for doc_id in self.doc_ids:
            phrases = self.results[doc_id][n_gram][:top_k]
            for phrase, freq in phrases:
                phrase_docs[phrase].add(doc_id)
                
        # Add nodes (phrases)
        for phrase, docs in phrase_docs.items():
            max_freq_doc = max(docs, key=lambda doc: dict(self.results[doc][n_gram])[phrase])
            G.add_node(phrase, 
                      color=doc_color_map[max_freq_doc],
                      docs=list(docs),
                      size=len(docs) * 100)
            
        # Add edges between phrases that co-occur
        phrases = list(phrase_docs.keys())
        for i in range(len(phrases)):
            for j in range(i + 1, len(phrases)):
                common_docs = phrase_docs[phrases[i]] & phrase_docs[phrases[j]]
                if len(common_docs) >= min_overlap:
                    G.add_edge(phrases[i], phrases[j], 
                             weight=len(common_docs),
                             width=len(common_docs) * 0.5)
        
        # Create layout
        pos = nx.spring_layout(G, k=1/np.sqrt(len(G.nodes())), iterations=50)
        
        # Prepare edge traces
        edge_x = []
        edge_y = []
        for edge in G.edges():
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])
    
        edge_trace = go.Scatter(
            x=edge_x, y=edge_y,
            line=dict(width=0.5, color='#888'),
            hoverinfo='none',
            mode='lines')
    
        # Prepare node traces
        node_x = []
        node_y = []
        node_colors = []
        node_sizes = []
        node_text = []
        hover_text = []
        
        for node in G.nodes():
            x, y = pos[node]
            node_x.append(x)
            node_y.append(y)
            node_colors.append(G.nodes[node]['color'])
            node_sizes.append(G.nodes[node]['size'])
            
            # Just display the node text without labels
            node_text.append(node)
            # Create hover text with just the document IDs
            hover_text.append(', '.join(G.nodes[node]['docs']))
    
        node_trace = go.Scatter(
            x=node_x, y=node_y,
            mode='markers+text',
            hoverinfo='text',
            text=node_text,
            hovertext=hover_text,
            textposition="top center",
            textfont=dict(size=8),
            marker=dict(
                showscale=False,
                color=node_colors,
                size=[s/10 for s in node_sizes],
                line_width=2))
    
        # Create figure
        fig = go.Figure(data=[edge_trace, node_trace],
                       layout=go.Layout(
                           showlegend=False,
                           hovermode='closest',
                           margin=dict(b=20,l=5,r=5,t=20),
                           xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                           yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                       )
        
        # Add legend for documents
        for doc_id, color in doc_color_map.items():
            fig.add_trace(go.Scatter(
                x=[None], y=[None],
                mode='markers',
                name=doc_id,
                marker=dict(size=10, color=color),
                showlegend=True
            ))
        
        return fig
    
    def analyze_network_metrics(self, G: nx.Graph) -> pd.DataFrame:
        """
        Compute various network metrics for the phrase network.
        
        Args:
            G: NetworkX graph from create_phrase_network
            
        Returns:
            DataFrame with network metrics
        """
        metrics = {
            'degree_centrality': nx.degree_centrality(G),
            'betweenness_centrality': nx.betweenness_centrality(G),
            'eigenvector_centrality': nx.eigenvector_centrality(G, max_iter=1000),
            'clustering_coefficient': nx.clustering(G)
        }
        
        # Convert to DataFrame
        df = pd.DataFrame(metrics)
        
        # Add document information
        df['documents'] = [', '.join(G.nodes[node]['docs']) for node in df.index]
        df['num_documents'] = [len(G.nodes[node]['docs']) for node in df.index]
        
        return df.sort_values('degree_centrality', ascending=False)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px

def compute_similarities(phrases_by_text):
    """
    Compute similarity measures between documents based on their phrases.
    """
    similarity_measures = calculate_similarity_measures(phrases_by_text)
    return similarity_measures

def cluster_documents(similarity_measure, n_clusters=2, clustering_method='kmeans', doc_ids=None, doc_pattern=None):
    """
    Cluster documents based on a similarity measure.
    """
    # Convert the similarity measure to a DataFrame
    sim_df = pd.DataFrame(similarity_measure, 
                          index=list(phrases_by_text.keys()), 
                          columns=list(phrases_by_text.keys()))

    # Filter documents based on doc_ids or doc_pattern
    filtered_sim_df = filter_documents(sim_df, doc_ids, doc_pattern)

    if filtered_sim_df.empty:
        print("No documents found after filtering.")
        print("Available documents:", list(sim_df.index))
        return None, None
    else:
        # Perform clustering using the filtered similarity DataFrame
        if clustering_method == 'kmeans':
            clusterer = KMeans(n_clusters=n_clusters, random_state=42)
        else:
            raise ValueError(f"Unsupported clustering method: {clustering_method}")
        
        clusters = clusterer.fit_predict(filtered_sim_df)

        return filtered_sim_df, clusters

def visualize_clusters(similarity_measure, clusters, doc_ids=None, doc_pattern=None):
    """
    Visualize document clusters based on a similarity measure using Plotly.
    """
    # Convert the similarity measure to a DataFrame
    sim_df = pd.DataFrame(similarity_measure, 
                          index=list(phrases_by_text.keys()), 
                          columns=list(phrases_by_text.keys()))

    # Filter documents based on doc_ids or doc_pattern
    filtered_sim_df = filter_documents(sim_df, doc_ids, doc_pattern)

    # Create a DataFrame for plotting
    plot_data = pd.DataFrame({
        'Document': filtered_sim_df.index,
        'Cluster': clusters
    })

    # Create an interactive plot using Plotly
    fig = px.scatter(plot_data, x=plot_data.index, y=[0] * len(plot_data), color='Cluster',
                     labels={'x': 'Document', 'color': 'Cluster'},
                     title='Document Clusters',
                     hover_data={'Document': plot_data['Document']})
    
    # Customize the plot layout
    fig.update_layout(yaxis=dict(showticklabels=False, zeroline=False),
                      xaxis=dict(tickangle=45),
                      hoverlabel=dict(bgcolor="white", font_size=16),
                      legend=dict(title='Clusters', orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
    
    fig.show()

def analyze_clusters(similarity_measure, clusters, doc_ids=None, doc_pattern=None):
    """
    Analyze document clusters based on a similarity measure.
    """
    # Convert the similarity measure to a DataFrame
    sim_df = pd.DataFrame(similarity_measure, 
                          index=list(phrases_by_text.keys()), 
                          columns=list(phrases_by_text.keys()))

    # Filter documents based on doc_ids or doc_pattern
    filtered_sim_df = filter_documents(sim_df, doc_ids, doc_pattern)

    n_clusters = len(np.unique(clusters))

    print("\nCluster Analysis:")
    print("----------------")
    print(f"Number of clusters: {n_clusters}")

    # Calculate silhouette score
    if n_clusters > 1:
        silhouette_avg = silhouette_score(filtered_sim_df, clusters)
        print(f"\nSilhouette Score: {silhouette_avg:.3f}")
    else:
        print("\nSilhouette Score: Not applicable (only one cluster)")

    # Analyze clusters
    for cluster_id in np.unique(clusters):
        cluster_docs = filtered_sim_df.index[clusters == cluster_id]
        print(f"\nCluster {cluster_id}:")
        print("Documents:", ', '.join(cluster_docs))
        print(f"Size: {len(cluster_docs)} documents")

        # Calculate average similarity within the cluster
        cluster_sim = filtered_sim_df.loc[cluster_docs, cluster_docs].values
        avg_sim = np.mean(cluster_sim)
        print(f"Average similarity: {avg_sim:.3f}")

In [ ]:
network_viz = PhraseVisualization(phrase_results)

In [ ]:
for min_overlap in [1,2,3,4,5,10,15]:
    for top_k in [5,10,15,20]:
        fig = network_viz.create_interactive_phrase_network(
            n_gram=4, 
            min_overlap=min_overlap, 
            top_k=top_k
        )
        fig.write_image('out/phrases/phrase_' + 'top_' + str(top_k) + '_min_' + str(min_overlap) + '.png')
        fig.write_html('out/phrases/phrase_' + 'top_' + str(top_k) + '_min_' + str(min_overlap) + '.html')
        #fig.show()

In [ ]:
# Find phrases for each document
phrases_by_text = find_phrases_by_text(corpus_gensim_words_sents)

# Compute similarity measures
similarity_measures = compute_similarities(phrases_by_text)

# Cluster documents based on cosine similarity
cosine_sim_matrix = similarity_measures["cosine_similarity"]
filtered_sim_df, clusters = cluster_documents(cosine_sim_matrix, 
                                              n_clusters=2, clustering_method='kmeans',
                                              doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib')

if clusters is not None:
    # Visualize clusters
    visualize_clusters(cosine_sim_matrix, clusters, doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib')

    # Analyze clusters
    analyze_clusters(cosine_sim_matrix, clusters, doc_pattern='Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib')

In [ ]:
formulas

### PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff  # Added this import for dendrogram

def create_document_vectors(df):
    """
    Create feature vectors for each document using the pre-calculated relative frequencies.
    """
    # Get feature columns
    feature_columns = ['pos', 'Case', 'Gender', 'Number', 'Degree', 'Mood', 
                      'Person', 'Tense', 'VerbForm', 'Voice', 'Aspect', 
                      'NumForm', 'Abbr']
    
    # Create pivot table for each feature
    document_features = []
    
    for feature in feature_columns:
        # Create pivot table using count_rel (relative frequency)
        feature_pivot = pd.pivot_table(
            df,
            values='count_rel',
            index='doc_id',
            columns=feature,
            aggfunc='sum',
            fill_value=0
        )
        
        # Rename columns to include feature name
        feature_pivot.columns = [f"{feature}_{col}" if pd.notna(col) else f"{feature}_NA" 
                               for col in feature_pivot.columns]
        
        document_features.append(feature_pivot)
    
    # Concatenate all feature pivots
    doc_vectors_df = pd.concat(document_features, axis=1)
    
    # Reset index to make doc_id a column
    doc_vectors_df = doc_vectors_df.reset_index()
    
    return doc_vectors_df

def cluster_and_visualize_documents(doc_vectors_df, n_clusters=5, interactive=False):
    """
    Perform clustering analysis and create visualizations for documents.
    
    Parameters:
    -----------
    doc_vectors_df : pandas.DataFrame
        Document vectors created by create_document_vectors
    n_clusters : int
        Number of clusters for K-means
    interactive : bool
        If True, creates interactive Plotly visualizations instead of static matplotlib plots
    """
    # Separate document IDs and features
    doc_ids = doc_vectors_df['doc_id']
    features = doc_vectors_df.drop('doc_id', axis=1)
    
    # Scale the features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # Perform PCA for dimensionality reduction
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(features_scaled)
    
    # Perform K-means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(features_scaled)
    
    if interactive:
        # Create the main figure
        fig = go.Figure()
        
        # 1. Interactive PCA scatter plot
        scatter_data = pd.DataFrame({
            'PC1': pca_result[:, 0],
            'PC2': pca_result[:, 1],
            'Document': doc_ids,
            'Cluster': cluster_labels
        })
        
        fig.add_trace(go.Scatter(
            x=scatter_data['PC1'],
            y=scatter_data['PC2'],
            mode='markers+text',
            text=scatter_data['Document'],
            hovertemplate="<b>Document:</b> %{text}<br>" +
                         "<b>PC1:</b> %{x:.3f}<br>" +
                         "<b>PC2:</b> %{y:.3f}<br>" +
                         "<b>Cluster:</b> %{marker.color}<extra></extra>",
            marker=dict(
                color=cluster_labels,
                colorscale='Viridis',
                showscale=True,
                size=10
            ),
            textposition="top center"
        ))
        
        # Update layout
        fig.update_layout(
            title='Interactive PCA Visualization of Documents',
            xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
            yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
            height=800,
            width=1000,
            showlegend=False,
            # Add buttons for different views
            updatemenus=[{
                'buttons': [
                    {
                        'method': 'relayout',
                        'label': 'Reset View',
                        'args': ['autorange', True]
                    }
                ],
                'type': 'buttons',
                'direction': 'left',
                'pad': {'r': 10, 't': 10},
                'showactive': False,
                'x': 0.11,
                'y': 1.1
            }]
        )
        
        fig.show()
        
        # Create separate figures for additional visualizations
        
        # 2. Interactive dendrogram
        linkage_matrix = linkage(features_scaled, 'ward')
        dendro_fig = ff.create_dendrogram(
            features_scaled,
            labels=doc_ids.values,
            orientation='left',
            linkagefun=lambda x: linkage(x, 'ward')
        )
        dendro_fig.update_layout(
            title='Hierarchical Clustering Dendrogram',
            width=800,
            height=600
        )
        dendro_fig.show()
        
        # 3. Feature importance heatmap
        feature_importance = pd.DataFrame(
            pca.components_[:2].T,
            columns=['PC1', 'PC2'],
            index=features.columns
        )
        top_features = feature_importance.abs().sum(axis=1).sort_values(ascending=False).head(20)
        
        heatmap_fig = go.Figure(data=go.Heatmap(
            z=feature_importance.loc[top_features.index].values,
            x=['PC1', 'PC2'],
            y=top_features.index,
            colorscale='RdBu',
            zmid=0
        ))
        heatmap_fig.update_layout(
            title='Top Feature Contributions to Principal Components',
            width=800,
            height=600
        )
        heatmap_fig.show()
        
        # 4. Cluster size bar plot
        cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
        bar_fig = go.Figure(data=go.Bar(
            x=cluster_sizes.index,
            y=cluster_sizes.values,
            name='Documents per Cluster'
        ))
        bar_fig.update_layout(
            title='Distribution of Documents Across Clusters',
            xaxis_title='Cluster',
            yaxis_title='Number of Documents',
            width=800,
            height=400
        )
        bar_fig.show()
        
    else:
        # Original matplotlib visualization (code remains the same)
        fig = plt.figure(figsize=(20, 15))
        
        # 1. PCA scatter plot with clusters
        plt.subplot(2, 2, 1)
        scatter = plt.scatter(pca_result[:, 0], pca_result[:, 1], 
                            c=cluster_labels, cmap='viridis',
                            alpha=0.6)
        
        for i, doc_id in enumerate(doc_ids):
            plt.annotate(doc_id, (pca_result[i, 0], pca_result[i, 1]),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=8, alpha=0.7)
        
        plt.title('Document Clustering Visualization (PCA)')
        plt.xlabel(f'First Principal Component (variance explained: {pca.explained_variance_ratio_[0]:.2%})')
        plt.ylabel(f'Second Principal Component (variance explained: {pca.explained_variance_ratio_[1]:.2%})')
        plt.colorbar(scatter, label='Cluster')
        
        # 2. Hierarchical clustering dendrogram
        plt.subplot(2, 2, 2)
        linkage_matrix = linkage(features_scaled, 'ward')
        dendrogram(linkage_matrix, labels=doc_ids.values, leaf_rotation=90)
        plt.title('Hierarchical Clustering of Documents')
        plt.xlabel('Document ID')
        plt.ylabel('Distance')
        
        # 3. Feature importance
        plt.subplot(2, 2, 3)
        feature_importance = pd.DataFrame(
            pca.components_[:2].T,
            columns=['PC1', 'PC2'],
            index=features.columns
        )
        top_features = feature_importance.abs().sum(axis=1).sort_values(ascending=False).head(20)
        
        sns.heatmap(feature_importance.loc[top_features.index],
                    cmap='coolwarm', center=0,
                    xticklabels=True, yticklabels=True)
        plt.title('Top 20 Feature Contributions to Principal Components')
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        
        # 4. Cluster sizes
        plt.subplot(2, 2, 4)
        cluster_info = pd.DataFrame({
            'Cluster': cluster_labels,
            'Document': doc_ids
        }).groupby('Cluster').count()
        
        cluster_info.plot(kind='bar')
        plt.title('Documents per Cluster')
        plt.xlabel('Cluster')
        plt.ylabel('Number of Documents')
        
        plt.tight_layout()
        plt.show()
    
    # Convert features back to DataFrame with proper index
    features_df = pd.DataFrame(features, index=doc_ids.index, columns=features.columns)
    
    return doc_ids, cluster_labels, pca_result, features_df


def analyze_document_clusters(doc_ids, cluster_labels, features, original_df):
    """
    Analyze the characteristics of each document cluster.
    """
    cluster_analysis = []
    
    # Create DataFrame with document IDs and cluster labels
    cluster_df = pd.DataFrame({
        'doc_id': doc_ids,
        'cluster': cluster_labels
    })
    
    for cluster_id in range(len(set(cluster_labels))):
        # Get documents in this cluster
        cluster_docs = cluster_df[cluster_df['cluster'] == cluster_id]['doc_id'].tolist()
        
        # Get feature values for these documents
        cluster_mask = cluster_df['cluster'] == cluster_id
        cluster_features = features.loc[cluster_mask]
        
        # Calculate mean feature values for this cluster
        mean_features = cluster_features.mean()
        
        # Get top distinctive features (highest mean values)
        top_features = mean_features.nlargest(10)
        
        # Get most frequent tokens in these documents
        cluster_tokens = original_df[original_df['doc_id'].isin(cluster_docs)].copy()
        top_tokens = cluster_tokens.nlargest(10, 'count_rel')
        
        cluster_analysis.append({
            'cluster_id': cluster_id,
            'size': len(cluster_docs),
            'documents': cluster_docs,
            'distinctive_features': {
                feature: f"{value:.3f}" 
                for feature, value in top_features.items()
            },
            'top_tokens': top_tokens[['pos', 'Case', 'Gender', 'Number', 'count_rel']]\
                .to_dict('records')
        })
    
    return cluster_analysis

# Example usage:
"""
# Create document vectors using pre-calculated frequencies
doc_vectors_df = create_document_vectors(corpus_df_freq)

# Perform clustering
doc_ids, cluster_labels, pca_result, features = cluster_and_visualize_documents(doc_vectors_df, n_clusters=5)

# Analyze clusters
cluster_analysis = analyze_document_clusters(doc_ids, cluster_labels, features, corpus_df_freq)

# Print analysis
for cluster in cluster_analysis:
    print(f"\nCluster {cluster['cluster_id']} ({cluster['size']} documents):")
    print("\nDocuments:", ', '.join(cluster['documents']))
    print("\nDistinctive features:")
    for feature, value in cluster['distinctive_features'].items():
        print(f"{feature}: {value}")
    print("\nTop tokens:")
    for token in cluster['top_tokens'][:5]:  # Show top 5 tokens
        print(f"POS: {token['pos']}, Case: {token['Case']}, "
              f"Gender: {token['Gender']}, Number: {token['Number']}, "
              f"Frequency: {token['count_rel']:.4f}")
"""

In [ ]:
doc_vectors_df = create_document_vectors(corpus_df_freq)
doc_ids, cluster_labels, pca_result, features = cluster_and_visualize_documents(
    doc_vectors_df, 
    n_clusters=5, 
    interactive=True  # This will create interactive Plotly visualizations
)
#cluster_analysis = analyze_document_clusters(doc_ids, cluster_labels, features, corpus_df_freq)

In [ ]:
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Load your data
# corpus_df_freq = pd.read_pickle("corpora/spacy_corpus_all_df_freq.pkl")  # Uncomment to load your data
Skip to main panel
data.ipynb
Data
emerging_words = {}
for doc in analyzer.doc_ids:
    emerging_words[doc] = {}
    # Token-based binning
    for POS in ['NOUN', 'VERB', 'ADJ']:
        
        emerged = analyzer.detect_emerged_words(
            doc_id=doc,
            bin_type='percent',
            bin_size=1000,
            min_freq=5,
            min_bins=5,
            pos=POS
        )
        emerging_words[doc][POS] = emerged

2025-04-07 11:25:42,080 - INFO - Detecting emerged words in PomnLw2_TEI_final.conllu
2025-04-07 11:25:42,081 - INFO - Analyzing document: PomnLw2_TEI_final.conllu
2025-04-07 11:25:42,170 - INFO - Created 10 bins with approximately 1595 tokens each
2025-04-07 11:25:42,583 - INFO - Found 100 emerged words
2025-04-07 11:25:42,584 - INFO - Detecting emerged words in PomnLw2_TEI_final.conllu
2025-04-07 11:25:42,585 - INFO - Analyzing document: PomnLw2_TEI_final.conllu
2025-04-07 11:25:42,614 - INFO - Created 10 bins with approximately 421 tokens each
2025-04-07 11:25:42,655 - INFO - Found 27 emerged words
2025-04-07 11:25:42,656 - INFO - Detecting emerged words in PomnLw2_TEI_final.conllu
2025-04-07 11:25:42,656 - INFO - Analyzing document: PomnLw2_TEI_final.conllu
2025-04-07 11:25:42,672 - INFO - Created 10 bins with approximately 293 tokens each
2025-04-07 11:25:42,688 - INFO - Found 16 emerged words
2025-04-07 11:25:42,689 - INFO - Detecting emerged words in AGZ4_TEI_final.conllu
2025-04-07 11:25:42,690 - INFO - Analyzing document: AGZ4_TEI_final.conllu
2025-04-07 11:25:42,710 - INFO - Created 10 bins with approximately 1693 tokens each
2025-04-07 11:25:43,145 - INFO - Found 84 emerged words
2025-04-07 11:25:43,146 - INFO - Detecting emerged words in AGZ4_TEI_final.conllu
2025-04-07 11:25:43,146 - INFO - Analyzing document: AGZ4_TEI_final.conllu
2025-04-07 11:25:44,026 - INFO - Created 10 bins with approximately 870 tokens each
2025-04-07 11:25:44,132 - INFO - Found 31 emerged words
2025-04-07 11:25:44,133 - INFO - Detecting emerged words in AGZ4_TEI_final.conllu
2025-04-07 11:25:44,134 - INFO - Analyzing document: AGZ4_TEI_final.conllu
2025-04-07 11:25:44,152 - INFO - Created 10 bins with approximately 672 tokens each
# Step 1: Select relevant numeric features
# Here, we'll assume that you want to cluster based on 'count' and 'count_rel'
numeric_features = corpus_df_freq[['count_rel']]

In [ ]:
# Step 2: Handle missing values (optional)
# You can drop rows with NaN values or fill them
numeric_features = numeric_features.dropna()  # Or use .fillna(0)

In [ ]:
# Step 3: Standardize the data
scaler = StandardScaler()
scaled_features = scaler.fit_transform(numeric_features)

In [ ]:
# Step 4: Apply t-SNE
tsne = TSNE(n_components=1, random_state=42)
tsne_results = tsne.fit_transform(scaled_features)

In [ ]:
# Step 5: Create a DataFrame for t-SNE results
tsne_df = pd.DataFrame(tsne_results, columns=['x'])
tsne_df['doc_id'] = corpus_df_freq['doc_id']  # Include doc_id for reference

In [ ]:
# Step 6: Visualize the t-SNE results
plt.figure(figsize=(12, 8))
sns.scatterplot(data=tsne_df, x='x', y='x', hue='doc_id', palette='viridis', s=100)
plt.title('t-SNE Visualization of Corpus Frequencies')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.legend(title='Document ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import csr_matrix, hstack
encoder = OneHotEncoder(sparse_output=True)  # Set sparse=True to create a sparse matrix directly

sparse_features = encoder.fit_transform(corpus_df_freq.drop(columns='doc_id'))
count_sparse = csr_matrix(corpus_df_freq['count'].values).reshape(-1, 1)
sparse_matrix = hstack([sparse_features, count_sparse])
corpus_df_freq_enc = pd.DataFrame.sparse.from_spmatrix(sparse_matrix)

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix

# Assuming you have your sparse_matrix from previous steps

# Step 1: Clustering
n_clusters = 5  # Define the number of clusters
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(sparse_matrix)  # Fit the KMeans model

# Step 2: Create DataFrame for cluster labels
count_df = pd.DataFrame({
    'doc_id': corpus_df['doc_id'].unique(),  # Get unique doc_id values
    'cluster': cluster_labels  # Add cluster labels
})

In [ ]:
# Step 3: Reduce dimensionality with PCA
pca = PCA(n_components=50, init='random')  # Use init='random'
reduced_matrix = pca.fit_transform(sparse_matrix)

# Step 4: Run t-SNE on the reduced matrix
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(reduced_matrix)

# Step 5: Create a DataFrame for t-SNE results
tsne_df = pd.DataFrame(tsne_results, columns=['x', 'y'])
tsne_df['doc_id'] = count_df['doc_id']  # Include the doc_id
tsne_df['cluster'] = count_df['cluster']  # Include the cluster labels

In [ ]:
# Step 6: Visualize the results
plt.figure(figsize=(12, 8))
sns.scatterplot(data=tsne_df, x='x', y='y', hue='cluster', style='cluster', palette='viridis', s=100)
plt.title('t-SNE Visualization of Clusters')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.legend(title='Cluster')
plt.grid(True)
plt.show()

In [ ]:
corpus_df_freq_enc

In [ ]:
corpus_df_freq_sparse = encoder.fit_transform(corpus_df_freq)
# Optional: Create a DataFrame to see the one-hot encoded structure
corpus_df_freq_sparse_enc = pd.DataFrame.sparse.from_spmatrix(corpus_df_freq_sparse,
                                                columns=encoder.get_feature_names_out(corpus_df_features))

# Show the sparse matrix in a more readable format
print(corpus_df_freq_sparse_enc)

In [ ]:
from sklearn.preprocessing import LabelEncoder
    doc_encoder = LabelEncoder()
    feature_encoder = LabelEncoder()
    corpus_df_freq['doc_idx'] = doc_encoder.fit_transform(corpus_df_freq['doc_id'])

In [ ]:
import pandas as pd
corpus_df = pd.read_pickle("corpora/spacy_corpus_all_df.pkl")
#corpus_df_freq = pd.read_pickle("corpora/spacy_corpus_all_df_freq.pkl")

In [ ]:
corpus_df_freq

In [ ]:
corpus_df_features = ['pos', 'doc_id',
               'Case', 'Gender', 'Number', 'Degree', 'Mood', 'Person', 'Tense',
               'VerbForm', 'Voice', 'Aspect', 'NumForm', 'Abbr']
corpus_df_freq = corpus_df[corpus_df_features].groupby("doc_id", observed=True).value_counts()

In [ ]:
corpus_df_freq[corpus_df_freq > 0]

In [ ]:
del corpus_df_freq

In [ ]:
# Use a COO matrix to build the sparse matrix incrementally
row_indices = []
col_indices = []
data_values = []

# Use the feature combinations directly without creating a new column
for _, row in corpus_df_freq.iterrows():
    if row['count_rel'] > 0:  # Only include rows with non-zero relative counts
        # Create a tuple of features to use as a unique identifier
        feature_comb = tuple(row[corpus_df_features[1:]])
        feature_idx = hash(feature_comb)  # Use hash as an index
        
        # Append the current row's data to the COO structure
        row_indices.append(row['doc_idx'])
        col_indices.append(feature_idx)
        data_values.append(row['count_rel'])

In [ ]:
corpus_df_freq[corpus_df_freq["count"] > 0]

In [ ]:
# Create a COO matrix from the indices and data
sparse_matrix = sp.coo_matrix((data_values, (row_indices, col_indices)))

# Convert to CSR format for efficient arithmetic and slicing
sparse_matrix = sparse_matrix.tocsr()

In [ ]:
# Step 5: Create sparse matrix
spacy_corpus_all_df_freq_sparse = sp.csr_matrix((data, (rows, cols)))

# Optionally save the sparse matrix
sp.save_npz("corpora/spacy_corpus_all_df_freq_sparse.npz", spacy_corpus_all_df_freq_sparse)

# ?we need to pivot the df
if mode == "build":
    corpus_df_freq_pivot = corpus_df_freq.pivot(
        #observed=True,
        index='doc_id',
        columns=['pos', 'Case', 'Gender', 'Number', 'Degree', 'Mood', 'Person', 'Tense', 'VerbForm', 'Voice', 'Aspect', 'NumForm', 'Abbr'],
        values='count_rel',
        #fill_value=0  # fill missing with 0
    )
    corpus_df_freq_pivot.to_pickle("corpora/spacy_corpus_all_df_freq_pivot.pkl")
elif mode == "load":
    corpus_df_freq_pivot = pd.read_pickle("corpora/spacy_corpus_all_df_freq_pivot.pkl")


### T-SNE

In [ ]:
corpus_df_freq['feature_comb'] = corpus_df_freq[['pos', 'Case', 'Gender', 'Number', 'Degree', 
                                                      'Mood', 'Person', 'Tense', 'VerbForm', 
                                                      'Voice', 'Aspect', 'NumForm', 'Abbr']].astype(str).agg('_'.join, axis=1)

In [ ]:
# make df sparse (to avoid memory crashes)
import scipy.sparse as sp
corpus_df_freq_sparse = sp.csr_matrix(corpus_df_freq)
# memory crashes
#corpus_df_freq['feature_comb'] = corpus_df_freq[['pos', 'Case', 'Gender', 'Number', 'Degree', 
#                                                 'Mood', 'Person', 'Tense', 'VerbForm', 
#                                                 'Voice', 'Aspect', 'NumForm', 'Abbr']].astype(str).agg('_'.join, axis=1)
# incrementally building sparse matrix
# Initialize empty lists for sparse matrix inputs
data, rows, cols = [], [], []

# hash feature combinations
def get_feature_index(row):
    return hash((row['pos'], row['Case'], row['Gender'], row['Number'], row['Degree'], 
                 row['Mood'], row['Person'], row['Tense'], row['VerbForm'], 
                 row['Voice'], row['Aspect'], row['NumForm'], row['Abbr']))
    
# Process in smaller chunks by iterating over doc_ids
for doc_id, group in corpus_df_freq.groupby('doc_id'):
    # Encode doc_id and feature combination indices
    doc_idx = hash(doc_id)
    group['feature_idx'] = group.apply(get_feature_index, axis=1)
    
    # Append data for sparse matrix
    data.extend(group['count_rel'].values)
    rows.extend([doc_idx] * len(group))
    cols.extend(group['feature_idx'].values)

# Construct sparse matrix
corpus_df_freq_sm = sp.csr_matrix((data, (rows, cols)))

In [ ]:
corpus_df_freq_sm.head()

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
tsne = TSNE(n_components=2, random_state=42)
tsne_result = tsne.fit_transform(corpus_df_freq_pivot)

In [ ]:
# one-hot encoding (PCA)
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=True)
encoded_features = encoder.fit_transform(corpus_df_freq[[c for c in corpus_df_freq.columns if c not in ["doc_id"]]]) #lengthy

In [ ]:
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out())
print(encoded_df.shape)

In [ ]:
print(encoded_df.shape, corpus_df[['doc_id']].shape)

In [ ]:
# Concatenate with the original DataFrame (excluding the original categorical columns)
corpus_df_encoded = pd.concat([corpus_df[['doc_id']], encoded_df], axis=1)

In [ ]:
pca = PCA(n_components=2)
pca_result = pca.fit_transform(df_encoded.drop(columns=['text_id']))

# Create a DataFrame with PCA results
pca_df = pd.DataFrame(pca_result, columns=['PCA1', 'PCA2'])

# Visualize the results
plt.scatter(pca_df['PCA1'], pca_df['PCA2'])
plt.title('PCA of Categorical Variables')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.show()



# visualize with t-SNE
from sklearn.manifold import TSNE

# t-SNE
tsne = TSNE(n_components=2, random_state=42)
tsne_result = tsne.fit_transform(df_encoded.drop(columns=['text_id']))

# Create a DataFrame with t-SNE results
tsne_df = pd.DataFrame(tsne_result, columns=['TSNE1', 'TSNE2'])

# Visualize the results
plt.scatter(tsne_df['TSNE1'], tsne_df['TSNE2'])
plt.title('t-SNE of Categorical Variables')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.show()

In [ ]:
#from spacy import displacy
#displacy.serve(list(spacy_corpus.docs[0].sents)[0:10], style="ent")
#displacy.render(list(spacy_corpus_docbin.get_docs(conll_parser.nlp.vocab))[0], style="ent")

### CONLLU to Gensim

In [ ]:
#OLD
from conllu import parse
from gensim.corpora import Dictionary
from nltk.corpus import stopwords
import re
import os

class GensimCorpus:
    def __init__(self, corpus_dir, 
                 corpus_files, 
                 stopwords_list=None, min_freq=5, 
                 use_lemmas=False, mode='doc'):
        self.corpus_dir = corpus_dir
        self.corpus_files = corpus_files
        self.use_lemmas = use_lemmas
        self.min_freq = min_freq
        self.mode = mode
        
        # Set stopwords if provided, else use default
        self.stopwords_list = stopwords_list if stopwords_list else set(cltk.stops.lat)

        # Load the corpus
        self.corpus = self._load_corpus()
        # Create the dictionary
        self.dictionary = self._create_dictionary()

    def _load_corpus(self):
        corpus = {}
        for file in self.corpus_files:
            with open(os.path.join(self.corpus_dir, file), "r", encoding="utf-8") as f:
                txt = f.read()
                sents = parse(txt)
                corpus[file] = sents
        return corpus

    def _create_dictionary(self):
        # Create a Gensim dictionary from the corpus
        texts = list(self.get_texts())  # Convert generator to list
        dictionary = Dictionary(texts)
        dictionary.filter_extremes(no_below=self.min_freq)  # Filter low-frequency words
        return dictionary

    def get_texts(self):
        return self._conllu_iterator()

    def preprocess_text(self, text):
        text = text.lower()
        text = re.sub(r'[\d]', '', text)  # Remove numbers
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
        text = ' '.join(text.split())  # Remove extra whitespace
        return text

    def _conllu_iterator(self, mode='doc'):
        # Process CoNLL-U formatted corpus
        for fileid, sents in self.corpus.items():
            if mode == 'doc':
                document = []

            for sent in sents:
                tokens = [
                    token["lemma"] if self.use_lemmas and "lemma" in token else token["form"]
                    for token in sent if "form" in token
                ]
                tokens = [self.preprocess_text(token) for token in tokens]  # Preprocess tokens
                tokens = [token for token in tokens if token not in self.stopwords_list]  # Filter stopwords

                if tokens:  # Only yield non-empty documents
                    if mode == 'doc':
                        document.extend(tokens)
                    elif mode == 'sent':
                        yield tokens

            if mode == 'doc' and document:
                yield document

    def __iter__(self):
        # Yield BoW representation of each document
        for doc in self.get_texts():
            yield self.dictionary.doc2bow(doc)

    def __len__(self):
        # Return the number of documents in the corpus
        return sum(1 for _ in self.get_texts())  # Count non-empty documents

In [ ]:
# working but no sentence-based processing

from conllu import parse
from gensim.corpora import MmCorpus, Dictionary
from nltk.corpus import stopwords
import re
import os
import json
from typing import List, Dict, Optional
import logging

class GensimCorpus:
    def __init__(self, corpus_dir: str, 
                 corpus_files: List[str], 
                 stopwords_list: Optional[set] = None, 
                 min_freq: int = 5,
                 use_lemmas: bool = False, 
                 mode: str = 'doc'):
        # Initialize logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
        
        self.corpus_dir = corpus_dir
        self.corpus_files = corpus_files
        self.use_lemmas = use_lemmas
        self.min_freq = min_freq
        self.mode = mode
        
        # Track processed files and errors
        self.successful_files: List[str] = []
        self.failed_files: Dict[str, str] = {}  # filename: error message
        
        # Set stopwords if provided, else use default
        self.stopwords_list = stopwords_list if stopwords_list else set(cltk.stops.lat)
        
        # Load the corpus
        self.corpus = self._load_corpus()
        
        # Create the dictionary only if we have successfully processed files
        if self.successful_files:
            self.dictionary = self._create_dictionary()
        else:
            self.logger.warning("No files were successfully processed. Dictionary not created.")
            self.dictionary = None

    def _load_corpus(self) -> dict:
        """Load corpus with error handling and file tracking."""
        corpus = {}
        for file in self.corpus_files:
            try:
                with open(os.path.join(self.corpus_dir, file), "r", encoding="utf-8") as f:
                    txt = f.read()
                    try:
                        sents = parse(txt)
                        # Verify the parsed content
                        if not sents:
                            raise ValueError("No sentences found in the file")
                        
                        corpus[file] = sents
                        self.successful_files.append(file)
                        selfblo.logger.info(f"Successfully processed {file}")
                    except Exception as e:
                        error_msg = f"CoNLL-U parsing error: {str(e)}"
                        self.failed_files[file] = error_msg
                        self.logger.error(f"Failed to parse {file}: {error_msg}")
                    
            except FileNotFoundError:
                error_msg = f"File not found: {file}"
                self.failed_files[file] = error_msg
                self.logger.error(error_msg)
                
            except Exception as e:
                error_msg = f"Unexpected error: {str(e)}"
                self.failed_files[file] = error_msg
                self.logger.error(f"Error processing {file}: {error_msg}")
                
        if not corpus:
            self.logger.warning("No files were successfully processed")
            
        return corpus

    def _create_dictionary(self):
        """Create dictionary with additional error handling."""
        try:
            texts = list(self.get_texts())
            if not texts:
                raise ValueError("No texts available to create dictionary")
            
            dictionary = Dictionary(texts)
            dictionary.filter_extremes(no_below=self.min_freq)
            return dictionary
            
        except Exception as e:
            self.logger.error(f"Error creating dictionary: {str(e)}")
            return None

    def get_texts(self):
        return self._conllu_iterator()

    def preprocess_text(self, text):
        text = text.lower()
        text = re.sub(r'[\d]', '', text)  # Remove numbers
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
        text = ' '.join(text.split())  # Remove extra whitespace
        return text

    def _conllu_iterator(self, mode='doc'):
        for fileid, sents in self.corpus.items():
            if fileid not in self.successful_files:
                continue
                
            if mode == 'doc':
                document = []
            for sent in sents:
                tokens = [
                    token["lemma"] if self.use_lemmas and "lemma" in token else token["form"]
                    for token in sent if "form" in token
                ]
                tokens = [self.preprocess_text(token) for token in tokens]
                tokens = [token for token in tokens if token not in self.stopwords_list]
                if tokens:
                    if mode == 'doc':
                        document.extend(tokens)
                    elif mode == 'sent':
                        yield tokens
            if mode == 'doc' and document:
                yield document

    def __iter__(self):
        # Yield BoW representation of each document
        for doc in self.get_texts():
            yield self.dictionary.doc2bow(doc)

    def __len__(self):
        # Return the number of documents in the corpus
        return sum(1 for _ in self.get_texts())

    def get_processing_summary(self) -> Dict[str, any]:
        """Return a summary of processed files and any errors encountered."""
        return {
            "total_files": len(self.corpus_files),
            "successful_files": self.successful_files,
            "failed_files": self.failed_files,
            "success_rate": len(self.successful_files) / len(self.corpus_files) if self.corpus_files else 0
        }

    def get_successful_files(self) -> List[str]:
        """Return list of successfully processed files in order of processing."""
        return self.successful_files

    def get_failed_files(self) -> Dict[str, str]:
        """Return dictionary of failed files and their error messages."""
        return self.failed_files

    def serialize(self, output_prefix: str, directory: str = None):
        """
        Serialize both the corpus and the successful files list.
        
        Args:
            output_prefix: Path prefix for the output files (without extension)
            directory: Optional directory path where files should be saved. 
                      If provided, will create directory if it doesn't exist.
        """
        # Handle directory creation if specified
        if directory:
            os.makedirs(directory, exist_ok=True)
            output_path = os.path.join(directory, output_prefix)
        else:
            output_path = output_prefix
            
        # Serialize the corpus using MmCorpus
        MmCorpus.serialize(f"{output_path}.mm", self)
        
        # Serialize the dictionary
        self.dictionary.save(f"{output_path}.dict")
        
        # Save the metadata (successful files and other relevant info)
        metadata = {
            'successful_files': self.successful_files,
            'failed_files': self.failed_files,
            'use_lemmas': self.use_lemmas,
            'mode': self.mode,
            'min_freq': self.min_freq
        }
        
        with open(f"{output_path}.metadata.json", 'w', encoding='utf-8') as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, input_prefix: str, directory: str = None):
        """
        Load a serialized corpus, dictionary, and metadata.
        
        Args:
            input_prefix: Path prefix for the input files (without extension)
            directory: Optional directory path where files are stored
            
        Returns:
            GensimCorpus: A new instance with loaded data
        """
        # Handle directory if specified
        if directory:
            input_path = os.path.join(directory, input_prefix)
        else:
            input_path = input_prefix
            
        # Create an empty instance
        instance = cls.__new__(cls)
        
        # Load metadata
        with open(f"{input_path}.metadata.json", 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        
        # Set instance attributes from metadata
        instance.successful_files = metadata['successful_files']
        instance.failed_files = metadata['failed_files']
        instance.use_lemmas = metadata['use_lemmas']
        instance.mode = metadata['mode']
        instance.min_freq = metadata['min_freq']
        
        # Load dictionary
        instance.dictionary = Dictionary.load(f"{input_path}.dict")
        
        # Load corpus
        instance.corpus = MmCorpus(f"{input_path}.mm")
        
        # Initialize logger
        instance.logger = logging.getLogger(__name__)
        
        return instance

In [ ]:
from conllu import parse
from gensim.corpora import MmCorpus, Dictionary
from nltk.corpus import stopwords
import re
import os
import json
from typing import List, Dict, Optional, Tuple
import logging

class GensimCorpus:
    def __init__(self, corpus_dir: str, 
                 corpus_files: List[str], 
                 stopwords_list: Optional[set] = None, 
                 min_freq: int = 5,
                 use_lemmas: bool = False, 
                 mode: str = 'doc'):

        # Initialize logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
        
        self.corpus_dir = corpus_dir
        self.corpus_files = corpus_files
        self.use_lemmas = use_lemmas
        self.min_freq = min_freq
        self.mode = mode
        
        self.sentence_metadata: List[Tuple[str, int]] = []  # List of (fileid, sent_idx) tuples
        # Track processed files and errors
        self.successful_files: List[str] = []
        self.failed_files: Dict[str, str] = {}  # filename: error message
        
        # Set stopwords if provided, else use default
        self.stopwords_list = stopwords_list if stopwords_list else set(cltk.stops.lat)
        
        # Load the corpus
        self.corpus = self._load_corpus()
        
        # Create the dictionary only if we have successfully processed files
        if self.successful_files:
            self.dictionary = self._create_dictionary()
        else:
            self.logger.warning("No files were successfully processed. Dictionary not created.")
            self.dictionary = None

    def _load_corpus(self) -> dict:
        """Load corpus with error handling and file tracking."""
        corpus = {}
        for file in self.corpus_files:
            try:
                with open(os.path.join(self.corpus_dir, file), "r", encoding="utf-8") as f:
                    txt = f.read()
                    try:
                        sents = parse(txt)
                        # Verify the parsed content
                        if not sents:
                            raise ValueError("No sentences found in the file")
                        
                        corpus[file] = sents
                        self.successful_files.append(file)
                        self.logger.info(f"Successfully processed {file}")
                    except Exception as e:
                        error_msg = f"CoNLL-U parsing error: {str(e)}"
                        self.failed_files[file] = error_msg
                        self.logger.error(f"Failed to parse {file}: {error_msg}")
                    
            except FileNotFoundError:
                error_msg = f"File not found: {file}"
                self.failed_files[file] = error_msg
                self.logger.error(error_msg)
                
            except Exception as e:
                error_msg = f"Unexpected error: {str(e)}"
                self.failed_files[file] = error_msg
                self.logger.error(f"Error processing {file}: {error_msg}")
                
        if not corpus:
            self.logger.warning("No files were successfully processed")
            
        return corpus

    def _create_dictionary(self):
        """Create dictionary with additional error handling."""
        try:
            texts = list(self.get_texts())
            if not texts:
                raise ValueError("No texts available to create dictionary")
            
            dictionary = Dictionary(texts)
            dictionary.filter_extremes(no_below=self.min_freq)
            return dictionary
            
        except Exception as e:
            self.logger.error(f"Error creating dictionary: {str(e)}")
            return None

    def get_texts(self):
        return self._conllu_iterator()

    def preprocess_text(self, text):
        text = text.lower()
        text = re.sub(r'[\d]', '', text)  # Remove numbers
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
        text = ' '.join(text.split())  # Remove extra whitespace
        return text

    def _conllu_iterator(self):
        
        for fileid, sents in self.corpus.items():
            if fileid not in self.successful_files:
                continue
            if self.mode == 'doc':
                document = []
                for sent in sents:
                    tokens = self._process_sentence(sent)
                    if tokens:
                        document.extend(tokens)
                if document:
                    yield document
            
            elif self.mode == 'sent':
                for sent_idx, sent in enumerate(sents):
                    tokens = self._process_sentence(sent)
                    if tokens:
                        # Track the sentence metadata
                        self.sentence_metadata.append((fileid, sent_idx))
                        yield tokens

    def _process_sentence(self, sent):
        """Helper method to process a single sentence."""
        tokens = [
            token["lemma"] if self.use_lemmas and "lemma" in token else token["form"]
            for token in sent if "form" in token
        ]
        tokens = [self.preprocess_text(token) for token in tokens]
        tokens = [token for token in tokens if token not in self.stopwords_list]
        return tokens
    def get_sentence_info(self, idx: int) -> Tuple[str, int]:
        """Get the file and sentence index for a given corpus position."""
        if self.mode != 'sent':
            raise ValueError("Sentence info is only available in 'sent' mode")
        return self.sentence_metadata[idx]


    def __iter__(self):
        # Yield BoW representation of each document
        for doc in self.get_texts():
            yield self.dictionary.doc2bow(doc)

    def __len__(self):
        """Return the number of documents or sentences in the corpus."""
        if self.mode == 'doc':
            return len(self.successful_files)
        else:
            return len(self.sentence_metadata)

    def get_processing_summary(self) -> Dict[str, any]:
        """Return a summary of processed files and any errors encountered."""
        return {
            "total_files": len(self.corpus_files),
            "successful_files": self.successful_files,
            "failed_files": self.failed_files,
            "success_rate": len(self.successful_files) / len(self.corpus_files) if self.corpus_files else 0
        }

    def get_successful_files(self) -> List[str]:
        """Return list of successfully processed files in order of processing."""
        return self.successful_files

    def get_failed_files(self) -> Dict[str, str]:
        """Return dictionary of failed files and their error messages."""
        return self.failed_files

    def serialize(self, output_prefix: str, directory: str = None):
        """
        Serialize both the corpus and the successful files list.
        
        Args:
            output_prefix: Path prefix for the output files (without extension)
            directory: Optional directory path where files should be saved. 
                      If provided, will create directory if it doesn't exist.
        """
        # Handle directory creation if specified
        if directory:
            os.makedirs(directory, exist_ok=True)
            output_path = os.path.join(directory, output_prefix)
        else:
            output_path = output_prefix
            
        # Serialize the corpus using MmCorpus
        MmCorpus.serialize(f"{output_path}.mm", self)
        
        # Serialize the dictionary
        self.dictionary.save(f"{output_path}.dict")
        
        metadata = {
            'successful_files': self.successful_files,
            'failed_files': self.failed_files,
            'use_lemmas': self.use_lemmas,
            'mode': self.mode,
            'min_freq': self.min_freq,
            'sentence_metadata': self.sentence_metadata if self.mode == 'sent' else []
        }
        with open(f"{output_path}.metadata.json", 'w', encoding='utf-8') as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)
    
    def create_subcorpus(self, fileids: List[str]) -> 'GensimCorpus':
        """
        Creates a new corpus instance from selected files using existing corpus data.
        
        Args:
            fileids: List of file IDs to include in the subcorpus
        
        Returns:
            GensimCorpus: New corpus instance with only the selected files
        """
        # Create a new instance
        subcorpus = GensimCorpus.__new__(GensimCorpus)
        
        # Copy basic attributes
        subcorpus.corpus_dir = self.corpus_dir
        subcorpus.corpus_files = fileids
        subcorpus.use_lemmas = self.use_lemmas
        subcorpus.min_freq = self.min_freq
        subcorpus.mode = self.mode
        subcorpus.stopwords_list = self.stopwords_list
        
        # Set up logging
        subcorpus.logger = logging.getLogger(__name__)
        
        # Filter the corpus data
        subcorpus.corpus = {
            fileid: self.corpus[fileid] 
            for fileid in fileids 
            if fileid in self.corpus
        }
        
        # Filter successful and failed files
        subcorpus.successful_files = [
            f for f in self.successful_files 
            if f in fileids
        ]
        subcorpus.failed_files = {
            k: v for k, v in self.failed_files.items() 
            if k in fileids
        }
        
        # Filter sentence metadata if in sentence mode
        if self.mode == 'sent':
            subcorpus.sentence_metadata = [
                (fileid, sent_idx) 
                for fileid, sent_idx in self.sentence_metadata
                if fileid in fileids
            ]
        else:
            subcorpus.sentence_metadata = []
        
        # Reuse the existing dictionary
        subcorpus.dictionary = self.dictionary
        
        return subcorpus
    
    @classmethod
    def load(cls, input_prefix: str, directory: str = None):
        """
        Load a serialized corpus, dictionary, and metadata.
        
        Args:
            input_prefix: Path prefix for the input files (without extension)
            directory: Optional directory path where files are stored
            
        Returns:
            GensimCorpus: A new instance with loaded data
        """
        # Handle directory if specified
        if directory:
            input_path = os.path.join(directory, input_prefix)
        else:
            input_path = input_prefix
            
        # Create an empty instance
        instance = cls.__new__(cls)
        
        # Load metadata
        with open(f"{input_path}.metadata.json", 'r', encoding='utf-8') as f:
            metadata = json.load(f)

        
        # Set instance attributes from metadata
        instance.successful_files = metadata['successful_files']
        instance.failed_files = metadata['failed_files']
        instance.use_lemmas = metadata['use_lemmas']
        instance.mode = metadata['mode']
        instance.min_freq = metadata['min_freq']
        
        if 'sentence_metadata' in metadata:
            instance.sentence_metadata = metadata['sentence_metadata']
        else:
            instance.sentence_metadata = []
        
        # Load dictionary
        instance.dictionary = Dictionary.load(f"{input_path}.dict")
        
        # Load corpus
        instance.corpus = MmCorpus(f"{input_path}.mm")
        
        # Initialize logger
        instance.logger = logging.getLogger(__name__)
        
        return instance

In [ ]:
stopwords=["ab", "ac", "ad", "adhic", "aliqui", "aliquis", "an", "ante", "apud", "at", "atque", "aut", "autem", "cum", "cur", "de", "deinde", "dum", "ego", "enim", "ergo", "es", "est", "et", "etiam", "etsi", "ex", "fio", "haud", "hic", "iam", "idem", "igitur", "ille", "in", "infra", "inter", "interim", "ipse", "is", "ita", "magis", "modo", "mox", "nam", "ne", "nec", "necque", "neque", "nisi", "non", "nos", "o", "ob", "per", "possum", "post", "pro", "quae", "quam", "quare", "qui", "quia", "quicumque", "quidem", "quilibet", "quis", "quisnam", "quisquam", "quisque", "quisquis", "quo", "quoniam", "sed", "si", "sic", "sive", "sub", "sui", "sum", "super", "suus", "tam", "tamen", "trans", "tu", "tum", "ubi", "uel", "uero", "unus", "ut"]
mode = mode_selector.value
from gensim.corpora import MmCorpus
from gensim.corpora import Dictionary
if mode == 'build':
    corpus_gensim_words = GensimCorpus(corpus_dir=corpus_dir, corpus_files=corpus_files, stopwords_list=stopwords)
    corpus_gensim_words.serialize("gensim_words", directory="corpora")
        
    corpus_gensim_lemmas = GensimCorpus(corpus_dir=corpus_dir, corpus_files=corpus_files, stopwords_list=stopwords, use_lemmas=True)
    corpus_gensim_lemmas.serialize("gensim_lemmas", directory="corpora")    
    #corpus_gensim_words = GensimCorpus(corpus_dir, corpus_files, stopwords_list=stopwords)
    #corpus_gensim_lemmas = GensimCorpus(corpus_dir, corpus_files, stopwords_list=stopwords, use_lemmas=True)
    #MmCorpus.serialize('corpora/gensim_words.mm', corpus_gensim_words, corpus_gensim_words.dictionary)
    #corpus_gensim_words.dictionary.save('corpora/gensim_words.dict')
    #MmCorpus.serialize('corpora/gensim_lemmas.mm', corpus_gensim_lemmas, corpus_gensim_lemmas.dictionary)
    #corpus_gensim_lemmas.dictionary.save('corpora/gensim_lemmas.dict')
elif mode == 'load':
    corpus_gensim_words = GensimCorpus.load(input_prefix="gensim_words",directory="corpora")
    corpus_gensim_lemmas = GensimCorpus.load(input_prefix="gensim_lemmas",directory="corpora")
    #corpus_gensim_words = MmCorpus('corpora/gensim_words.mm')
    #corpus_gensim_words_dict = Dictionary.load('corpora/gensim_words.dict')
    #corpus_gensim_lemmas = MmCorpus('corpora/gensim_lemmas.mm')
    #corpus_gensim_lemmas_dict = Dictionary.load('corpora/gensim_lemmas.dict')

#### Sentence-based

In [ ]:
stopwords=["ab", "ac", "ad", "adhic", "aliqui", "aliquis", "an", "ante", "apud", "at", "atque", "aut", "autem", "cum", "cur", "de", "deinde", "dum", "ego", "enim", "ergo", "es", "est", "et", "etiam", "etsi", "ex", "fio", "haud", "hic", "iam", "idem", "igitur", "ille", "in", "infra", "inter", "interim", "ipse", "is", "ita", "magis", "modo", "mox", "nam", "ne", "nec", "necque", "neque", "nisi", "non", "nos", "o", "ob", "per", "possum", "post", "pro", "quae", "quam", "quare", "qui", "quia", "quicumque", "quidem", "quilibet", "quis", "quisnam", "quisquam", "quisque", "quisquis", "quo", "quoniam", "sed", "si", "sic", "sive", "sub", "sui", "sum", "super", "suus", "tam", "tamen", "trans", "tu", "tum", "ubi", "uel", "uero", "unus", "ut"]
mode = mode_selector.value
from gensim.corpora import MmCorpus
from gensim.corpora import Dictionary
if mode == 'build':
    corpus_gensim_sents_words = GensimCorpus(corpus_dir, corpus_files, stopwords_list=stopwords, mode='sent')
    corpus_gensim_sents_words.serialize("gensim_sents_words", directory="corpora")
    
    corpus_gensim_sents_lemmas = GensimCorpus(corpus_dir, corpus_files, stopwords_list=stopwords,
                                              mode='sent', use_lemmas=True)
    corpus_gensim_sents_lemmas.serialize("gensim_sents_lemmas", directory="corpora")
    
    #corpus_gensim_words = GensimCorpus(corpus_dir, corpus_files, stopwords_list=stopwords)
    #corpus_gensim_lemmas = GensimCorpus(corpus_dir, corpus_files, stopwords_list=stopwords, use_lemmas=True)
    #MmCorpus.serialize('corpora/gensim_words.mm', corpus_gensim_words, corpus_gensim_words.dictionary)
    #corpus_gensim_words.dictionary.save('corpora/gensim_words.dict')
    #MmCorpus.serialize('corpora/gensim_lemmas.mm', corpus_gensim_lemmas, corpus_gensim_lemmas.dictionary)
    #corpus_gensim_lemmas.dictionary.save('corpora/gensim_lemmas.dict')
elif mode == 'load':
    corpus_gensim_sents_words = GensimCorpus.load(input_prefix="gensim_sents_words",directory="corpora")
    corpus_gensim_sents_lemmas = GensimCorpus.load(input_prefix="gensim_sents_lemmas",directory="corpora")
    #corpus_gensim_words = MmCorpus('corpora/gensim_words.mm')
    #corpus_gensim_words_dict = Dictionary.load('corpora/gensim_words.dict')
    #corpus_gensim_lemmas = MmCorpus('corpora/gensim_lemmas.mm')
    #corpus_gensim_lemmas_dict = Dictionary.load('corpora/gensim_lemmas.dict')

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from gensim import models, matutils
import plotly.figure_factory as ff

def compute_similarity_matrices(corpus, dictionary, models_to_compute="all"):
    """
    Compute similarity matrices using different models
    
    Parameters:
    - corpus: gensim corpus object
    - dictionary: gensim dictionary object
    - models_to_compute: str or list, which models to compute similarities for
                        can be "all" or any of ["tfidf", "lsi", "lda"]
    
    Returns:
    - Dictionary containing similarity matrices for requested models
    """
    corpus_list = list(corpus)
    n_docs = len(corpus_list)
    
    if models_to_compute == "all":
        models_to_compute = ["tfidf", "lsi", "lda"]
    elif isinstance(models_to_compute, str):
        models_to_compute = [models_to_compute]
        
    similarity_matrices = {}
    
    # Create base models
    if "tfidf" in models_to_compute or "lsi" in models_to_compute:
        tfidf_model = models.TfidfModel(corpus_list)
        tfidf_corpus = list(tfidf_model[corpus_list])
        
        if "tfidf" in models_to_compute:
            # Compute TF-IDF similarities
            tfidf_matrix = np.zeros((n_docs, n_docs))
            for i in range(n_docs):
                for j in range(n_docs):
                    tfidf_matrix[i, j] = matutils.cossim(tfidf_corpus[i], tfidf_corpus[j])
            similarity_matrices['tfidf'] = tfidf_matrix
    
        if "lsi" in models_to_compute:
            # Compute LSI similarities
            lsi_model = models.LsiModel(tfidf_corpus, id2word=dictionary, 
                                      num_topics=min(100, n_docs))
            lsi_corpus = list(lsi_model[tfidf_corpus])
            lsi_matrix = np.zeros((n_docs, n_docs))
            for i in range(n_docs):
                for j in range(n_docs):
                    lsi_matrix[i, j] = matutils.cossim(lsi_corpus[i], lsi_corpus[j])
            similarity_matrices['lsi'] = lsi_matrix
    
    if "lda" in models_to_compute:
        # Compute LDA similarities
        lda_model = models.LdaModel(corpus_list, id2word=dictionary,
                                  num_topics=min(50, n_docs), passes=10)
        lda_corpus = list(lda_model[corpus_list])
        lda_matrix = np.zeros((n_docs, n_docs))
        for i in range(n_docs):
            for j in range(n_docs):
                lda_matrix[i, j] = matutils.cossim(lda_corpus[i], lda_corpus[j])
        similarity_matrices['lda'] = lda_matrix
    
    return similarity_matrices

def create_similarity_visualizations(similarity_matrix, doc_labels=None, title_prefix="Document"):
    """
    Create interactive visualizations for a similarity matrix
    
    Parameters:
    - similarity_matrix: numpy array of shape (n_docs, n_docs)
    - doc_labels: list of document labels
    - title_prefix: prefix for plot titles
    
    Returns:
    - Dictionary containing plotly figures for heatmap, dendrogram, and scatter plot
    """
    n_docs = similarity_matrix.shape[0]
    if doc_labels is None:
        doc_labels = [f"{title_prefix} {i+1}" for i in range(n_docs)]
    
    # 1. Heatmap
    heatmap = go.Figure(data=go.Heatmap(
        z=similarity_matrix,
        x=doc_labels,
        y=doc_labels,
        colorscale='Blues',
        zmin=0,
        zmax=1
    ))
    
    heatmap.update_layout(
        title=f'{title_prefix} Similarity Heatmap',
        xaxis_title='Documents',
        yaxis_title='Documents',
        width=800,
        height=800
    )
    
    # 2. Dendrogram
    # Convert similarities to distances and ensure proper format
    distances = 1 - similarity_matrix
    np.fill_diagonal(distances, 0)  # Ensure diagonal is zero
    
    # Create dendrogram directly from the distance matrix
    dendro = go.Figure()
    
    # Compute linkage matrix
    condensed_dist = squareform(distances)
    Z = hierarchy.linkage(condensed_dist, method='ward')
    
    # Get dendrogram data
    dendro_data = hierarchy.dendrogram(Z, labels=doc_labels, no_plot=True)
    
    # Create dendrogram using go.Scatter
    x_coords = []
    y_coords = []
    for i, d in enumerate(dendro_data['dcoord']):
        x_coords.extend([dendro_data['icoord'][i][j] for j in range(4)])
        x_coords.append(None)
        y_coords.extend([d[j] for j in range(4)])
        y_coords.append(None)
    
    dendro.add_trace(go.Scatter(
        x=x_coords,
        y=y_coords,
        mode='lines',
        line=dict(color='rgb(0, 116, 217)'),
        hoverinfo='none'
    ))
    
    # Update layout
    dendro.update_layout(
        title=f'{title_prefix} Similarity Dendrogram',
        showlegend=False,
        xaxis=dict(
            ticktext=dendro_data['ivl'],
            tickvals=[x for x in dendro_data['leaves_color_list']],
            tickmode="array",
            showticklabels=True
        ),
        width=800,
        height=600,
        yaxis=dict(
            title='Distance'
        )
    )
    
    # 3. PCA Scatter Plot
    pca = PCA(n_components=2)
    coords = pca.fit_transform(similarity_matrix)
    
    scatter = go.Figure(data=go.Scatter(
        x=coords[:, 0],
        y=coords[:, 1],
        mode='markers+text',
        text=doc_labels,
        textposition="top center",
        marker=dict(
            size=10,
            color=np.arange(len(coords)),
            colorscale='Blues',
            showscale=True
        )
    ))
    
    scatter.update_layout(
        title=f'{title_prefix} 2D Projection (PCA)',
        xaxis_title='First Principal Component',
        yaxis_title='Second Principal Component',
        width=800,
        height=600
    )
    
    return {
        'heatmap': heatmap,
        'dendrogram': dendro,
        'scatter': scatter
    }

In [ ]:
# takes too long to compute
from gensim import models, matutils
import numpy as np
from scipy import sparse
from tqdm import tqdm
from typing import Dict, List, Union, Optional
import logging

def compute_sent_similarity_matrices(corpus: 'GensimCorpus',
                                   models_to_compute: Union[str, List[str]] = "all",
                                   batch_size: int = 1000,
                                   min_similarity: float = 0.3,
                                   use_sparse: bool = True) -> Dict[str, Union[np.ndarray, sparse.csr_matrix]]:
    """
    Compute sentence similarity matrices using different models with batched processing
    
    Parameters:
    - corpus: GensimCorpus object in 'sent' mode
    - models_to_compute: str or list, which models to compute similarities for
                        can be "all" or any of ["tfidf", "lsi", "lda"]
    - batch_size: int, number of sentences to process at once
    - min_similarity: float, minimum similarity threshold (used with sparse matrices)
    - use_sparse: bool, whether to use sparse matrices (recommended for large corpora)
    
    Returns:
    - Dictionary containing similarity matrices for requested models
    """
    if corpus.mode != 'sent':
        raise ValueError("Corpus must be in 'sent' mode")
    
    logger = logging.getLogger(__name__)
    
    # Convert corpus to list and get dimensions
    logger.info("Converting corpus to list...")
    corpus_list = list(corpus)
    n_sents = len(corpus_list)
    
    if models_to_compute == "all":
        models_to_compute = ["tfidf", "lsi", "lda"]
    elif isinstance(models_to_compute, str):
        models_to_compute = [models_to_compute]
    
    similarity_matrices = {}
    
    def create_matrix():
        """Helper to create empty matrix based on settings"""
        if use_sparse:
            return [], [], []  # rows, cols, data for COO format
        return np.zeros((n_sents, n_sents))
    
    def process_batch_similarities(model_corpus, start_idx, batch):
        """Process similarities for a batch of sentences"""
        batch_vectors = list(batch)
        
        if use_sparse:
            rows, cols, data = [], [], []
            for i, sent_vector in enumerate(model_corpus):
                if i < start_idx + len(batch_vectors):
                    sims = [matutils.cossim(sent_vector, batch_vec) 
                           for batch_vec in batch_vectors]
                    
                    # Store significant similarities
                    for j, sim in enumerate(sims):
                        if sim >= min_similarity and i != (start_idx + j):
                            rows.append(i)
                            cols.append(start_idx + j)
                            data.append(sim)
                            
                            # Add symmetric pair if needed
                            if start_idx + j < i:
                                rows.append(start_idx + j)
                                cols.append(i)
                                data.append(sim)
            return rows, cols, data
        else:
            for i, sent_vector in enumerate(model_corpus):
                if i < start_idx + len(batch_vectors):
                    sims = [matutils.cossim(sent_vector, batch_vec) 
                           for batch_vec in batch_vectors]
                    matrix[i, start_idx:start_idx + len(sims)] = sims
                    matrix[start_idx:start_idx + len(sims), i] = sims
            return matrix
    
    # Create base models and compute similarities
    if "tfidf" in models_to_compute or "lsi" in models_to_compute:
        logger.info("Creating TF-IDF model...")
        tfidf_model = models.TfidfModel(corpus_list)
        tfidf_corpus = list(tfidf_model[corpus_list])
        
        if "tfidf" in models_to_compute:
            logger.info("Computing TF-IDF similarities...")
            matrix = create_matrix()
            
            for i in tqdm(range(0, n_sents, batch_size)):
                batch = tfidf_corpus[i:i + batch_size]
                if use_sparse:
                    rows, cols, data = process_batch_similarities(tfidf_corpus, i, batch)
                    matrix[0].extend(rows)
                    matrix[1].extend(cols)
                    matrix[2].extend(data)
                else:
                    matrix = process_batch_similarities(tfidf_corpus, i, batch)
            
            if use_sparse:
                similarity_matrices['tfidf'] = sparse.csr_matrix(
                    (matrix[2], (matrix[0], matrix[1])),
                    shape=(n_sents, n_sents)
                )
            else:
                similarity_matrices['tfidf'] = matrix
    
        if "lsi" in models_to_compute:
            logger.info("Creating LSI model...")
            lsi_model = models.LsiModel(tfidf_corpus, 
                                      id2word=corpus.dictionary,
                                      num_topics=min(100, n_sents))
            lsi_corpus = list(lsi_model[tfidf_corpus])
            
            logger.info("Computing LSI similarities...")
            matrix = create_matrix()
            
            for i in tqdm(range(0, n_sents, batch_size)):
                batch = lsi_corpus[i:i + batch_size]
                if use_sparse:
                    rows, cols, data = process_batch_similarities(lsi_corpus, i, batch)
                    matrix[0].extend(rows)
                    matrix[1].extend(cols)
                    matrix[2].extend(data)
                else:
                    matrix = process_batch_similarities(lsi_corpus, i, batch)
            
            if use_sparse:
                similarity_matrices['lsi'] = sparse.csr_matrix(
                    (matrix[2], (matrix[0], matrix[1])),
                    shape=(n_sents, n_sents)
                )
            else:
                similarity_matrices['lsi'] = matrix
    
    if "lda" in models_to_compute:
        logger.info("Creating LDA model...")
        lda_model = models.LdaModel(corpus_list, 
                                  id2word=corpus.dictionary,
                                  num_topics=min(50, n_sents), 
                                  passes=10)
        lda_corpus = list(lda_model[corpus_list])
        
        logger.info("Computing LDA similarities...")
        matrix = create_matrix()
        
        for i in tqdm(range(0, n_sents, batch_size)):
            batch = lda_corpus[i:i + batch_size]
            if use_sparse:
                rows, cols, data = process_batch_similarities(lda_corpus, i, batch)
                matrix[0].extend(rows)
                matrix[1].extend(cols)
                matrix[2].extend(data)
            else:
                matrix = process_batch_similarities(lda_corpus, i, batch)
        
        if use_sparse:
            similarity_matrices['lda'] = sparse.csr_matrix(
                (matrix[2], (matrix[0], matrix[1])),
                shape=(n_sents, n_sents)
            )
        else:
            similarity_matrices['lda'] = matrix
    
    return similarity_matrices

In [ ]:
#let's try to filter short sentences
from gensim import models, matutils
import numpy as np
from scipy import sparse
from tqdm import tqdm
from typing import Dict, List, Union, Optional, Tuple
import logging

def filter_sentences(corpus: 'GensimCorpus',
                    min_words: int = 5,
                    max_words: Optional[int] = None) -> Tuple[List[List[int]], List[int], List[Tuple[str, int]]]:
    """
    Filter sentences based on word count and create a mapping to original indices.
    
    Returns:
    - filtered_corpus: List of filtered bow vectors
    - original_indices: Mapping to original sentence indices
    - filtered_metadata: Filtered sentence metadata
    """
    filtered_corpus = []
    original_indices = []
    filtered_metadata = []
    
    for idx, (bow, (fileid, sent_idx)) in enumerate(zip(corpus, corpus.sentence_metadata)):
        word_count = sum(count for _, count in bow)
        
        if word_count >= min_words and (max_words is None or word_count <= max_words):
            filtered_corpus.append(bow)
            original_indices.append(idx)
            filtered_metadata.append((fileid, sent_idx))
    
    return filtered_corpus, original_indices, filtered_metadata

def compute_sent_similarity_matrices_filtered(corpus: 'GensimCorpus',
                                           min_words: int = 5,
                                           max_words: Optional[int] = None,
                                           models_to_compute: Union[str, List[str]] = "all",
                                           batch_size: int = 1000,
                                           min_similarity: float = 0.3,
                                           use_sparse: bool = True) -> Dict[str, dict]:
    """
    Compute sentence similarity matrices with preliminary filtering
    
    Parameters:
    - corpus: GensimCorpus object in 'sent' mode
    - min_words: Minimum number of words in a sentence
    - max_words: Optional maximum number of words in a sentence
    - models_to_compute: Which models to use ("all" or list of ["tfidf", "lsi", "lda"])
    - batch_size: Number of sentences to process at once
    - min_similarity: Minimum similarity threshold
    - use_sparse: Whether to use sparse matrices
    
    Returns:
    - Dictionary containing for each model:
        - 'matrix': similarity matrix
        - 'indices': mapping to original indices
        - 'metadata': filtered sentence metadata
    """
    if corpus.mode != 'sent':
        raise ValueError("Corpus must be in 'sent' mode")
    
    logger = logging.getLogger(__name__)
    
    # Filter sentences
    logger.info(f"Filtering sentences (min_words={min_words}, max_words={max_words})...")
    filtered_corpus, original_indices, filtered_metadata = filter_sentences(
        corpus, min_words, max_words
    )
    
    n_original = len(corpus.sentence_metadata)
    n_filtered = len(filtered_corpus)
    logger.info(f"Filtered {n_original} sentences to {n_filtered} "
                f"({n_filtered/n_original*100:.1f}%)")
    
    if models_to_compute == "all":
        models_to_compute = ["tfidf", "lsi", "lda"]
    elif isinstance(models_to_compute, str):
        models_to_compute = [models_to_compute]
    
    similarity_matrices = {}
    
    def create_matrix():
        if use_sparse:
            return [], [], []  # rows, cols, data for COO format
        return np.zeros((n_filtered, n_filtered))
    
    def process_batch_similarities(model_corpus, start_idx, batch):
        if use_sparse:
            rows, cols, data = [], [], []
            for i, sent_vector in enumerate(model_corpus):
                if i < start_idx + len(batch):
                    sims = [matutils.cossim(sent_vector, batch_vec) 
                           for batch_vec in batch]
                    
                    for j, sim in enumerate(sims):
                        if sim >= min_similarity and i != (start_idx + j):
                            rows.append(i)
                            cols.append(start_idx + j)
                            data.append(sim)
                            
                            if start_idx + j < i:
                                rows.append(start_idx + j)
                                cols.append(i)
                                data.append(sim)
            return rows, cols, data
        else:
            for i, sent_vector in enumerate(model_corpus):
                if i < start_idx + len(batch):
                    sims = [matutils.cossim(sent_vector, batch_vec) 
                           for batch_vec in batch]
                    matrix[i, start_idx:start_idx + len(sims)] = sims
                    matrix[start_idx:start_idx + len(sims), i] = sims
            return matrix
    
    # Create base models and compute similarities
    if "tfidf" in models_to_compute or "lsi" in models_to_compute:
        logger.info("Creating TF-IDF model...")
        tfidf_model = models.TfidfModel(filtered_corpus)
        tfidf_corpus = list(tfidf_model[filtered_corpus])
        
        if "tfidf" in models_to_compute:
            logger.info("Computing TF-IDF similarities...")
            matrix = create_matrix()
            
            for i in tqdm(range(0, n_filtered, batch_size)):
                batch = tfidf_corpus[i:i + batch_size]
                if use_sparse:
                    rows, cols, data = process_batch_similarities(tfidf_corpus, i, batch)
                    matrix[0].extend(rows)
                    matrix[1].extend(cols)
                    matrix[2].extend(data)
                else:
                    matrix = process_batch_similarities(tfidf_corpus, i, batch)
            
            if use_sparse:
                sim_matrix = sparse.csr_matrix(
                    (matrix[2], (matrix[0], matrix[1])),
                    shape=(n_filtered, n_filtered)
                )
            else:
                sim_matrix = matrix
                
            similarity_matrices['tfidf'] = {
                'matrix': sim_matrix,
                'indices': original_indices,
                'metadata': filtered_metadata
            }
    
        if "lsi" in models_to_compute:
            # Similar pattern for LSI...
            pass
    
    if "lda" in models_to_compute:
        # Similar pattern for LDA...
        pass
    
    return similarity_matrices

def get_similar_sentences_filtered(model_data: dict,
                                 sentence_idx: int,
                                 top_n: int = 5,
                                 return_original_idx: bool = True) -> List[dict]:
    """
    Get similar sentences, handling the filtering mapping
    """
    matrix = model_data['matrix']
    indices = model_data['indices']
    metadata = model_data['metadata']
    
    # Find the filtered index
    if return_original_idx:
        try:
            filtered_idx = indices.index(sentence_idx)
        except ValueError:
            return []  # Sentence was filtered out
    else:
        filtered_idx = sentence_idx
    
    if isinstance(matrix, sparse.csr_matrix):
        similarities = matrix[filtered_idx].toarray().flatten()
    else:
        similarities = matrix[filtered_idx]
    
    # Get top N similar sentences
    similar_idxs = np.argsort(similarities)[::-1][1:top_n+1]
    
    results = []
    for idx in similar_idxs:
        fileid, sent_idx = metadata[idx]
        results.append({
            'fileid': fileid,
            'sentence_idx': sent_idx,
            'original_idx': indices[idx],
            'similarity': similarities[idx]
        })
    
    return results

In [ ]:
import numpy as np
import json
import pickle
#import h5py
import pandas as pd
from pathlib import Path

def save_similarity_matrices(similarity_matrices, output_dir, format='npz', prefix=''):
    """
    Save similarity matrices using specified format
    
    Parameters:
    - similarity_matrices: dict of numpy arrays
    - output_dir: directory to save matrices
    - format: 'npz', 'npy', 'hdf5', 'pickle', or 'parquet'
    - prefix: prefix for saved files (e.g., 'topic50_' or 'window5_')
    
    Returns:
    - dict with saved file paths
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_paths = {}
    
    # Ensure prefix ends with underscore if not empty
    if prefix and not prefix.endswith('_'):
        prefix = f"{prefix}_"

    if format == 'npz':
        output_path = output_dir / f'{prefix}similarity_matrices.npz'
        np.savez_compressed(output_path, **similarity_matrices)
        saved_paths['npz'] = output_path

    elif format == 'npy':
        for model_name, matrix in similarity_matrices.items():
            output_path = output_dir / f'{prefix}{model_name}_similarities.npy'
            np.save(output_path, matrix)
            saved_paths[model_name] = output_path

    elif format == 'hdf5':
        output_path = output_dir / f'{prefix}similarity_matrices.h5'
        with h5py.File(output_path, 'w') as f:
            for model_name, matrix in similarity_matrices.items():
                f.create_dataset(model_name, data=matrix, 
                               compression='gzip', 
                               compression_opts=9)
        saved_paths['hdf5'] = output_path

    elif format == 'pickle':
        output_path = output_dir / f'{prefix}similarity_matrices.pkl'
        with open(output_path, 'wb') as f:
            pickle.dump(similarity_matrices, f, protocol=pickle.HIGHEST_PROTOCOL)
        saved_paths['pickle'] = output_path
        
    elif format == 'parquet':
        for model_name, matrix in similarity_matrices.items():
            # Convert matrix to DataFrame with appropriate column names
            df = pd.DataFrame(matrix)
            df.columns = [f'doc_{i}' for i in range(matrix.shape[1])]
            
            output_path = output_dir / f'{prefix}{model_name}_similarities.parquet'
            df.to_parquet(output_path, compression='snappy')
            saved_paths[model_name] = output_path

    # Save metadata
    metadata = {
        'format': format,
        'prefix': prefix,
        'models': list(similarity_matrices.keys()),
        'shapes': {model: matrix.shape for model, matrix in similarity_matrices.items()},
        'files': {k: str(v) for k, v in saved_paths.items()}
    }
    
    metadata_path = output_dir / f'{prefix}metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    return saved_paths

def load_similarity_matrices(input_dir, format='npz', prefix=''):
    """
    Load similarity matrices from specified format
    
    Parameters:
    - input_dir: directory containing saved matrices
    - format: 'npz', 'npy', 'hdf5', 'pickle', or 'parquet'
    - prefix: prefix used when saving files
    
    Returns:
    - dict of numpy arrays
    """
    input_dir = Path(input_dir)
    
    # Ensure prefix ends with underscore if not empty
    if prefix and not prefix.endswith('_'):
        prefix = f"{prefix}_"
    
    # Load metadata if available
    metadata_path = input_dir / f'{prefix}metadata.json'
    if metadata_path.exists():
        with open(metadata_path) as f:
            metadata = json.load(f)
    else:
        metadata = None

    if format == 'npz':
        npz_path = input_dir / f'{prefix}similarity_matrices.npz'
        with np.load(npz_path) as data:
            return {key: data[key] for key in data.files}

    elif format == 'npy':
        if metadata:
            return {model: np.load(input_dir / f'{prefix}{model}_similarities.npy') 
                   for model in metadata['models']}
        else:
            matrices = {}
            for npy_file in input_dir.glob(f'{prefix}*_similarities.npy'):
                model_name = npy_file.stem.replace('_similarities', '')
                model_name = model_name[len(prefix):] if prefix else model_name
                matrices[model_name] = np.load(npy_file)
            return matrices

    elif format == 'hdf5':
        matrices = {}
        with h5py.File(input_dir / f'{prefix}similarity_matrices.h5', 'r') as f:
            for model_name in f.keys():
                matrices[model_name] = f[model_name][:]
        return matrices

    elif format == 'pickle':
        with open(input_dir / f'{prefix}similarity_matrices.pkl', 'rb') as f:
            return pickle.load(f)
            
    elif format == 'parquet':
        if metadata:
            matrices = {}
            for model_name in metadata['models']:
                df = pd.read_parquet(input_dir / f'{prefix}{model_name}_similarities.parquet')
                matrices[model_name] = df.values
            return matrices
        else:
            matrices = {}
            for parquet_file in input_dir.glob(f'{prefix}*_similarities.parquet'):
                model_name = parquet_file.stem.replace('_similarities', '')
                model_name = model_name[len(prefix):] if prefix else model_name
                df = pd.read_parquet(parquet_file)
                matrices[model_name] = df.values
            return matrices

    raise ValueError(f"Unsupported format: {format}")

def benchmark_serialization(similarity_matrices, output_dir, prefix=''):
    """
    Benchmark different serialization methods
    
    Parameters:
    - similarity_matrices: dict of similarity matrices
    - output_dir: directory to save matrices
    - prefix: prefix for saved files
    
    Returns:
    - dict with benchmarking results
    """
    import time
    import os
    
    results = {}
    formats = ['npz', 'npy', 'hdf5', 'pickle', 'parquet']
    
    for fmt in formats:
        output_subdir = Path(output_dir) / fmt
        
        # Measure save time
        start_time = time.time()
        saved_paths = save_similarity_matrices(
            similarity_matrices, 
            output_subdir, 
            format=fmt,
            prefix=prefix
        )
        save_time = time.time() - start_time
        
        # Measure file sizes
        if fmt in ['npy', 'parquet']:
            total_size = sum(os.path.getsize(str(p)) for p in saved_paths.values())
        else:
            total_size = os.path.getsize(str(list(saved_paths.values())[0]))
        
        # Measure load time
        start_time = time.time()
        loaded_matrices = load_similarity_matrices(
            output_subdir, 
            format=fmt,
            prefix=prefix
        )
        load_time = time.time() - start_time
        
        results[fmt] = {
            'save_time': save_time,
            'load_time': load_time,
            'total_size': total_size,
            'files': saved_paths
        }
    
    return results

In [ ]:
if mode =="build":
    similarity_matrices_words = create_similarity_matrices(corpus_gensim_words, corpus_gensim_words_dict)
    similarity_matrices_lemmas = create_similarity_matrices(corpus_gensim_lemmas, corpus_gensim_lemmas_dict)
    save_similarity_matrices(similarity_matrices_words,output_dir="models",prefix="words_")
    save_similarity_matrices(similarity_matrices_lemmas,output_dir="models",prefix="lemmas_")
elif mode == "load":
    similarity_matrices_words = load_similarity_matrices(input_dir="models", prefix="words_")
    similarity_matrices_lemmas = load_similarity_matrices(input_dir="models", prefix="lemmas_")

#### Subcorpus

In [ ]:
# better to use class
def create_subcorpus(source_corpus: GensimCorpus, fileids: List[str]) -> GensimCorpus:
    """
    Creates a new corpus instance from selected files of an existing corpus.
    Does not modify the source corpus or rebuild from files.
    
    Args:
        source_corpus: Source GensimCorpus instance
        fileids: List of file IDs to include in the subcorpus
    
    Returns:
        GensimCorpus: New corpus instance with only the selected files
    """
    # Create a new instance
    subcorpus = GensimCorpus.__new__(GensimCorpus)
    
    # Copy basic attributes
    subcorpus.corpus_dir = source_corpus.corpus_dir
    subcorpus.corpus_files = fileids
    subcorpus.use_lemmas = source_corpus.use_lemmas
    subcorpus.min_freq = source_corpus.min_freq
    subcorpus.mode = source_corpus.mode
    subcorpus.stopwords_list = source_corpus.stopwords_list
    
    # Set up logging
    subcorpus.logger = logging.getLogger(__name__)
    
    # Filter the corpus data
    subcorpus.corpus = {
        fileid: source_corpus.corpus[fileid] 
        for fileid in fileids 
        if fileid in source_corpus.corpus
    }
    
    # Filter successful and failed files
    subcorpus.successful_files = [
        f for f in source_corpus.successful_files 
        if f in fileids
    ]
    subcorpus.failed_files = {
        k: v for k, v in source_corpus.failed_files.items() 
        if k in fileids
    }
    
    # Filter sentence metadata if in sentence mode
    if source_corpus.mode == 'sent':
        subcorpus.sentence_metadata = [
            (fileid, sent_idx) 
            for fileid, sent_idx in source_corpus.sentence_metadata
            if fileid in fileids
        ]
    else:
        subcorpus.sentence_metadata = []
    
    # Reuse the existing dictionary
    subcorpus.dictionary = source_corpus.dictionary
    
    return subcorpus

In [ ]:
books = ['Ksg', 'Lib', 'StPPP11', 'StPPP8', 'Pomn', 'AKap']
if mode == 'build':
    corpus_gensim_sents_words_books = corpus_gensim_sents_words.create_subcorpus(fileids=[work for work in corpus_gensim_sents_words.successful_files 
                                                               if work.startswith(tuple(books))])
    corpus_gensim_sents_lemmas_books = corpus_gensim_sents_lemmas.create_subcorpus(fileids=[work for work in corpus_gensim_sents_lemmas.successful_files 
                                                               if work.startswith(tuple(books))])
    corpus_gensim_sents_words_books.serialize('sents_words_books','corpora')
    corpus_gensim_sents_lemmas_books.serialize('sents_lemmas_books','corpora')    
elif mode == 'load':
    corpus_gensim_sents_lemmas_books = GensimCorpus.load('sents_lemmas_books','corpora')
    corpus_gensim_sents_words_books = GensimCorpus.load('sents_words_books','corpora')

In [ ]:
corpus_gensim_sents_words_books = corpus_gensim_sents_words.create_subcorpus(fileids=[work for work in corpus_gensim_sents_words.successful_files 
                                                               if work.startswith(tuple(books))])
corpus_gensim_sents_lemmas_books = corpus_gensim_sents_lemmas.create_subcorpus(fileids=[work for work in corpus_gensim_sents_lemmas.successful_files 
                                                               if work.startswith(tuple(books))])
corpus_gensim_sents_words_books.serialize('sents_words_books','corpora')
corpus_gensim_sents_lemmas_books.serialize('sents_lemmas_books','corpora')  

In [ ]:
#similarity_matrices_sents_lemmas_books = compute_sent_similarity_matrices(corpus=corpus_gensim_sents_lemmas_books,
    #corpus_gensim_sents_lemmas_books.dictionary,
#    models_to_compute='tfidf', batch_size=1000, min_similarity=0.3, use_sparse=True)
similarity_matrices_sents_lemmas_books = compute_sent_similarity_matrices_filtered(
    corpus_gensim_sents_lemmas_books,
    min_words=5,  # Filter out sentences with fewer than 5 words
    max_words=50,  # Optional: filter out very long sentences
    models_to_compute=['tfidf'],
    min_similarity=0.3,
    use_sparse=True
)

save_similarity_matrices(similarity_matrices_sents_lemmas_books,output_dir="models",prefix="sents_lemmas_books")

In [ ]:
similarity_matrices_sents_words_books = compute_sent_similarity_matrices_filtered(
    corpus_gensim_sents_lemmas_books,
    #corpus_gensim_sents_lemmas_books.dictionary,
    min_words=5,  # Filter out sentences with fewer than 5 words
    max_words=50,  # Optional: filter out very long sentences
    models_to_compute=['tfidf'],
    min_similarity=0.3,
    use_sparse=True
                                                                                 )
save_similarity_matrices(similarity_matrices_sents_words_books,output_dir="models",prefix="sents_words_books")

# Text-level

## Tfidf, LSI, LSA

In [ ]:
#doc_labels = [f"Document {i+1}" for i in range(len(corpus_gensim_words))]
for model_name, similarity_matrix in similarity_matrices_words.items():
    visualizations = create_similarity_visualizations(
        similarity_matrix,
        doc_labels=[lbl.replace("_TEI_final.conllu","") for lbl in corpus_gensim_words.successful_files],
        title_prefix=model_name.upper()
    )
    
    # Display visualizations
    for viz_name, fig in visualizations.items():
        fig.show()

   
#doc_labels = [f"Document {i+1}" for i in range(len(corpus_gensim_words))]
for model_name, similarity_matrix in similarity_matrices_lemmas.items():
    visualizations = create_similarity_visualizations(
        similarity_matrix,
        doc_labels=[lbl.replace("_TEI_final.conllu","") for lbl in corpus_gensim_lemmas.successful_files],
        title_prefix=model_name.upper()
    )
    
    # Display visualizations
    for viz_name, fig in visualizations.items():
        fig.show()

## Doc2Vec

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from dataclasses import dataclass
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from typing import Optional, Dict, List, Tuple, Union
import logging

@dataclass
class EmbeddingData:
    """Container for embedding data and metadata"""
    vectors: np.ndarray
    doc_ids: List[str]
    fileid_metadata: Union[List[str], List[Tuple[str, int]]]
    mode: str  # 'doc' or 'sent'

class Doc2VecAnalyzer:
    """Handles Doc2Vec model training, dimension reduction and clustering"""
    def __init__(self, corpus: 'GensimCorpus', 
                 vector_size: int = 100,
                 min_count: int = 2,
                 epochs: int = 20,
                 workers: int = 4):
        """
        Initialize and train Doc2Vec model from GensimCorpus
        
        Args:
            corpus: GensimCorpus instance
            vector_size: Dimensionality of vectors
            min_count: Min frequency for vocabulary
            epochs: Number of training epochs
            workers: Number of worker threads
        """
        self.logger = logging.getLogger(__name__)
        self.corpus = corpus
        self.mode = corpus.mode
        
        # Train Doc2Vec model
        self.model = self._train_model(
            vector_size=vector_size,
            min_count=min_count,
            epochs=epochs,
            workers=workers
        )
        
        # Extract embeddings and metadata
        self.embedding_data = self._prepare_embedding_data()
        
        # Storage for reduced embeddings and clusters
        self.reduced_data: Dict[str, pd.DataFrame] = {}
        self.cluster_data: Dict[str, pd.DataFrame] = {}
    
    def _train_model(self, **model_params) -> Doc2Vec:
        """Train Doc2Vec model on corpus"""
        self.logger.info("Preparing documents for Doc2Vec training...")
        
        # Prepare tagged documents
        tagged_docs = []
        if self.mode == 'doc':
            for doc, fileid in zip(self.corpus.get_texts(), 
                                 self.corpus.successful_files):
                tagged_docs.append(TaggedDocument(doc, [fileid]))
        else:  # sent mode
            for sent, (fileid, sent_idx) in zip(self.corpus.get_texts(),
                                              self.corpus.sentence_metadata):
                doc_id = f"{fileid}_{sent_idx}"
                tagged_docs.append(TaggedDocument(sent, [doc_id]))
        
        self.logger.info(f"Training Doc2Vec model with params: {model_params}")
        model = Doc2Vec(tagged_docs, **model_params)
        self.logger.info("Doc2Vec model training completed")
        
        return model
    
    def _prepare_embedding_data(self) -> EmbeddingData:
        """Extract embedding data from trained model"""
        if self.mode == 'doc':
            doc_ids = self.corpus.successful_files
            fileid_metadata = self.corpus.successful_files
        else:
            doc_ids = [f"{fid}_{idx}" for fid, idx in self.corpus.sentence_metadata]
            fileid_metadata = self.corpus.sentence_metadata
        
        vectors = np.vstack([self.model.dv[doc_id] for doc_id in doc_ids])
        
        return EmbeddingData(
            vectors=vectors,
            doc_ids=doc_ids,
            fileid_metadata=fileid_metadata,
            mode=self.mode
        )
    
    def compute_reduced_embeddings(self, 
                                 method: str = 'tsne',
                                 n_components: int = 2,
                                 **kwargs) -> pd.DataFrame:
        """Compute reduced dimensionality embeddings"""
        method_key = f"{method}_{n_components}d"
        
        # Check if already computed
        if method_key in self.reduced_data:
            return self.reduced_data[method_key]
        
        self.logger.info(f"Computing {method.upper()} reduction...")
        
        # Compute reduction
        if method == 'tsne':
            reducer = TSNE(n_components=n_components, 
                         perplexity=kwargs.get('perplexity', 30.0))
        elif method == 'pca':
            reducer = PCA(n_components=n_components)
        else:
            raise ValueError("Method must be 'tsne' or 'pca'")
            
        reduced_vectors = reducer.fit_transform(self.embedding_data.vectors)
        
        # Create DataFrame with metadata
        data = {
            'x': reduced_vectors[:, 0],
            'y': reduced_vectors[:, 1],
            'doc_id': self.embedding_data.doc_ids,
        }
        
        if n_components == 3:
            data['z'] = reduced_vectors[:, 2]
            
        if self.mode == 'sent':
            data['fileid'] = [meta[0] for meta in self.embedding_data.fileid_metadata]
            data['sent_idx'] = [meta[1] for meta in self.embedding_data.fileid_metadata]
        else:
            data['fileid'] = self.embedding_data.fileid_metadata
            
        df = pd.DataFrame(data)
        self.reduced_data[method_key] = df
        
        self.logger.info(f"{method.upper()} reduction completed")
        return df
    
    def compute_clusters(self,
                        method: str = 'kmeans',
                        embeddings: str = 'original',
                        **kwargs) -> pd.DataFrame:
        """Compute clusters from embeddings"""
        method_key = f"{method}_{embeddings}"
        
        # Check if already computed
        if method_key in self.cluster_data:
            return self.cluster_data[method_key]
        
        self.logger.info(f"Computing {method.upper()} clustering...")
        
        # Get appropriate vectors
        if embeddings == 'original':
            vectors = self.embedding_data.vectors
        else:
            if embeddings not in self.reduced_data:
                raise ValueError(f"Embeddings '{embeddings}' not found. Compute them first.")
            df = self.reduced_data[embeddings]
            vectors = df[['x', 'y']].values
            if 'z' in df.columns:
                vectors = np.column_stack([vectors, df['z'].values])
        
        # Compute clusters
        if method == 'kmeans':
            n_clusters = kwargs.get('n_clusters', int(np.sqrt(len(vectors))))
            clusterer = KMeans(n_clusters=n_clusters)
        elif method == 'dbscan':
            clusterer = DBSCAN(
                eps=kwargs.get('eps', 0.5),
                min_samples=kwargs.get('min_samples', 5)
            )
        else:
            raise ValueError("Method must be 'kmeans' or 'dbscan'")
            
        labels = clusterer.fit_predict(vectors)
        
        # Create DataFrame with results
        df = pd.DataFrame({
            'doc_id': self.embedding_data.doc_ids,
            'cluster': [f'Cluster {l}' for l in labels]
        })
        
        self.cluster_data[method_key] = df
        self.logger.info(f"{method.upper()} clustering completed")
        return df
    
    def most_similar(self, doc_id: str, top_n: int = 5) -> List[Tuple[str, float]]:
        """Find most similar documents to given doc_id"""
        if doc_id not in self.model.dv:
            raise ValueError(f"Document ID {doc_id} not found in model")
        return self.model.dv.most_similar(doc_id, topn=top_n)
    
    def save_model(self, path: str):
        """Save Doc2Vec model"""
        self.model.save(path)
    
    @classmethod
    def load_model(cls, path: str, corpus: 'GensimCorpus') -> 'Doc2VecAnalyzer':
        """Load saved Doc2Vec model"""
        instance = cls.__new__(cls)
        instance.logger = logging.getLogger(__name__)
        instance.corpus = corpus
        instance.mode = corpus.mode
        instance.model = Doc2Vec.load(path)
        instance.embedding_data = instance._prepare_embedding_data()
        instance.reduced_data = {}
        instance.cluster_data = {}
        return instance

In [ ]:
analyzer = Doc2VecAnalyzer(
    corpus_gensim_sents_words_books,
    vector_size=100,
    min_count=5,
    epochs=10
)

In [ ]:
analyzer.compute_reduced_embeddings('tsne', n_components=2, perplexity=30.0) #very long

In [ ]:
analyzer.compute_reduced_embeddings('pca', n_components=2)

In [ ]:
from dataclasses import dataclass
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from typing import Optional, Dict, List, Tuple, Union
import holoviews as hv
from holoviews import opts
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool, BoxSelectTool, TapTool
from bokeh.palettes import Spectral11

@dataclass
class EmbeddingData:
    """Container for embedding data and metadata"""
    vectors: np.ndarray
    doc_ids: List[str]
    fileid_metadata: Union[List[str], List[Tuple[str, int]]]
    mode: str  # 'doc' or 'sent'

class Doc2VecAnalyzer:
    """Handles model computation and dimensionality reduction"""
    def __init__(self, doc2vec_model, corpus):
        self.model = doc2vec_model
        self.corpus = corpus
        self.embedding_data = self._prepare_embedding_data()
        self.reduced_data: Dict[str, pd.DataFrame] = {}
        self.cluster_data: Dict[str, pd.DataFrame] = {}
    
    def _prepare_embedding_data(self) -> EmbeddingData:
        """Extract embedding data from model"""
        doc_ids = (self.corpus.successful_files if self.corpus.mode == 'doc' 
                  else [f"{fid}_{idx}" for fid, idx in self.corpus.sentence_metadata])
        
        vectors = np.vstack([self.model.dv[doc_id] for doc_id in doc_ids])
        
        fileid_metadata = (self.corpus.successful_files if self.corpus.mode == 'doc'
                          else self.corpus.sentence_metadata)
        
        return EmbeddingData(
            vectors=vectors,
            doc_ids=doc_ids,
            fileid_metadata=fileid_metadata,
            mode=self.corpus.mode
        )
    
    def compute_reduced_embeddings(self, 
                                 method: str = 'tsne',
                                 n_components: int = 2,
                                 **kwargs) -> pd.DataFrame:
        """
        Compute reduced dimensionality embeddings
        """
        method_key = f"{method}_{n_components}d"
        
        # Check if already computed
        if method_key in self.reduced_data:
            return self.reduced_data[method_key]
        
        # Compute reduction
        if method == 'tsne':
            reducer = TSNE(n_components=n_components, 
                         perplexity=kwargs.get('perplexity', 30.0))
        elif method == 'pca':
            reducer = PCA(n_components=n_components)
        else:
            raise ValueError("Method must be 'tsne' or 'pca'")
            
        reduced_vectors = reducer.fit_transform(self.embedding_data.vectors)
        
        # Create DataFrame with metadata
        data = {
            'x': reduced_vectors[:, 0],
            'y': reduced_vectors[:, 1],
            'doc_id': self.embedding_data.doc_ids,
        }
        
        if n_components == 3:
            data['z'] = reduced_vectors[:, 2]
            
        if self.embedding_data.mode == 'sent':
            data['fileid'] = [meta[0] for meta in self.embedding_data.fileid_metadata]
            data['sent_idx'] = [meta[1] for meta in self.embedding_data.fileid_metadata]
        else:
            data['fileid'] = self.embedding_data.fileid_metadata
            
        df = pd.DataFrame(data)
        self.reduced_data[method_key] = df
        return df
    
    def compute_clusters(self,
                        method: str = 'kmeans',
                        embeddings: str = 'original',
                        **kwargs) -> pd.DataFrame:
        """
        Compute clusters from embeddings
        
        Args:
            method: 'kmeans' or 'dbscan'
            embeddings: 'original' or key from reduced_data
            **kwargs: Arguments for clustering method
        """
        method_key = f"{method}_{embeddings}"
        
        # Check if already computed
        if method_key in self.cluster_data:
            return self.cluster_data[method_key]
        
        # Get appropriate vectors
        if embeddings == 'original':
            vectors = self.embedding_data.vectors
        else:
            if embeddings not in self.reduced_data:
                raise ValueError(f"Embeddings '{embeddings}' not found. Compute them first.")
            df = self.reduced_data[embeddings]
            vectors = df[['x', 'y']].values
            if 'z' in df.columns:
                vectors = np.column_stack([vectors, df['z'].values])
        
        # Compute clusters
        if method == 'kmeans':
            n_clusters = kwargs.get('n_clusters', int(np.sqrt(len(vectors))))
            clusterer = KMeans(n_clusters=n_clusters)
        elif method == 'dbscan':
            clusterer = DBSCAN(
                eps=kwargs.get('eps', 0.5),
                min_samples=kwargs.get('min_samples', 5)
            )
        else:
            raise ValueError("Method must be 'kmeans' or 'dbscan'")
            
        labels = clusterer.fit_predict(vectors)
        
        # Create DataFrame with results
        df = pd.DataFrame({
            'doc_id': self.embedding_data.doc_ids,
            'cluster': [f'Cluster {l}' for l in labels]
        })
        
        self.cluster_data[method_key] = df
        return df

class Doc2VecVisualizer:
    """Handles visualization of embeddings and clusters"""
    def __init__(self, analyzer: Doc2VecAnalyzer):
        self.analyzer = analyzer
    
    def bokeh_plot(self,
                   embeddings_key: str,
                   cluster_key: Optional[str] = None,
                   width: int = 800,
                   height: int = 600) -> figure:
        """Create Bokeh plot from pre-computed embeddings and clusters"""
        # Get embedding data
        if embeddings_key not in self.analyzer.reduced_data:
            raise ValueError(f"Embeddings '{embeddings_key}' not found")
        df = self.analyzer.reduced_data[embeddings_key].copy()
        
        # Add cluster data if requested
        if cluster_key:
            if cluster_key not in self.analyzer.cluster_data:
                raise ValueError(f"Clusters '{cluster_key}' not found")
            df = df.merge(self.analyzer.cluster_data[cluster_key], on='doc_id')
        
        source = ColumnDataSource(df)
        
        # Create figure
        tools = [BoxSelectTool(), TapTool(), 'pan', 'wheel_zoom', 'reset', 'save']
        p = figure(width=width, height=height, tools=tools,
                  title=f'Document Vectors ({embeddings_key})')
        
        # Add hover tool
        hover = HoverTool(tooltips=[
            ('Document', '@doc_id'),
            ('File ID', '@fileid'),
        ])
        if self.analyzer.embedding_data.mode == 'sent':
            hover.tooltips.append(('Sentence Index', '@sent_idx'))
        if cluster_key:
            hover.tooltips.append(('Cluster', '@cluster'))
        p.add_tools(hover)
        
        # Plot points
        if cluster_key:
            unique_clusters = sorted(df['cluster'].unique())
            colors = Spectral11[:len(unique_clusters)]
            for cluster, color in zip(unique_clusters, colors):
                mask = df['cluster'] == cluster
                cluster_source = ColumnDataSource(df[mask])
                p.circle('x', 'y', size=8, alpha=0.6,
                        color=color, legend_label=cluster,
                        source=cluster_source)
            p.legend.click_policy = 'hide'
        else:
            p.circle('x', 'y', size=8, alpha=0.6, color='navy',
                    source=source)
        
        return p
    
    def holoviews_plot(self,
                      embeddings_key: str,
                      cluster_key: Optional[str] = None,
                      width: int = 800,
                      height: int = 600) -> hv.Points:
        """Create HoloViews plot from pre-computed embeddings and clusters"""
        hv.extension('bokeh')
        
        # Get embedding data
        if embeddings_key not in self.analyzer.reduced_data:
            raise ValueError(f"Embeddings '{embeddings_key}' not found")
        df = self.analyzer.reduced_data[embeddings_key].copy()
        
        # Add cluster data if requested
        if cluster_key:
            if cluster_key not in self.analyzer.cluster_data:
                raise ValueError(f"Clusters '{cluster_key}' not found")
            df = df.merge(self.analyzer.cluster_data[cluster_key], on='doc_id')
        
        # Create plot
        if cluster_key:
            points = hv.Points(df, kdims=['x', 'y'], 
                             vdims=['doc_id', 'fileid', 'cluster'])
            plot = points.opts(
                opts.Points(
                    color='cluster',
                    cmap='Category20',
                    size=8,
                    alpha=0.6,
                    width=width,
                    height=height,
                    tools=['hover', 'box_select', 'tap'],
                    title=f'Document Vectors ({embeddings_key})',
                    legend_position='right'
                )
            )
        else:
            points = hv.Points(df, kdims=['x', 'y'], 
                             vdims=['doc_id', 'fileid'])
            plot = points.opts(
                opts.Points(
                    color='navy',
                    size=8,
                    alpha=0.6,
                    width=width,
                    height=height,
                    tools=['hover', 'box_select', 'tap'],
                    title=f'Document Vectors ({embeddings_key})'
                )
            )
        
        return plot

In [ ]:
# Compute clusters
analyzer.compute_clusters('kmeans', embeddings='original', n_clusters=10)
analyzer.compute_clusters('kmeans', embeddings='pca_2d', n_clusters=10)
analyzer.compute_clusters('dbscan', embeddings='tsne_2d', eps=0.5)

In [ ]:
visualizer = Doc2VecVisualizer(analyzer)
bokeh_plot = visualizer.bokeh_plot(
    embeddings_key='pca_2d',  # This matches the key created by compute_reduced_embeddings
    width=800,
    height=600
)
show(bokeh_plot)  # If in Jupyter notebook

In [ ]:
# Option 2: HoloViews plot of PCA
hv_plot = visualizer.holoviews_plot(
    embeddings_key='pca_2d',
    width=800,
    height=600
)
hv_plot  # If in Jupyter notebook

In [ ]:
# You can also add clustering:
# First compute clusters
#analyzer.compute_clusters('kmeans', embeddings='pca_2d', n_clusters=5)

# Then visualize with clusters
from bokeh.plotting import figure, show

bokeh_plot_clustered = visualizer.bokeh_plot(
    embeddings_key='pca_2d',
    cluster_key='kmeans_pca_2d',
    
)
show(bokeh_plot_clustered)

## Topic Modelling

### Build Models

In [ ]:
from gensim.models import LdaModel
lda_model_words = LdaModel(corpus_gensim_words.corpus,
                         num_topics=20, 
                         id2word=corpus_gensim_words.dictionary, passes=10)

In [ ]:
from gensim.models import LdaModel
mode = mode_selector.value
if mode == 'build':
    lda_model_words = LdaModel(corpus_gensim_words.corpus,
                         num_topics=20, 
                         id2word=corpus_gensim_words.dictionary, passes=10)
    lda_model_words.save('models/model_lda_words.gensim')
    lda_model_lemmas = LdaModel(corpus_gensim_lemmas,
                         num_topics=20, 
                         id2word=corpus_gensim_lemmas.dictionary, passes=10)
    lda_model_lemmas.save('models/model_lda_lemmas.gensim')
elif mode == 'load':
    lda_model_words = LdaModel.load('models/model_lda_words.gensim')
    lda_model_lemmas = LdaModel.load('models/model_lda_lemmas.gensim')

In [ ]:
from gensim.models import LdaModel

lda_model_words = LdaModel(corpus_gensim_words.corpus,
                     num_topics=20, 
                     id2word=corpus_gensim_words.dictionary, passes=10)
lda_model_words.save('models/model_lda_words.gensim')

lda_model_lemmas = LdaModel(corpus_gensim_lemmas,
                     num_topics=20, 
                     id2word=corpus_gensim_lemmas.dictionary, passes=10)
lda_model_lemmas.save('models/model_lda_lemmas.gensim')

In [ ]:
from gensim.models import LdaModel

lda_model_words = LdaModel.load('models/model_lda_words.gensim')
lda_model_lemmas = LdaModel.load('models/model_lda_lemmas.gensim')

In [ ]:
topics_words = lda_model_words.print_topics(num_words=5)  # Display top 5 words per topic
topics_lemmas = lda_model_lemmas.print_topics(num_words=5)  # Display top 5 words per topic

In [ ]:
for topic in topics_words:
    print(f"Topic {topic[0]}: {topic[1]}")

In [ ]:
for topic in topics_lemmas:
    print(f"Topic {topic[0]}: {topic[1]}")

#### Clustering documents

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from collections import defaultdict

def get_document_topics_matrix(lda_model, corpus):
    """
    Convert document-topic distributions to a matrix format.
    
    Args:
        lda_model: Trained LDA model
        corpus: List of documents in bag-of-words format
        
    Returns:
        numpy array with shape (n_documents, n_topics)
    """
    # Initialize matrix
    n_docs = len(corpus)
    n_topics = lda_model.num_topics
    doc_topic_matrix = np.zeros((n_docs, n_topics))
    
    # Fill matrix with topic distributions
    for i, doc in enumerate(corpus):
        topic_dist = lda_model.get_document_topics(doc, minimum_probability=0)
        for topic_id, prob in topic_dist:
            doc_topic_matrix[i, topic_id] = prob
            
    return doc_topic_matrix

def cluster_documents(doc_topic_matrix, n_clusters=5):
    """
    Cluster documents based on their topic distributions.
    
    Args:
        doc_topic_matrix: Matrix of document-topic distributions
        n_clusters: Number of clusters to create
        
    Returns:
        kmeans: Fitted KMeans model
        labels: Cluster assignments
        silhouette_avg: Silhouette score for clustering
    """
    # Perform K-means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(doc_topic_matrix)
    
    # Calculate silhouette score
    silhouette_avg = silhouette_score(doc_topic_matrix, labels)
    
    return kmeans, labels, silhouette_avg

def analyze_clusters(lda_model, doc_topic_matrix, labels):
    """
    Analyze the characteristics of each cluster.
    
    Returns dictionary with cluster analysis results.
    """
    cluster_analysis = defaultdict(dict)
    
    for cluster_id in np.unique(labels):
        # Get documents in this cluster
        cluster_docs = doc_topic_matrix[labels == cluster_id]
        
        # Calculate average topic distribution for cluster
        avg_topic_dist = cluster_docs.mean(axis=0)
        
        # Find dominant topics
        dominant_topics = sorted(
            enumerate(avg_topic_dist),
            key=lambda x: x[1],
            reverse=True
        )[:3]
        
        # Store results
        cluster_analysis[f'Cluster {cluster_id}'] = {
            'size': len(cluster_docs),
            'dominant_topics': [
                (topic_id, prob, lda_model.show_topic(topic_id, topn=5))
                for topic_id, prob in dominant_topics
            ]
        }
    
    return cluster_analysis

def plot_cluster_sizes(labels):
    """Plot distribution of documents across clusters."""
    plt.figure(figsize=(10, 6))
    cluster_sizes = np.bincount(labels)
    plt.bar(range(len(cluster_sizes)), cluster_sizes)
    plt.title('Distribution of Documents Across Clusters')
    plt.xlabel('Cluster ID')
    plt.ylabel('Number of Documents')
    plt.show()

# Example usage:
# corpus = ... # Your corpus in bag-of-words format
# doc_topic_matrix = get_document_topics_matrix(lda_model_words, corpus)
# kmeans, labels, silhouette = cluster_documents(doc_topic_matrix)
# cluster_analysis = analyze_clusters(lda_model_words, doc_topic_matrix, labels)
# plot_cluster_sizes(labels)

In [ ]:
import numpy as np
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

def visualize_document_clusters(doc_topic_matrix, labels, method='tsne', plot_size=(15, 10)):
    """
    Visualize document clusters using dimensionality reduction.
    
    Args:
        doc_topic_matrix: Matrix of document-topic distributions
        labels: Cluster assignments from KMeans
        method: 'tsne' or 'pca' for dimensionality reduction
        plot_size: Tuple of (width, height) for the plot
    """
    # Set up the plot style
    plt.figure(figsize=plot_size)
    sns.set_style("whitegrid")
    
    # Reduce dimensionality
    if method.lower() == 'tsne':
        reducer = TSNE(n_components=2, random_state=42)
        embedding = reducer.fit_transform(doc_topic_matrix)
        title = 't-SNE visualization of document clusters'
    else:  # PCA
        reducer = PCA(n_components=2, random_state=42)
        embedding = reducer.fit_transform(doc_topic_matrix)
        title = 'PCA visualization of document clusters'
    
    # Create scatter plot
    scatter = plt.scatter(
        embedding[:, 0], 
        embedding[:, 1],
        c=labels,
        cmap='tab20',
        alpha=0.6
    )
    
    # Add labels and legend
    plt.title(title, fontsize=14, pad=20)
    plt.xlabel(f'{method.upper()} Component 1', fontsize=12)
    plt.ylabel(f'{method.upper()} Component 2', fontsize=12)
    
    # Add legend with cluster labels
    legend1 = plt.legend(*scatter.legend_elements(),
                        loc="upper right",
                        title="Clusters")
    plt.add_artist(legend1)
    
    # Add cluster centers if available
    try:
        centers = kmeans.cluster_centers_
        centers_reduced = reducer.transform(centers)
        plt.scatter(
            centers_reduced[:, 0],
            centers_reduced[:, 1],
            c='red',
            marker='x',
            s=200,
            linewidths=3,
            label='Cluster Centers'
        )
        plt.legend()
    except:
        pass
    
    plt.tight_layout()
    plt.show()

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA


def create_interactive_cluster_viz(doc_topic_matrix, labels, doc_labels=None, docs_text=None, 
                                 method='tsne', show_text=True, text_size=10, 
                                 show_features=False, lda_model=None,
                                 jitter_scale=0.02):
    """
    Create enhanced interactive visualization with jitter and optional feature arrows.
    
    Args:
        doc_topic_matrix: Matrix of document-topic distributions
        labels: Cluster assignments
        doc_labels: List of labels for each document
        docs_text: List of full document texts
        method: 'tsne' or 'pca'
        show_text: Whether to show text labels
        text_size: Size of text labels
        show_features: Whether to show feature (topic) arrows (only works with PCA)
        lda_model: LDA model (required if show_features=True)
        jitter_scale: Amount of random jitter to add (0 for no jitter)
    """
    # Reduce dimensionality
    if method.lower() == 'tsne':
        reducer = TSNE(n_components=2, random_state=42)
        embedding = reducer.fit_transform(doc_topic_matrix)
        if show_features:
            print("Feature arrows only available with PCA method")
            show_features = False
    else:  # PCA
        reducer = PCA(n_components=2, random_state=42)
        embedding = reducer.fit_transform(doc_topic_matrix)
    
    # Add jitter to prevent overlap
    if jitter_scale > 0:
        jitter = np.random.normal(0, jitter_scale, embedding.shape)
        embedding = embedding + jitter
    
    # Create DataFrame for plotting
    plot_df = pd.DataFrame({
        'x': embedding[:, 0],
        'y': embedding[:, 1],
        'Cluster': [f'Cluster {l}' for l in labels],
    })
    
    # Add document labels and text
    if doc_labels is not None:
        plot_df['Label'] = doc_labels
    else:
        plot_df['Label'] = [f'Doc {i}' for i in range(len(labels))]
    
    if docs_text is not None:
        plot_df['Display Text'] = [text[:50] + '...' if len(text) > 50 else text 
                                 for text in docs_text]
        plot_df['Full Text'] = docs_text
    else:
        plot_df['Display Text'] = plot_df['Label']
        plot_df['Full Text'] = plot_df['Label']
    
    # Create base scatter plot
    fig = px.scatter(
        plot_df,
        x='x',
        y='y',
        color='Cluster',
        hover_name='Label',
        hover_data={
            'x': False,
            'y': False,
            'Display Text': True,
            'Full Text': True
        },
        title=f'Interactive {method.upper()} visualization of document clusters'
    )
    
    if show_text:
        fig.add_trace(
            go.Scatter(
                x=plot_df['x'],
                y=plot_df['y'],
                mode='text',
                text=plot_df['Display Text'],
                textposition="top center",
                textfont=dict(size=text_size),
                showlegend=False,
                hoverinfo='none'
            )
        )
    
    # Add feature arrows if using PCA and features requested
    if show_features and method.lower() == 'pca' and lda_model is not None:
        # Get feature loadings from PCA
        loadings = reducer.components_.T
        
        # Add arrows for each topic
        for i in range(loadings.shape[0]):
            fig.add_trace(
                go.Scatter(
                    x=[0, loadings[i, 0] * 3],  # Scale arrow length
                    y=[0, loadings[i, 1] * 3],
                    mode='lines+text',
                    name=f'Topic {i}',
                    line=dict(color='red', width=1),
                    text=['']*1 + [f'Topic {i}'],
                    textposition='top center',
                    showlegend=False
                )
            )
    
    # Update layout
    fig.update_layout(
        title_x=0.5,
        plot_bgcolor='white',
        width=1200,
        height=800,
        showlegend=True,
        margin=dict(l=50, r=50, t=50, b=50)
    )
    
    fig.update_traces(
        selector=dict(mode='markers'),
        marker=dict(size=8, opacity=0.7)
    )
    
    return fig, plot_df

def analyze_clusters(doc_topic_matrix, labels, docs_text=None, doc_labels=None, lda_model=None, top_n_docs=5):
    """
    Analyze cluster contents and characteristics.
    
    Args:
        doc_topic_matrix: Matrix of document-topic distributions
        labels: Cluster assignments
        docs_text: List of document texts
        doc_labels: List of document labels
        lda_model: LDA model for topic analysis
        top_n_docs: Number of top documents to show per cluster
    """
    analysis = defaultdict(dict)
    
    for cluster_id in np.unique(labels):
        # Get indices of documents in this cluster
        cluster_indices = np.where(labels == cluster_id)[0]
        
        # Get cluster documents
        cluster_docs = doc_topic_matrix[cluster_indices]
        
        # Basic statistics
        analysis[f'Cluster {cluster_id}'] = {
            'size': len(cluster_indices),
            'proportion': len(cluster_indices) / len(labels)
        }
        
        # Average topic distribution for cluster
        avg_topics = np.mean(cluster_docs, axis=0)
        top_topics = np.argsort(avg_topics)[::-1][:3]
        
        analysis[f'Cluster {cluster_id}']['dominant_topics'] = [
            (topic_id, avg_topics[topic_id])
            for topic_id in top_topics
        ]
        
        # Top documents in cluster (by centrality)
        cluster_center = np.mean(cluster_docs, axis=0)
        doc_distances = np.linalg.norm(cluster_docs - cluster_center, axis=1)
        top_doc_indices = np.argsort(doc_distances)[:top_n_docs]
        
        # Store representative documents
        rep_docs = []
        for idx in top_doc_indices:
            doc_info = {
                'index': cluster_indices[idx],
                'distance': doc_distances[idx]
            }
            if doc_labels is not None:
                doc_info['label'] = doc_labels[cluster_indices[idx]]
            if docs_text is not None:
                doc_info['text'] = docs_text[cluster_indices[idx]]
            rep_docs.append(doc_info)
            
        analysis[f'Cluster {cluster_id}']['representative_docs'] = rep_docs
    
    return analysis

def print_cluster_analysis(analysis, lda_model=None):
    """Pretty print cluster analysis results."""
    for cluster, info in analysis.items():
        print(f"\n{'='*50}")
        print(f"{cluster}:")
        print(f"Size: {info['size']} documents ({info['proportion']*100:.1f}% of total)")
        
        print("\nDominant Topics:")
        for topic_id, weight in info['dominant_topics']:
            print(f"- Topic {topic_id} (weight: {weight:.3f})")
            if lda_model is not None:
                top_words = lda_model.show_topic(topic_id, topn=5)
                print(f"  Top words: {', '.join(word for word, _ in top_words)}")
        
        print("\nRepresentative Documents:")
        for i, doc in enumerate(info['representative_docs'], 1):
            print(f"\n{i}.", end=' ')
            if 'label' in doc:
                print(f"Label: {doc['label']}")
            if 'text' in doc:
                text = doc['text'][:100] + '...' if len(doc['text']) > 100 else doc['text']
                print(f"Text: {text}")

def save_visualization(fig, filename, formats=None):
    """
    Save visualization in multiple formats.
    
    Args:
        fig: Plotly figure object
        filename: Base filename (without extension)
        formats: List of formats to save (default: ['html', 'png', 'svg'])
    """
    if formats is None:
        formats = ['html', 'png', 'svg']
    
    for fmt in formats:
        if fmt == 'html':
            fig.write_html(f"{filename}.html")
        elif fmt == 'png':
            fig.write_image(f"{filename}.png")
        elif fmt == 'svg':
            fig.write_image(f"{filename}.svg")
        else:
            print(f"Unsupported format: {fmt}")

# Example usage:
"""
# Create and save visualization
fig, plot_df = create_interactive_cluster_viz(
    doc_topic_matrix_words,
    words_labels,
    docs_text=texts_raw,
    show_text=True,
    text_size=8,
    jitter_scale=0.02,  # Add some jitter to spread out dense clusters
    show_features=True,  # Show feature arrows (only works with PCA)
    lda_model=lda_model_words,
    method='pca'  # Use PCA if you want feature arrows
)

# Save in multiple formats
save_visualization(fig, 'topic_clusters', formats=['html', 'png', 'svg'])

# Analyze clusters
analysis = analyze_clusters(
    doc_topic_matrix_words,
    words_labels,
    docs_text=texts_raw,
    doc_labels=doc_labels,
    lda_model=lda_model_words
)

# Print analysis results
print_cluster_analysis(analysis, lda_model=lda_model_words)
"""

def truncate_text(text, max_length=50):
    """Helper function to truncate text to a reasonable length."""
    if len(text) <= max_length:
        return text
    return text[:max_length] + '...'
# Example usage:
# Assuming you have:
# - doc_topic_matrix from previous code
# - labels from KMeans clustering
# - Optional: list of document texts in docs_text

# For static visualization:
# visualize_document_clusters(doc_topic_matrix, labels, method='tsne')

# For interactive visualization:
# fig = create_interactive_cluster_viz(
#     doc_topic_matrix, 
#     labels,
#     docs_text=docs_text,  # Optional
#     method='tsne'
# )
# fig.show()

In [ ]:
corpus_gensim_words = GensimCorpus.load(input_prefix="gensim_words",directory="corpora")
corpus_gensim_lemmas = GensimCorpus.load(input_prefix="gensim_lemmas",directory="corpora")

In [ ]:
# First, get your document-topic matrix and cluster labels
doc_topic_matrix_words = get_document_topics_matrix(lda_model_words, corpus_gensim_words.corpus)
doc_topic_matrix_lemmas = get_document_topics_matrix(lda_model_lemmas, corpus_gensim_lemmas.corpus)

words_kmeans, words_labels, words_silhouette = cluster_documents(doc_topic_matrix_words, n_clusters=8)
lemmas_kmeans, lemmas_labels, lemmas_silhouette = cluster_documents(doc_topic_matrix_lemmas, n_clusters=8)

# For static visualization
#visualize_document_clusters(doc_topic_matrix, labels, method='tsne')

In [ ]:
# Create visualization with all enhancements
fig, plot_df = create_interactive_cluster_viz(
    doc_topic_matrix_words,
    words_labels,
    doc_labels=[doc.replace('_TEI_final.conllu', '') for doc in corpus_gensim_words.successful_files],
    text_size=8,
    jitter_scale=0.02,  # Adjust this value to control point spread
    show_features=True,  # Show topic arrows
    lda_model=lda_model_words,
    method='pca'  # Use PCA for feature arrows
)
save_visualization(fig, 'out/topic_modelling/all_words_pca')

In [ ]:
# Create visualization with all enhancements
fig, plot_df = create_interactive_cluster_viz(
    doc_topic_matrix_words,
    words_labels,
    doc_labels=[doc.replace('_TEI_final.conllu', '') for doc in corpus_gensim_words.successful_files],
    text_size=8,
    jitter_scale=0.02,  # Adjust this value to control point spread
    show_features=True,  # Show topic arrows
    lda_model=lda_model_words,
    method='tsne'  # Use PCA for feature arrows
)
save_visualization(fig, 'out/topic_modelling/all_words_tsne')

In [ ]:
# Analyze clusters
analysis = analyze_clusters(
    doc_topic_matrix_words,
    words_labels,
    doc_labels=[doc.replace('_TEI_final.conllu', '') for doc in corpus_gensim_words.successful_files],
    #docs_text=texts_raw,
    lda_model=lda_model_words
)

# Print detailed analysis
print_cluster_analysis(analysis, lda_model=lda_model_words)

In [ ]:
# Create visualization with all enhancements
fig, plot_df = create_interactive_cluster_viz(
    doc_topic_matrix_lemmas,
    lemmas_labels,
    doc_labels=[doc.replace('_TEI_final.conllu', '') for doc in corpus_gensim_lemmas.successful_files],
    text_size=8,
    jitter_scale=0.02,  # Adjust this value to control point spread
    show_features=True,  # Show topic arrows
    lda_model=lda_model_lemmas,
    method='tsne'  # Use PCA for feature arrows
)
save_visualization(fig, 'out/topic_modelling/all_lemmas_tsne')

In [ ]:
# Create visualization with all enhancements
fig, plot_df = create_interactive_cluster_viz(
    doc_topic_matrix_lemmas,
    lemmas_labels,
    doc_labels=[doc.replace('_TEI_final.conllu', '') for doc in corpus_gensim_lemmas.successful_files],
    text_size=8,
    jitter_scale=0.02,  # Adjust this value to control point spread
    show_features=True,  # Show topic arrows
    lda_model=lda_model_lemmas,
    method='pca'  # Use PCA for feature arrows
)
save_visualization(fig, 'out/topic_modelling/all_lemmas_pca')

In [ ]:
# Analyze clusters
analysis = analyze_clusters(
    doc_topic_matrix_lemmas,
    lemmas_labels,
    doc_labels=[doc.replace('_TEI_final.conllu', '') for doc in corpus_gensim_lemmas.successful_files],
    #docs_text=texts_raw,
    lda_model=lda_model_lemmas
)

# Print detailed analysis
print_cluster_analysis(analysis, lda_model=lda_model_lemmas)

#### Only books

In [ ]:
import re
doc_pattern=re.compile('Ksg[^H]|AKap|PomnLw|StPPP|AGZ|Lib')
fileids = [ doc for doc in corpus_gensim_words.successful_files if doc_pattern.match(doc) ]

In [ ]:
#corpus_gensim_words_books = create_subcorpus(corpus_gensim_words,fileids=fileids)
#corpus_gensim_lemmas_books = create_subcorpus(corpus_gensim_lemmas,fileids=fileids)

In [ ]:
from gensim.models import LdaModel
import numpy as np
import re
from typing import List, Dict, Union, Optional

def filter_gensim_corpus(corpus_gensim, 
                        file_list: Optional[List[str]] = None,
                        file_pattern: Optional[str] = None,
                        topic_threshold: Optional[Dict[int, float]] = None,
                        lda_model: Optional[LdaModel] = None) -> Dict:
    """
    Filter a loaded GensimCorpus object and prepare it for LDA analysis.
    
    Args:
        corpus_gensim: Loaded GensimCorpus object
        file_list: List of specific files to include
        file_pattern: Regex pattern to match filenames
        topic_threshold: Dict of {topic_id: min_probability} for filtering
        lda_model: LDA model for topic-based filtering
    
    Returns:
        Dict containing filtered indices and corpus
    """
    # Get all available files
    available_files = corpus_gensim.successful_files
    
    # Step 1: Filter files
    if file_list is not None:
        selected_files = [f for f in file_list if f in available_files]
    elif file_pattern is not None:
        pattern = re.compile(file_pattern)
        selected_files = [f for f in available_files if pattern.search(f)]
    else:
        selected_files = available_files
        
    print(f"Selected {len(selected_files)} files out of {len(available_files)}")
    
    # Step 2: Get document indices based on mode
    if corpus_gensim.mode == 'doc':
        # In document mode, each file is a document
        doc_indices = [i for i, f in enumerate(available_files) if f in selected_files]
    else:  # sent mode
        # In sentence mode, use sentence_metadata to find relevant indices
        doc_indices = [
            i for i, (file, _) in enumerate(corpus_gensim.sentence_metadata)
            if file in selected_files
        ]
    
    # Step 3: Get the subset of corpus
    subset_corpus = [
        corpus_gensim.corpus[i] for i in doc_indices
    ]
    
    # Step 4: Apply topic filtering if requested
    if topic_threshold and lda_model:
        filtered_indices = []
        filtered_corpus = []
        
        for i, doc in zip(doc_indices, subset_corpus):
            topic_dist = dict(lda_model.get_document_topics(doc, minimum_probability=0))
            meets_threshold = all(
                topic_dist.get(topic_id, 0) >= min_prob
                for topic_id, min_prob in topic_threshold.items()
            )
            if meets_threshold:
                filtered_indices.append(i)
                filtered_corpus.append(doc)
                
        doc_indices = filtered_indices
        subset_corpus = filtered_corpus
        
    print(f"Final subset contains {len(subset_corpus)} documents")
    
    # Return both indices and corpus for further analysis
    return {
        'indices': doc_indices,
        'corpus': subset_corpus,
        'selected_files': selected_files
    }

def analyze_clusters(lda_model, doc_topic_matrix, labels):
    """
    Analyze the characteristics of each cluster.
    
    Args:
        lda_model: Trained LDA model
        doc_topic_matrix: Matrix of document-topic distributions (numpy array)
        labels: Cluster assignments
        
    Returns:
        Dictionary with cluster analysis results
    """
    cluster_analysis = defaultdict(dict)
    
    for cluster_id in np.unique(labels):
        # Get mask for documents in this cluster
        cluster_mask = labels == cluster_id
        
        # Get documents in this cluster
        cluster_docs = doc_topic_matrix[cluster_mask]
        
        if len(cluster_docs) == 0:
            continue
            
        # Calculate average topic distribution for cluster
        avg_topic_dist = np.mean(cluster_docs, axis=0)
        
        # Find dominant topics
        # Make sure avg_topic_dist is 1D array
        if avg_topic_dist.ndim > 1:
            avg_topic_dist = avg_topic_dist.flatten()
            
        dominant_topics = sorted(
            enumerate(avg_topic_dist),
            key=lambda x: x[1],
            reverse=True
        )[:3]
        
        # Store results
        cluster_analysis[f'Cluster {cluster_id}'] = {
            'size': len(cluster_docs),
            'dominant_topics': [
                (topic_id, prob, lda_model.show_topic(topic_id, topn=5))
                for topic_id, prob in dominant_topics
            ]
        }
    
    return cluster_analysis

def print_cluster_analysis(analysis, lda_model=None):
    """
    Pretty print cluster analysis results.
    
    Args:
        analysis: Dictionary from analyze_clusters()
        lda_model: Optional LDA model for showing topic words
    """
    if not analysis:
        print("No clusters to analyze.")
        return
        
    total_docs = sum(info['size'] for info in analysis.values())
    
    for cluster, info in analysis.items():
        print(f"\n{'='*50}")
        print(f"{cluster}:")
        print(f"Size: {info['size']} documents ({(info['size']/total_docs)*100:.1f}% of total)")
        
        print("\nDominant Topics:")
        for topic_id, prob, topic_words in info['dominant_topics']:
            print(f"- Topic {topic_id} (weight: {prob:.3f})")
            if topic_words:  # Only print if we have topic words
                word_probs = ', '.join(f"{word} ({prob:.3f})" for word, prob in topic_words)
                print(f"  Top words: {word_probs}")

def analyze_gensim_subset(corpus_gensim, lda_model, 
                         file_list=None, file_pattern=None, 
                         topic_threshold=None, n_clusters=5,
                         viz_methods=['pca', 'tsne'],
                         output_dir='visualizations',
                         filename_prefix='cluster_viz',
                         perplexity=30,
                         show_arrows=True,
                         custom_doc_names=None):
    """
    Complete analysis pipeline for a GensimCorpus subset.
    
    Args:
        corpus_gensim: Loaded GensimCorpus object
        lda_model: Trained LDA model
        file_list: List of files to include
        file_pattern: Regex pattern for files
        topic_threshold: Dict of {topic_id: min_probability}
        n_clusters: Number of clusters
        viz_methods: List of visualization methods ['pca', 'tsne']
        output_dir: Directory to save visualizations
        filename_prefix: Prefix for saved files
        perplexity: Perplexity parameter for t-SNE
        show_arrows: Whether to show feature arrows in PCA plot
        custom_doc_names: Optional list of custom names for documents
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get filtered subset
    subset = filter_gensim_corpus(
        corpus_gensim,
        file_list=file_list,
        file_pattern=file_pattern,
        topic_threshold=topic_threshold,
        lda_model=lda_model
    )
    
    if len(subset['corpus']) == 0:
        raise ValueError("No documents found in subset")
    
    # Get topic distributions
    doc_topic_matrix = get_document_topics_matrix(lda_model, subset['corpus'])
    
    if len(doc_topic_matrix) < n_clusters:
        n_clusters = max(2, len(doc_topic_matrix) // 2)
        print(f"Reducing number of clusters to {n_clusters} due to small sample size")
    
    # Perform clustering
    kmeans, labels, silhouette = cluster_documents(doc_topic_matrix, n_clusters)
    print(f"Silhouette Score: {silhouette:.3f}")
    
    # Create document labels
    if custom_doc_names is not None:
        if len(custom_doc_names) != len(subset['indices']):
            raise ValueError(f"Number of custom names ({len(custom_doc_names)}) "
                           f"doesn't match number of documents ({len(subset['indices'])})")
        doc_labels = custom_doc_names
    else:
        if corpus_gensim.mode == 'doc':
            doc_labels = [f"Doc_{i}_{f}" for i, f in zip(subset['indices'], subset['selected_files'])]
        else:
            doc_labels = [
                f"Doc_{i}_{corpus_gensim.sentence_metadata[idx][0]}_{corpus_gensim.sentence_metadata[idx][1]}"
                for i, idx in enumerate(subset['indices'])
            ]
    
    # Create visualizations
    visualizations = {}
    for method in viz_methods:
        try:
            # Create interactive visualization
            fig, plot_df = create_interactive_cluster_viz(
                doc_topic_matrix,
                labels,
                doc_labels=doc_labels,
                show_text=True,
                text_size=8,
                jitter_scale=0.02,
                show_features=(method == 'pca' and show_arrows),
                lda_model=lda_model,
                method=method,
                perplexity=perplexity
            )
            
            # Save visualization
            base_filename = os.path.join(output_dir, f'{filename_prefix}_{method}')
            save_visualization(fig, base_filename)
            visualizations[method] = {'figure': fig, 'plot_df': plot_df}
            
            # Create static visualization
            static_fig = visualize_document_clusters(
                doc_topic_matrix, 
                labels, 
                method=method,
                perplexity=perplexity
            )
            static_fig.savefig(os.path.join(output_dir, f'{filename_prefix}_{method}_static.png'))
            plt.close(static_fig)
            
        except Exception as e:
            print(f"Error creating {method} visualization: {str(e)}")
            continue
    
    # Analyze clusters
    try:
        cluster_analysis = analyze_clusters(lda_model, doc_topic_matrix, labels)
    except Exception as e:
        print(f"Error in cluster analysis: {str(e)}")
        cluster_analysis = None
    
    # Plot cluster sizes
    try:
        plt.figure(figsize=(10, 6))
        plot_cluster_sizes(labels)
        plt.savefig(os.path.join(output_dir, f'{filename_prefix}_cluster_sizes.png'))
        plt.close()
    except Exception as e:
        print(f"Error plotting cluster sizes: {str(e)}")
    
    return {
        'subset': subset,
        'doc_topic_matrix': doc_topic_matrix,
        'cluster_labels': labels,
        'silhouette_score': silhouette,
        'visualizations': visualizations,
        'cluster_analysis': cluster_analysis,
        'doc_labels': doc_labels
    }

# Example usage:
"""
# With custom document names
doc_names = [
    "Introduction_ch1",
    "Methods_ch2",
    "Results_ch3",
    # ... more names ...
]

results = analyze_gensim_subset(
    corpus_gensim_words,
    lda_model_words,
    file_pattern=r'your_pattern.*\.txt',
    n_clusters=5,
    viz_methods=['pca', 'tsne'],
    output_dir='outputs/analysis_1',
    filename_prefix='manuscript_analysis',
    perplexity=25,
    show_arrows=False,
    custom_doc_names=doc_names
)

# Default usage without custom names
results = analyze_gensim_subset(
    corpus_gensim_words,
    lda_model_words,
    file_pattern=r'your_pattern.*\.txt',
    n_clusters=5,
    viz_methods=['pca', 'tsne'],
    output_dir='outputs/analysis_1',
    filename_prefix='default_analysis',
    perplexity=30,
    show_arrows=True
)
"""

In [ ]:
def calculate_perplexity(n_samples):
    """
    Calculate appropriate perplexity value based on sample size.
    
    Args:
        n_samples: Number of documents
        
    Returns:
        float: Appropriate perplexity value
    """
    # t-SNE typically works well with perplexity between 5 and 50
    # Perplexity must be less than n_samples
    suggested_perplexity = min(30, n_samples - 1)  # Default is 30
    return max(5, suggested_perplexity)  # Don't go below 5

def create_interactive_cluster_viz(doc_topic_matrix, labels, doc_labels=None, docs_text=None, 
                                 method='tsne', show_text=True, text_size=10, 
                                 show_features=False, lda_model=None,
                                 jitter_scale=0.02, perplexity=30):
    """
    Create enhanced interactive visualization with jitter and optional feature arrows.
    
    Args:
        doc_topic_matrix: Matrix of document-topic distributions
        labels: Cluster assignments
        doc_labels: List of labels for each document
        docs_text: List of full document texts
        method: 'tsne' or 'pca'
        show_text: Whether to show text labels
        text_size: Size of text labels
        show_features: Whether to show feature arrows (only works with PCA)
        lda_model: LDA model (required if show_features=True)
        jitter_scale: Amount of random jitter to add
        perplexity: Perplexity parameter for t-SNE
    """
    # Reduce dimensionality
    if method.lower() == 'tsne':
        # Adjust perplexity if needed
        actual_perplexity = min(perplexity, len(doc_topic_matrix) - 1)
        if actual_perplexity != perplexity:
            print(f"Adjusting perplexity from {perplexity} to {actual_perplexity} due to sample size")
            
        reducer = TSNE(
            n_components=2, 
            random_state=42,
            perplexity=actual_perplexity
        )
        embedding = reducer.fit_transform(doc_topic_matrix)
        if show_features:
            print("Feature arrows only available with PCA method")
            show_features = False
    else:  # PCA
        reducer = PCA(n_components=2, random_state=42)
        embedding = reducer.fit_transform(doc_topic_matrix)
    
    # Add jitter to prevent overlap
    if jitter_scale > 0:
        jitter = np.random.normal(0, jitter_scale, embedding.shape)
        embedding = embedding + jitter
    
    # Create DataFrame for plotting
    plot_df = pd.DataFrame({
        'x': embedding[:, 0],
        'y': embedding[:, 1],
        'Cluster': [f'Cluster {l}' for l in labels],
    })
    
    # Add document labels and text
    if doc_labels is not None:
        plot_df['Label'] = doc_labels
    else:
        plot_df['Label'] = [f'Doc {i}' for i in range(len(labels))]
    
    if docs_text is not None:
        plot_df['Display Text'] = [text[:50] + '...' if len(text) > 50 else text 
                                 for text in docs_text]
        plot_df['Full Text'] = docs_text
    else:
        plot_df['Display Text'] = plot_df['Label']
        plot_df['Full Text'] = plot_df['Label']
    
    # Create base scatter plot
    fig = px.scatter(
        plot_df,
        x='x',
        y='y',
        color='Cluster',
        hover_name='Label',
        hover_data={
            'x': False,
            'y': False,
            'Display Text': True,
            'Full Text': True
        },
        title=f'Interactive {method.upper()} visualization of document clusters'
    )
    
    if show_text:
        fig.add_trace(
            go.Scatter(
                x=plot_df['x'],
                y=plot_df['y'],
                mode='text',
                text=plot_df['Display Text'],
                textposition="top center",
                textfont=dict(size=text_size),
                showlegend=False,
                hoverinfo='none'
            )
        )
    
    # Add feature arrows if using PCA and features requested
    if show_features and method.lower() == 'pca' and lda_model is not None:
        loadings = reducer.components_.T
        for i in range(loadings.shape[0]):
            fig.add_trace(
                go.Scatter(
                    x=[0, loadings[i, 0] * 3],
                    y=[0, loadings[i, 1] * 3],
                    mode='lines+text',
                    name=f'Topic {i}',
                    line=dict(color='red', width=1),
                    text=['']*1 + [f'Topic {i}'],
                    textposition='top center',
                    showlegend=False
                )
            )
    
    # Update layout
    fig.update_layout(
        title_x=0.5,
        plot_bgcolor='white',
        width=1200,
        height=800,
        showlegend=True,
        margin=dict(l=50, r=50, t=50, b=50)
    )
    
    fig.update_traces(
        selector=dict(mode='markers'),
        marker=dict(size=8, opacity=0.7)
    )
    
    return fig, plot_df

# Also update the static visualization
def visualize_document_clusters(doc_topic_matrix, labels, method='tsne', plot_size=(15, 10), perplexity=30):
    """
    Visualize document clusters using dimensionality reduction.
    
    Args:
        doc_topic_matrix: Matrix of document-topic distributions
        labels: Cluster assignments from KMeans
        method: 'tsne' or 'pca' for dimensionality reduction
        plot_size: Tuple of (width, height) for the plot
        perplexity: Perplexity parameter for t-SNE
    """
    # Set up the plot style
    plt.figure(figsize=plot_size)
    sns.set_style("whitegrid")
    
    # Reduce dimensionality
    if method.lower() == 'tsne':
        # Adjust perplexity if needed
        actual_perplexity = min(perplexity, len(doc_topic_matrix) - 1)
        if actual_perplexity != perplexity:
            print(f"Adjusting perplexity from {perplexity} to {actual_perplexity} due to sample size")
            
        reducer = TSNE(
            n_components=2, 
            random_state=42,
            perplexity=actual_perplexity
        )
        embedding = reducer.fit_transform(doc_topic_matrix)
        title = f't-SNE visualization of document clusters (perplexity={actual_perplexity})'
    else:  # PCA
        reducer = PCA(n_components=2, random_state=42)
        embedding = reducer.fit_transform(doc_topic_matrix)
        title = 'PCA visualization of document clusters'
    
    # Create scatter plot with direct legend
    unique_labels = np.unique(labels)
    for label in unique_labels:
        mask = labels == label
        plt.scatter(
            embedding[mask, 0], 
            embedding[mask, 1],
            label=f'Cluster {label}',
            alpha=0.6
        )
    
    # Add labels
    plt.title(title, fontsize=14, pad=20)
    plt.xlabel(f'{method.upper()} Component 1', fontsize=12)
    plt.ylabel(f'{method.upper()} Component 2', fontsize=12)
    
    # Add legend
    plt.legend(title="Clusters", bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    return plt.gcf()

In [ ]:
doc_names = [doc.replace('_TEI_final.conllu', '') for doc in corpus_gensim_words.successful_files]
doc_labels = [doc.replace('_TEI_final.conllu', '') for doc in fileids]

books_words_results = analyze_gensim_subset(
    corpus_gensim_words,
    lda_model_words,
    file_list=fileids,
    n_clusters=3,
    viz_methods=['pca', 'tsne'],
    output_dir='out/topic_modelling',
    filename_prefix='books_words',
    perplexity=5,
    show_arrows=False,
    custom_doc_names=doc_labels
)

In [ ]:
doc_names = [doc.replace('_TEI_final.conllu', '') for doc in corpus_gensim_words.successful_files]
doc_labels = [doc.replace('_TEI_final.conllu', '') for doc in fileids]

books_lemmas_results = analyze_gensim_subset(
    corpus_gensim_lemmas,
    lda_model_lemmas,
    file_list=fileids,
    n_clusters=3,
    viz_methods=['pca', 'tsne'],
    output_dir='out/topic_modelling',
    filename_prefix='books_lemmas',
    perplexity=5,
    show_arrows=False,
    custom_doc_names=doc_labels
)

In [ ]:
# Analyze subset of files
results_lemmas_books = analyze_gensim_subset(
    corpus_gensim_lemmas,
    lda_model_lemmas,
    file_list=fileids,
    n_clusters=5
)
save_visualization(results_lemmas_books['visualization'], 'books_topic_clusters_lemmas')
print_cluster_analysis(results_lemmas_books['cluster_analysis'], lda_model=lda_model_lemmas)

In [ ]:
# Example usage:
"""
# By file list:
file_list = ['doc1.txt', 'doc2.txt', 'doc3.txt']
# Or by pattern:
pattern = r'2023-.*\.txt'  # matches files from 2023

results = analyze_subset(
    lda_model_words,
    texts,  # your original texts
    dictionary,  # your original dictionary
    file_list=file_list,  # or pattern=pattern
    n_clusters=5
)

# Save visualization
save_visualization(results['visualization'], 'subset_topic_clusters')

# Print analysis
print(f"\nAnalyzing subset of {results['subset_size']} documents")
print_cluster_analysis(results['cluster_analysis'], lda_model=lda_model_words)

# Access other results
subset_indices = results['subset_indices']  # original indices of subset documents
doc_topic_matrix = results['doc_topic_matrix']  # topic distributions for subset
cluster_labels = results['cluster_labels']  # cluster assignments for subset
"""

def compare_subset_to_full(full_doc_topic_matrix, subset_indices, full_labels=None):
    """
    Compare topic distributions in subset to full corpus.
    
    Args:
        full_doc_topic_matrix: Topic distribution matrix for full corpus
        subset_indices: Indices of documents in subset
        full_labels: Cluster labels for full corpus (optional)
    """
    # Calculate average topic distribution
    full_avg = np.mean(full_doc_topic_matrix, axis=0)
    subset_avg = np.mean(full_doc_topic_matrix[subset_indices], axis=0)
    
    # Calculate differences
    diff = subset_avg - full_avg
    
    # Create comparison DataFrame
    comparison = pd.DataFrame({
        'Topic': range(len(full_avg)),
        'Full_Corpus': full_avg,
        'Subset': subset_avg,
        'Difference': diff
    })
    
    # Sort by absolute difference
    comparison['Abs_Diff'] = abs(comparison['Difference'])
    comparison = comparison.sort_values('Abs_Diff', ascending=False)
    
    # If cluster labels provided, compare cluster distributions
    if full_labels is not None:
        full_clusters = np.bincount(full_labels) / len(full_labels)
        subset_clusters = np.bincount(full_labels[subset_indices]) / len(subset_indices)
        
        cluster_comparison = pd.DataFrame({
            'Cluster': range(len(full_clusters)),
            'Full_Corpus': full_clusters,
            'Subset': subset_clusters,
            'Difference': subset_clusters - full_clusters
        })
        
        return comparison, cluster_comparison
    
    return comparison

### Visualize

In [ ]:
#Visualize topics - words
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import matplotlib.pyplot as plt
# Prepare the visualization
dictionary = corpus_gensim_words_dict
dictionary.filter_extremes(no_below=5, no_above=0.5)
lda_vis_data = gensimvis.prepare(lda_model_words, 
                                 corpus_gensim_words,                                 
                                 dictionary)

# Display the interactive visualization
pyLDAvis.display(lda_vis_data)

# Alternatively, you can save the visualization to an HTML file:
pyLDAvis.save_html(lda_vis_data, 'out/lda_topics_words_visualization.html')

In [ ]:
#Visualize topics - words
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import matplotlib.pyplot as plt
# Prepare the visualization
dictionary = corpus_gensim_lemmas_dict
dictionary.filter_extremes(no_below=5, no_above=0.5)
lda_vis_data = gensimvis.prepare(lda_model_lemmas, 
                                 corpus_gensim_lemmas,                                 
                                 dictionary)

# Display the interactive visualization
pyLDAvis.display(lda_vis_data)

# Alternatively, you can save the visualization to an HTML file:
pyLDAvis.save_html(lda_vis_data, 'out/lda_topics_lemmas_visualization.html')

## Similarity

### Build Model

In [ ]:
from gensim.similarities import Similarity
from gensim.corpora import Dictionary

from gensim.test.utils import get_tmpfile

index_tmpfile = get_tmpfile("index")
index = Similarity(index_tmpfile, corpus_gensim_lemmas, num_features=len(Dictionary.load('corpora/gensim_lemmas.dict')))  # build the index

In [ ]:
import numpy as np

# Create an empty matrix to store similarities
num_docs = len(corpus_gensim_lemmas)
similarity_matrix = np.zeros((num_docs, num_docs))

# Compute similarity between each document and every other document
for i, doc in enumerate(corpus_gensim_lemmas):
    similarities = index[doc]  # Get similarity of doc i with all other documents
    similarity_matrix[i] = similarities  # Store the similarities in the matrix

### Visualize

In [ ]:
import networkx as nx
from pyvis.network import Network

# Create a graph
G = nx.Graph()
document_names = corpus_files

# Add nodes and edges based on the similarity matrix
threshold = 0.5  # Set a threshold for visualization
for i in range(len(similarity_matrix)):
    for j in range(len(similarity_matrix)):
        if similarity_matrix[i, j] > threshold and i != j:
            G.add_edge(document_names[i], document_names[j], weight=similarity_matrix[i, j])

# Create a Pyvis network object
net = Network(notebook=True)

# Convert the NetworkX graph to a Pyvis network
net.from_nx(G)

# Show the interactive graph
net.show("out/docs_similarity.html")


## Lexical richness

### Compute lexical richness measures

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
import numpy as np
from collections import Counter
import math
from typing import List, Dict, Union
import warnings

class LexicalRichness:
    """
    A class to calculate various lexical richness measures using gensim corpus.
    """
    def __init__(self, texts: List[List[str]], min_words: int = 50):
        """
        Initialize with a list of tokenized texts.
        
        Args:
            texts: List of tokenized texts (each text is a list of strings)
            min_words: Minimum number of words required for meaningful analysis
        """
        self.min_words = min_words
        self.texts = texts
        self.dictionary = Dictionary(texts)
        self.corpus = [self.dictionary.doc2bow(text) for text in texts]
        
        # Pre-calculate token counts for efficiency
        self.token_counts = [len(text) for text in texts]
        self.type_counts = [len(set(text)) for text in texts]
        
    def _validate_text_length(self, index: int) -> bool:
        """Check if text meets minimum length requirement."""
        if self.token_counts[index] < self.min_words:
            warnings.warn(f"Text {index} has fewer than {self.min_words} words. Results may be unreliable.")
            return False
        return True
    
    def ttr(self, index: int) -> float:
        """
        Calculate Type-Token Ratio for a text.
        
        TTR = (number of unique words) / (total number of words)
        """
        if not self._validate_text_length(index):
            return None
            
        return self.type_counts[index] / self.token_counts[index]
    
    def root_ttr(self, index: int) -> float:
        """
        Calculate Root Type-Token Ratio.
        
        Root TTR = (number of unique words) / sqrt(total number of words)
        """
        if not self._validate_text_length(index):
            return None
            
        return self.type_counts[index] / math.sqrt(self.token_counts[index])
    
    def log_ttr(self, index: int) -> float:
        """
        Calculate Herdan's C (logarithmic TTR).
        
        Log TTR = log(number of unique words) / log(total number of words)
        """
        if not self._validate_text_length(index):
            return None
            
        return math.log10(self.type_counts[index]) / math.log10(self.token_counts[index])
    
    def mattr(self, index: int, window_size: int = 100) -> float:
        """
        Calculate Moving Average Type-Token Ratio.
        
        Args:
            index: Index of text to analyze
            window_size: Size of moving window
        """
        text = self.texts[index]
        if len(text) < window_size:
            warnings.warn(f"Text shorter than window size. Using full text TTR instead.")
            return self.ttr(index)
            
        # Calculate TTR for each window
        ttrs = []
        for i in range(len(text) - window_size + 1):
            window = text[i:i + window_size]
            ttrs.append(len(set(window)) / window_size)
        
        return np.mean(ttrs)
    
    def yules_k(self, index: int) -> float:
        """
        Calculate Yule's K measure.
        
        K = 10000 * (M2 - M1) / (M1 * M1)
        where M1 = number of unique words
        and M2 = sum of product of word frequency squared and number of words with that frequency
        """
        if not self._validate_text_length(index):
            return None
            
        text = self.texts[index]
        freq_dist = Counter(text)
        freq_of_freq = Counter(freq_dist.values())
        
        m1 = len(freq_dist)
        m2 = sum([freq ** 2 * freq_of_freq[freq] for freq in freq_of_freq])
        
        return 10000 * (m2 - m1) / (m1 * m1)
    
    def simpsons_d(self, index: int) -> float:
        """
        Calculate Simpson's D measure (lexical diversity).
        
        D = 1 - sum(n * (n-1)) / (N * (N-1))
        where n is frequency of each word type
        and N is total number of tokens
        """
        if not self._validate_text_length(index):
            return None
            
        text = self.texts[index]
        freq_dist = Counter(text)
        n = len(text)
        
        if n < 2:
            return 0
            
        sum_freq = sum(freq * (freq - 1) for freq in freq_dist.values())
        return 1 - (sum_freq / (n * (n - 1)))
    
    def get_all_metrics(self, index: int) -> Dict[str, float]:
        """
        Calculate all lexical richness measures for a given text.
        
        Returns:
            Dictionary containing all metrics
        """
        return {
            'ttr': self.ttr(index),
            'root_ttr': self.root_ttr(index),
            'log_ttr': self.log_ttr(index),
            'mattr': self.mattr(index),
            'yules_k': self.yules_k(index),
            'simpsons_d': self.simpsons_d(index)
        }
    
    def compare_texts(self, index1: int, index2: int) -> Dict[str, Dict[str, float]]:
        """
        Compare lexical richness measures between two texts.
        
        Returns:
            Dictionary containing metrics for both texts
        """
        metrics1 = self.get_all_metrics(index1)
        metrics2 = self.get_all_metrics(index2)
        
        return {
            f'text_{index1}': metrics1,
            f'text_{index2}': metrics2
        }

In [ ]:
lexrich_words = GensimLexicalRichness(corpus_gensim_words.corpus, corpus_gensim_words.dictionary)
lexrich_words_dict = {}
for i,f in enumerate(corpus_gensim_words.successful_files):
    txt = f.replace("_TEI_final.conllu", "")
    lexrich_words_dict[txt] = lexrich_words.get_all_metrics(i)
lexrich_words_df = pd.DataFrame.from_dict(lexrich_words_dict, orient="index")

In [ ]:
lexrich_lemmas = GensimLexicalRichness(corpus_gensim_lemmas.corpus, corpus_gensim_lemmas.dictionary)
lexrich_lemmas_dict = {}
for i,f in enumerate(corpus_gensim_lemmas.successful_files):
    txt = f.replace("_TEI_final.conllu", "")
    lexrich_lemmas_dict[txt] = lexrich_lemmas.get_all_metrics(i)
lexrich_lemmas_df = pd.DataFrame.from_dict(lexrich_lemmas_dict, orient="index")

### Visualize lexical richness measures

In [ ]:
class GensimLexicalRichnessViz(GensimLexicalRichness):
    """
    Extended class with visualization capabilities.
    """
    def __init__(self, corpus: CorpusABC, dictionary: Dictionary, 
                 labels: Optional[Union[List[str], Dict[int, str]]] = None,
                 min_words: int = 50):
        """Initialize with corpus, dictionary, and optional labels."""
        super().__init__(corpus, dictionary, min_words)
        self.labels = self._initialize_labels(labels, corpus)
        self.imputer = SimpleImputer(strategy='mean')
        
        # Define available visualizations
        self.available_plots = {
            'pca': 'PCA of All Measures',
            'distributions': 'Measure Distributions',
            'ttr_yule': 'TTR vs Yule\'s K',
            'length_ttr': 'Document Length vs TTR',
            'clustering_ttr': 'TTR-based Clustering',
            'clustering_yule': 'Yule\'s K-based Clustering',
            'clustering_hapax': 'Hapax Ratio Clustering',
            'clustering_all': 'Combined Measures Clustering'
        }
    
    def _initialize_labels(self, labels: Optional[Union[List[str], Dict[int, str]]], 
                          corpus: CorpusABC) -> Dict[int, str]:
        """Initialize document labels from various sources."""
        if labels is None:
            if hasattr(corpus, 'successful_files'):
                return {i: os.path.basename(f) for i, f in enumerate(corpus.successful_files)}
            else:
                return {i: f"Doc_{i}" for i in range(len(self.doc_lengths))}
        elif isinstance(labels, list):
            return {i: label for i, label in enumerate(labels)}
        elif isinstance(labels, dict):
            return labels
        else:
            raise ValueError("Labels must be None, a list, or a dictionary")
    
    def _handle_nan_values(self, df: pd.DataFrame, metrics: List[str]) -> np.ndarray:
        """Handle NaN values in the data using imputation."""
        X = df[metrics].values
        
        if np.isnan(X).any():
            print(f"Found NaN values in metrics: {metrics}")
            print("Using mean imputation for calculation")
            X_imputed = self.imputer.fit_transform(X)
        else:
            X_imputed = X
            
        return X_imputed
    
    def _prepare_metrics_dataframe(self, doc_indices: Optional[List[int]] = None) -> pd.DataFrame:
        """Create a DataFrame with all metrics and labels."""
        if doc_indices is None:
            doc_indices = range(len(self.doc_lengths))
            
        metrics_data = []
        for idx in doc_indices:
            metrics = self.get_all_metrics(idx)
            metrics['doc_id'] = idx
            metrics['label'] = self.labels[idx]
            metrics_data.append(metrics)
            
        df = pd.DataFrame(metrics_data)
        
        nan_counts = df.isna().sum()
        if nan_counts.any():
            print("\nNaN values found in the following metrics:")
            print(nan_counts[nan_counts > 0])
            
        return df
    
    def visualize_richness(self, 
                          doc_indices: Optional[List[int]] = None,
                          plots: Optional[List[str]] = None,
                          title: str = "Lexical Richness Analysis",
                          height_per_plot: int = 600) -> Tuple[go.Figure, pd.DataFrame]:
        """Create selected visualizations with improved readability."""
        if plots is None:
            plots = ['pca', 'distributions', 'ttr_yule', 'length_ttr']
        
        invalid_plots = set(plots) - set(self.available_plots.keys())
        if invalid_plots:
            raise ValueError(f"Invalid plot types: {invalid_plots}. "
                           f"Available options are: {list(self.available_plots.keys())}")
        
        df = self._prepare_metrics_dataframe(doc_indices)
        metrics_for_pca = ['ttr', 'root_ttr', 'log_ttr', 'yules_k', 'hapax_ratio']
        
        n_plots = len(plots)
        fig = make_subplots(
            rows=n_plots, cols=1,
            subplot_titles=[self.available_plots[plot] for plot in plots],
            vertical_spacing=0.2  # Increased spacing between plots
        )
        
        current_row = 1
        
        for plot_type in plots:
            if plot_type == 'pca':
                # PCA visualization with improved readability
                X = self._handle_nan_values(df, metrics_for_pca)
                scaler = StandardScaler()
                X_scaled = scaler.fit_transform(X)
                pca = PCA(n_components=2)
                X_pca = pca.fit_transform(X_scaled)
                explained_variance = pca.explained_variance_ratio_ * 100
                
                # Add scatter plot with better spacing
                fig.add_trace(
                    go.Scatter(
                        x=X_pca[:, 0],
                        y=X_pca[:, 1],
                        mode='markers+text',
                        text=df['label'],
                        name='Documents',
                        marker=dict(
                            size=12,
                            color='blue',
                            opacity=0.6
                        ),
                        textposition="top center",
                        textfont=dict(size=10),
                        hovertemplate=(
                            '%{text}<br>'
                            'PC1 (%{customdata[0]:.1f}% var): %{x:.3f}<br>'
                            'PC2 (%{customdata[1]:.1f}% var): %{y:.3f}'
                        ),
                        customdata=np.tile(explained_variance, (len(df), 1))
                    ),
                    row=current_row, col=1
                )
                
                fig.update_xaxes(title_text=f"PC1 ({explained_variance[0]:.1f}% variance)", row=current_row)
                fig.update_yaxes(title_text=f"PC2 ({explained_variance[1]:.1f}% variance)", row=current_row)
            
            elif plot_type == 'distributions':
                # Box plots with improved visibility
                for i, metric in enumerate(metrics_for_pca):
                    fig.add_trace(
                        go.Box(
                            y=df[metric],
                            name=metric,
                            boxpoints='outliers',  # Only show outliers as individual points
                            jitter=0.3,
                            pointpos=-1.8,
                            marker_color=px.colors.qualitative.Set3[i],
                            hovertemplate=(
                                'Measure: %{x}<br>'
                                'Value: %{y:.3f}'
                            )
                        ),
                        row=current_row, col=1
                    )
                
                fig.update_xaxes(title_text="Measure", row=current_row)
                fig.update_yaxes(title_text="Value", row=current_row)
            
            elif plot_type == 'ttr_yule':
                # TTR vs Yule's K with improved readability
                mask = df['ttr'].notna() & df['yules_k'].notna()
                fig.add_trace(
                    go.Scatter(
                        x=df.loc[mask, 'ttr'],
                        y=df.loc[mask, 'yules_k'],
                        mode='markers+text',
                        text=df.loc[mask, 'label'],
                        marker=dict(
                            size=12,
                            color='green',
                            opacity=0.6
                        ),
                        textposition="top center",
                        textfont=dict(size=10),
                        hovertemplate=(
                            '%{text}<br>'
                            'TTR: %{x:.3f}<br>'
                            "Yule's K: %{y:.3f}"
                        )
                    ),
                    row=current_row, col=1
                )
                
                fig.update_xaxes(title_text="TTR", row=current_row)
                fig.update_yaxes(title_text="Yule's K", row=current_row)
            
            elif plot_type == 'length_ttr':
                # Document length vs TTR with improved visibility
                fig.add_trace(
                    go.Scatter(
                        x=df['token_count'],
                        y=df['ttr'],
                        mode='markers+text',
                        text=df['label'],
                        marker=dict(
                            size=12,
                            color='purple',
                            opacity=0.6
                        ),
                        textposition="top center",
                        textfont=dict(size=10),
                        hovertemplate=(
                            '%{text}<br>'
                            'Tokens: %{x}<br>'
                            'TTR: %{y:.3f}'
                        )
                    ),
                    row=current_row, col=1
                )
                
                fig.update_xaxes(title_text="Document Length (tokens)", row=current_row)
                fig.update_yaxes(title_text="TTR", row=current_row)
            
            elif plot_type.startswith('clustering_'):
                measure = plot_type.split('_')[1]
                if measure == 'all':
                    # Improved clustering visualization for all measures
                    X = self._handle_nan_values(df, metrics_for_pca)
                    kmeans = KMeans(n_clusters=3, random_state=42)
                    clusters = kmeans.fit_predict(X)
                    
                    # Use PCA for visualization
                    pca = PCA(n_components=2)
                    X_pca = pca.fit_transform(X)
                    
                    # Add points for each cluster with better spacing
                    colors = px.colors.qualitative.Set3
                    for cluster in range(3):
                        mask = clusters == cluster
                        fig.add_trace(
                            go.Scatter(
                                x=X_pca[mask, 0],
                                y=X_pca[mask, 1],
                                mode='markers+text',
                                text=df.loc[mask, 'label'],
                                name=f'Cluster {cluster + 1}',
                                marker=dict(
                                    size=12,
                                    color=colors[cluster],
                                    opacity=0.6
                                ),
                                textposition="top center",
                                textfont=dict(size=10),
                                hovertemplate=(
                                    '%{text}<br>'
                                    'Cluster: %{text}'
                                )
                            ),
                            row=current_row, col=1
                        )
                else:
                    # Single measure clustering with improved layout
                    measure_map = {
                        'ttr': 'ttr',
                        'yule': 'yules_k',
                        'hapax': 'hapax_ratio'
                    }
                    if measure in measure_map:
                        cluster_fig = self._create_cluster_visualization(
                            df, 
                            measure_map[measure],
                            height_per_plot
                        )
                        for trace in cluster_fig.data:
                            fig.add_trace(trace, row=current_row, col=1)
            
            # Update layout for current subplot
            fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', row=current_row)
            fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', row=current_row)
            current_row += 1
        
        # Update overall layout
        fig.update_layout(
            title=title,
            height=height_per_plot * n_plots,
            width=800,
            showlegend=True,
            hovermode='closest',
            plot_bgcolor='white'
        )
        
        return fig, df

    def _create_cluster_visualization(self, df: pd.DataFrame, 
                                    measure: str, 
                                    height: int = 600,
                                    n_clusters: int = 3) -> go.Figure:
        """Create improved clustering visualization for a specific measure."""
        values = df[measure].values.reshape(-1, 1)
        values_clean = self._handle_nan_values(df, [measure])
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        clusters = kmeans.fit_predict(values_clean)
        
        fig = go.Figure()
        
        # Create array for cluster positions with more spacing
        cluster_positions = np.arange(n_clusters) * 2  # Double the spacing between clusters
        colors = px.colors.qualitative.Set3
        
        # Add points for each cluster
        for cluster in range(n_clusters):
            mask = clusters == cluster
            cluster_docs = np.where(mask)[0]
            
            # Add jitter to x-positions for better visibility
            jittered_x = np.random.normal(cluster_positions[cluster], 0.1, size=len(cluster_docs))
            
            fig.add_trace(go.Scatter(
                x=jittered_x.tolist(),
                y=values_clean[mask].flatten().tolist(),
                mode='markers+text',
                text=df.loc[mask, 'label'].tolist(),
                name=f'Cluster {cluster + 1}',
                marker=dict(
                    size=12,
                    color=colors[cluster],
                    opacity=0.6
                ),
                textposition="top center",
                textfont=dict(size=10),
                hovertemplate=(
                    '%{text}<br>'
                    f'{measure}: %{{y:.3f}}<br>'
                    'Cluster: %{text}'
                )
            ))
        
        # Add cluster centers
        fig.add_trace(go.Scatter(
            x=cluster_positions.tolist(),
            y=kmeans.cluster_centers_.flatten().tolist(),
            mode='markers',
            marker=dict(
                symbol='star',
                size=20,
                color='red',
                line=dict(color='black', width=2)
            ),
            name='Cluster Centers',
            showlegend=True
        ))
        
        # Update layout
        fig.update_layout(
            title=f"Document Clusters by {measure}",
            xaxis=dict(
                tickmode='array',
                ticktext=[f'Cluster {i+1}' for i in range(n_clusters)],
                tickvals=cluster_positions.tolist(),
                showgrid=True,
                gridcolor='LightGray'
            ),
            yaxis=dict(
                title=measure,
                showgrid=True,
                gridcolor='LightGray'
            ),
            height=height,
            plot_bgcolor='white',
            showlegend=True
        )
        
        return fig


In [ ]:
viz = GensimLexicalRichnessViz(corpus_gensim_words.corpus, corpus_gensim_words.dictionary, 
                               labels=[lbl.replace("_TEI_final.conllu","") for lbl in corpus_gensim_words.successful_files])
sel_plots=['pca', 'clustering_all', 'clustering_ttr', 'clustering_yule', 'clustering_hapax']
#, 'yules_k', 'hapax_ratio'
fig, df = viz.visualize_richness(plots=["clustering_ttr"], height_per_plot=800)   
#fig.update_layout(width=800,height=2000)
#fig.show()
fig.show()

In [ ]:
viz = GensimLexicalRichnessViz(corpus_gensim_lemmas.corpus, corpus_gensim_lemmas.dictionary, 
                               labels=[lbl.replace("_TEI_final.conllu","") for lbl in corpus_gensim_lemmas.successful_files])
fig_dashboard, df = viz.visualize_richness()
fig_dashboard.show()

## Phrases

### Build Model

In [ ]:
from conllu import parse
import re
import os

class GensimCorpusSents:
    def __init__(self, corpus_dir, 
                 corpus_files, 
                 stopwords_list=None, 
                 use_lemmas=False,
                 yield_dict=False # yield fileid:sents dictionary
                ):
        self.corpus_dir = corpus_dir
        self.corpus_files = corpus_files
        self.use_lemmas = use_lemmas
        
        # Set stopwords if provided, else use default
        self.stopwords_list = stopwords_list if stopwords_list else set(stopwords.words('english'))

        # Load the corpus
        self.corpus = self._load_corpus()

    def _load_corpus(self):
        """Load the CoNLL-U files into memory as sentences."""
        corpus = {}
        for file in self.corpus_files:
            with open(os.path.join(self.corpus_dir, file), "r", encoding="utf-8") as f:
                txt = f.read()
                sents = parse(txt)
                corpus[file] = sents
        return corpus

    def preprocess_text(self, text):
        """Preprocess a text string (token): lowercase, remove punctuation, numbers, and extra spaces."""
        text = text.lower()
        text = re.sub(r'[\d]', '', text)  # Remove numbers
        text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
        text = ' '.join(text.split())  # Remove extra whitespace
        return text

    def _conllu_iterator(self, text_list=None):
        """Iterate through the corpus and yield tokenized sentences.
        
        If `text_list` is provided, only sentences from the files in `text_list` will be yielded.
        If `text_list` is None, sentences from all corpus files will be yielded.
        """
        files_to_process = text_list if text_list else self.corpus_files
        texts_dict = dict.fromkeys(files_to_process)
        
        for fileid in files_to_process:
            sents = self.corpus[fileid]
            for sent in sents:
                tokens = [
                    token["lemma"] if self.use_lemmas and "lemma" in token else token["form"]
                    for token in sent if "form" in token
                ]
                tokens = [self.preprocess_text(token) for token in tokens]  # Preprocess tokens
                tokens = [token for token in tokens if token not in self.stopwords_list]  # Filter stopwords

                if tokens:  # Only yield non-empty sentences
                    yield tokens

    def get_sentences(self, text_list=None):
        """Return a list of tokenized sentences.
        
        If `text_list` is provided, it returns sentences from those specific files.
        Otherwise, it returns all sentences in the corpus.
        """
        return list(self._conllu_iterator(text_list))
    
    def find_phrases(self, text_list=None, doc_pattern=None):
        """Find phrases in the specified files or files matching the provided pattern."""
        if text_list:
            # If text_list is provided, find phrases in those specific files
            words_sents = self.get_sentences(text_list)
        elif doc_pattern:
            # If doc_pattern is provided, find phrases in files matching the pattern
            pattern = re.compile(doc_pattern)
            matching_files = [file for file in self.corpus_files if pattern.search(file)]
            words_sents = self.get_sentences(matching_files)
        else:
            # If neither text_list nor doc_pattern is provided, find phrases in all files
            words_sents = self.get_sentences()

        # Train the Phrases model on the selected sentences
        words_phrases_model = Phrases(words_sents, min_count=5, threshold=10)

        # Find and return the phrases
        phrases = words_phrases_model.find_phrases(words_sents).items()
        return list(phrases)

    def __iter__(self):
        """Iterate through the tokenized sentences in the entire corpus."""
        for sentence in self.get_sentences():
            yield sentence

    def __len__(self):
        """Return the total number of sentences in the corpus."""
        return sum(1 for _ in self.get_sentences())

In [ ]:
stopwords=["ab", "ac", "ad", "adhic", "aliqui", "aliquis", "an", "ante", "apud", "at", "atque", "aut", "autem", "cum", "cur", "de", "deinde", "dum", "ego", "enim", "ergo", "es", "est", "et", "etiam", "etsi", "ex", "fio", "haud", "hic", "iam", "idem", "igitur", "ille", "in", "infra", "inter", "interim", "ipse", "is", "ita", "magis", "modo", "mox", "nam", "ne", "nec", "necque", "neque", "nisi", "non", "nos", "o", "ob", "per", "possum", "post", "pro", "quae", "quam", "quare", "qui", "quia", "quicumque", "quidem", "quilibet", "quis", "quisnam", "quisquam", "quisque", "quisquis", "quo", "quoniam", "sed", "si", "sic", "sive", "sub", "sui", "sum", "super", "suus", "tam", "tamen", "trans", "tu", "tum", "ubi", "uel", "uero", "unus", "ut"]
corpus_gensim_words_sents = GensimCorpusSents(corpus_dir, corpus_files, stopwords_list=stopwords, use_lemmas=False)
corpus_gensim_lemmas_sents = GensimCorpusSents(corpus_dir, corpus_files, stopwords_list=stopwords, use_lemmas=True)

In [ ]:
from gensim.models import phrases
if mode == 'build':
    words_sents = corpus_gensim_words_sents.get_sentences()
    words_phrases_model = phrases.Phrases(corpus_gensim_words_sents)  # corpus_sents is your GensimCorpusSents instance

    lemmas_sents = corpus_gensim_lemmas_sents.get_sentences()
    lemmas_phrases_model = phrases.Phrases(corpus_gensim_lemmas_sents)  # corpus_sents is your GensimCorpusSents instance

    words_phrases_model.save("models/words_phrases_model.bin", separately=None, sep_limit=10485760, ignore=frozenset({}), pickle_protocol=4)
    lemmas_phrases_model.save("models/lemmas_phrases_model.bin", separately=None, sep_limit=10485760, ignore=frozenset({}), pickle_protocol=4)
elif mode == 'load':
    words_phrases_model.load("models/words_phrases_model.bin")
    lemmas_phrases_model.load("models/lemmas_phrases_model.bin")

In [ ]:
from gensim.models import phrases

words_sents = corpus_gensim_words_sents.get_sentences()
words_phrases_model = phrases.Phrases(corpus_gensim_words_sents)  # corpus_sents is your GensimCorpusSents instance

In [ ]:
lemmas_sents = corpus_gensim_lemmas_sents.get_sentences()
lemmas_phrases_model = phrases.Phrases(corpus_gensim_lemmas_sents)  # corpus_sents is your GensimCorpusSents instance

In [ ]:
words_sents = corpus_gensim_words_sents.get_sentences(text_list=['VAd_TEI_final.conllu',
 'VITELO_Persp1_TEI_final.conllu'])
words_phrases_model = phrases.Phrases(corpus_gensim_words_sents)  # corpus_sents is your GensimCorpusSents instance

In [ ]:
words_phrases_model.find_phrases(words_sents).items()

In [ ]:
for phrase, score in words_phrases_model.find_phrases(words_sents).items():
    print(phrase, score)

In [ ]:
for phrase, score in words_phrases_model.find_phrases(words_sents).items():
    print(phrase, score)
for phrase, score in lemmas_phrases_model.find_phrases(lemmas_sents).items():
    print(phrase, score)

### Comparing phrases between documents

In [ ]:
stopwords=["ab", "ac", "ad", "adhic", "aliqui", "aliquis", "an", "ante", "apud", "at", "atque", "aut", "autem", "cum", "cur", "de", "deinde", "dum", "ego", "enim", "ergo", "es", "est", "et", "etiam", "etsi", "ex", "fio", "haud", "hic", "iam", "idem", "igitur", "ille", "in", "infra", "inter", "interim", "ipse", "is", "ita", "magis", "modo", "mox", "nam", "ne", "nec", "necque", "neque", "nisi", "non", "nos", "o", "ob", "per", "possum", "post", "pro", "quae", "quam", "quare", "qui", "quia", "quicumque", "quidem", "quilibet", "quis", "quisnam", "quisquam", "quisque", "quisquis", "quo", "quoniam", "sed", "si", "sic", "sive", "sub", "sui", "sum", "super", "suus", "tam", "tamen", "trans", "tu", "tum", "ubi", "uel", "uero", "unus", "ut"]
gensim_acta_words_sents = GensimCorpusSents(corpus_dir, corpus_files=['PomnLw1_TEI_final.conllu'], stopwords_list=stopwords, use_lemmas=False)
gensim_acta_lemmas_sents = GensimCorpusSents(corpus_dir, corpus_files=['PomnLw1_TEI_final.conllu'], stopwords_list=stopwords, use_lemmas=True)

In [ ]:
acta_words_sents = gensim_acta_words_sents.get_sentences()
acta_words_phrases_model = phrases.Phrases(gensim_acta_words_sents, min_count=10,
                                           threshold=0.1, scoring="default")
acta_lemmas_sents = gensim_acta_lemmas_sents.get_sentences()
acta_lemmas_phrases_model = phrases.Phrases(gensim_acta_lemmas_sents, min_count=10,
                                            threshold=0.1, scoring="default")

In [ ]:
acta_words_phrases_model_f = acta_words_phrases_model.freeze()
acta_words_phrases_model_f.save('models/phrase_acta_words.pkl')
acta_lemmas_phrases_model_f = acta_lemmas_phrases_model.freeze()
acta_lemmas_phrases_model_f.save('models/phrase_acta_lemmas.pkl')

In [ ]:
for phrase, score in acta_lemmas_phrases_model.find_phrases(acta_lemmas_sents).items():
    print(phrase, score)

In [ ]:
import pandas as pd
pd.DataFrame(acta_words_phrases_model.find_phrases(acta_words_sents).items(),columns=['phrase', 'strength']).sort_values(['strength'], ascending=False)
pd.DataFrame(acta_lemmas_phrases_model.find_phrases(acta_lemmas_sents).items(),columns=['phrase', 'strength']).sort_values(['strength'], ascending=False)